# 🔥 The PyTorch Master Notebook
### From your first tensor to research-grade transformers — one notebook, start to finish

This notebook is meant to be **read once, top to bottom, like a course** — and then **kept forever as a reference** you jump back into before an interview, before starting a project, or before reading a research paper that uses PyTorch.

**How every topic is taught in this notebook:**
- **Why it exists** — the real problem it solves
- **Core intuition** — a plain-English mental model, no jargon left unexplained
- **The math**, kept as simple as the idea allows, written out step by step
- **A real-world analogy** so the idea sticks
- **Working code**, with a comment on almost every line explaining *what* it does and *why*

**Who this is for:** complete beginners who have basic Python knowledge, students preparing for ML Engineer / Data Scientist / MLOps / AI Infrastructure interviews, and anyone who wants a refresher before a research project. No prior deep learning knowledge is assumed — every term is explained the first time it shows up.

> 💡 **How to use this notebook:** Run the cells top to bottom the first time. After that, use the Table of Contents below to jump straight to the section you need to revise.

## 📑 Table of Contents

**Part A — Foundations**
0. [Setup & Installation](#setup)
1. [Tensors — Creation & Attributes](#tensors-creation)
2. [Tensors — Operations, Indexing, Reshaping, Broadcasting](#tensors-ops)
3. [Using the GPU (CUDA / MPS)](#gpu)
4. [Autograd — Automatic Differentiation](#autograd)
5. [The PyTorch Workflow — Linear Regression from Scratch to `nn.Module`](#workflow)

**Part B — Neural Network Building Blocks**

6. [`nn.Module` — How PyTorch Models Are Built](#nn-module)
7. [Loss Functions](#loss-functions)
8. [Optimizers & Learning-Rate Schedulers](#optimizers)
9. [`Dataset` & `DataLoader`](#datasets-dataloaders)
10. [The Training Loop & Evaluation Loop (+ full ANN example)](#training-loop)

**Part C — Architectures**

11. [Convolutional Neural Networks (CNN)](#cnn)
12. [Recurrent Neural Networks — RNN, LSTM, GRU](#rnn)
13. [Attention & Transformers](#transformers)

**Part D — Making Models Production- and Research-Ready**

14. [Saving & Loading Models](#save-load)
15. [Hyperparameters & Tuning with Optuna](#optuna)
16. [Transfer Learning](#transfer-learning)
17. [Regularization & Training Best Practices](#regularization)
18. [Advanced Topics: Mixed Precision, `torch.compile`, Custom Autograd, Metrics, Deployment](#advanced)

**Part E — Closing the Interview Gaps**

19. [PyTorch Internals — Memory Layout, Strides & Contiguity](#pytorch-internals)
20. [Debugging & Profiling PyTorch Like a Pro](#debugging-profiling)
21. [Distributed Training with DDP](#distributed-ddp)
22. [Fine-Tuning LLMs — LoRA, QLoRA & Quantization](#llm-finetuning)
23. [Career Roadmap, Portfolio Ideas & Where to Go Next](#career)

---

<a id="setup"></a>
# Part A — Foundations
## 0. What is PyTorch, and why does it matter?

**In one sentence:** PyTorch is a Python library for building things out of *tensors* (numbers arranged in grids) and automatically figuring out how to improve those things using calculus — which is exactly what "training a neural network" means.

### Why it exists (the problem it solves)
Before frameworks like PyTorch existed, if you wanted to train a neural network you had to hand-derive and hand-code the calculus (the *gradients*) for every single layer of your network. That is slow and extremely error-prone — one wrong derivative and your model silently fails to learn. PyTorch solves this with **autograd**: you just describe the *forward* computation (input → output), and PyTorch automatically works out how to go backward and compute every gradient for you. This single idea is why deep learning research exploded after frameworks like this appeared — researchers could try new ideas in hours instead of weeks.

### Why PyTorch specifically (vs. other frameworks)
- **It feels like normal Python/NumPy.** A PyTorch tensor behaves like a NumPy array that also happens to remember its own history for calculus purposes.
- **Dynamic computation graphs ("define-by-run").** The graph of operations is built fresh every time you run your code, so you can use normal Python `if`/`for` statements inside your model — great for debugging and for research where architectures change often.
- **It is the dominant framework in research and in industry** for training and fine-tuning deep learning models — including virtually all modern large language models — which is why it shows up constantly in ML Engineer, Data Scientist, and Applied/Research Scientist interviews.

### Core intuition to hold onto for this whole notebook
Think of building a model like building a machine out of Lego bricks (`nn.Module` layers), where each brick is a mathematical function. You:
1. Pass data through the bricks (**forward pass**) to get a prediction.
2. Measure how wrong the prediction was (**loss function**).
3. Ask PyTorch "if I wiggle every brick's internal numbers slightly, which direction makes the wrongness go down?" (**autograd → gradients**).
4. Actually wiggle every number a little bit in that direction (**optimizer step**).
5. Repeat thousands of times until the machine is good at its job (**training loop**).

Every single section below is really just a deeper look at one of these five steps.

### Installing PyTorch

Run the cell below **once** in your own environment (this notebook does not install anything for you automatically — installing depends on whether you have a GPU, and which OS you're on).

- **CPU only:** `pip install torch torchvision torchaudio`
- **NVIDIA GPU (CUDA):** visit https://pytorch.org/get-started/locally/ , pick your CUDA version, and copy the exact command shown there. The install command changes as new CUDA versions come out, so always get it fresh from that page rather than memorizing one.
- **Apple Silicon (M1/M2/M3/M4 Macs):** the standard `pip install torch torchvision torchaudio` already includes Apple's **MPS** GPU backend — no separate install needed.

We will also use a few standard libraries alongside PyTorch: `numpy` (numerical arrays), `matplotlib` (plotting), and later `optuna` (hyperparameter tuning) and `scikit-learn` (a couple of toy datasets and metrics).

In [ ]:
# Un-comment and run this line ONCE if torch/optuna/scikit-learn are not yet installed in your environment.
# The exact torch install command depends on your GPU/OS -- see https://pytorch.org/get-started/locally/
# !pip install torch torchvision torchaudio optuna scikit-learn matplotlib --quiet

import torch          # the core library: tensors + autograd + GPU support
import torch.nn as nn # the "neural network" toolbox: layers, loss functions, containers
import numpy as np    # numerical arrays, used alongside torch throughout this notebook

print("PyTorch version:", torch.__version__)   # confirms PyTorch imported correctly and shows which version you have
print("CUDA (NVIDIA GPU) available:", torch.cuda.is_available())  # True only if you have a working NVIDIA GPU setup


> 📌 **Note on this notebook's code cells:** every code cell below is written to run correctly on a **CPU-only machine** (so you can follow along on a laptop with no GPU), while also showing you the exact extra line(s) you'd add to use a GPU if one is available. Sections that need internet access to download a real dataset (CNN, transfer learning) say so clearly and also include a tiny synthetic-data version so you can still run and understand the code offline.

### 🎯 Interview Recap — Why PyTorch

**Q: Why is PyTorch so widely used in industry and research?**
A: It feels like normal Python/NumPy, it builds its computation graph dynamically ("define-by-run") which makes debugging and research iteration easy, and it's the dominant framework behind most modern deep learning research and production systems — including most large language models.

**Q: What's the one-sentence definition of PyTorch an interviewer wants to hear?**
A: A Python library for building tensor-based computations that can run on a GPU and automatically differentiate themselves (autograd) — which is exactly what's needed to train neural networks.

<a id="tensors-creation"></a>
## 1. Tensors — Creation & Attributes

### What is a tensor?
**In one sentence:** a tensor is just a grid of numbers with a certain number of dimensions, and PyTorch's version of that grid also knows how to run on a GPU and remember calculus history.

You already know tensors by other names:
| Dimensions | Math name | Real-world example |
|---|---|---|
| 0-D | scalar | a single number, e.g. today's temperature: `25.0` |
| 1-D | vector | a list of numbers, e.g. one row of exam scores: `[88, 92, 79]` |
| 2-D | matrix | a spreadsheet / table, e.g. a grayscale image (height × width) |
| 3-D | tensor | a color image: (channels × height × width), i.e. 3 stacked matrices for Red, Green, Blue |
| 4-D | tensor | a **batch** of color images: (batch × channels × height × width) |
| 5-D+ | tensor | e.g. a batch of videos: (batch × time × channels × height × width) |

So "tensor" is just the general word for "table of numbers with N dimensions" — a scalar, vector, and matrix are all just special, small cases of a tensor.

### Why PyTorch needs its own tensor type instead of just using NumPy arrays
A NumPy array is great at storing numbers and doing math on the **CPU**. PyTorch tensors do everything a NumPy array does, **plus**:
1. They can live on a **GPU**, where the same math runs 10-100x faster because a GPU can do thousands of small calculations simultaneously (this matters enormously once we reach [training neural networks](#gpu)).
2. They can remember **every operation done to them**, so PyTorch can automatically compute derivatives later (this is [autograd](#autograd), covered soon).

### Core intuition
Think of a tensor as an Excel-like grid, except it can have more than 2 dimensions, and every cell in the grid is the same data type (all floats, or all integers — never mixed, unlike a Python list).

In [ ]:
import torch

# ---- 0-D tensor (a scalar) ----
scalar = torch.tensor(7)          # torch.tensor() converts a Python number/list/nested-list into a tensor
print(scalar)                     # tensor(7)  -- notice PyTorch always prints the word "tensor"
print(scalar.ndim)                # 0  -- ndim = "number of dimensions". A scalar has none.
print(scalar.item())              # 7  -- .item() pulls the plain Python number out of a 0-D (or 1-element) tensor

# ---- 1-D tensor (a vector) ----
vector = torch.tensor([1, 2, 3])  # a plain Python list becomes a 1-D tensor
print(vector)                     # tensor([1, 2, 3])
print(vector.ndim)                # 1  -- one dimension (a single row/list of numbers)
print(vector.shape)               # torch.Size([3])  -- shape tells you the size along each dimension: 3 elements

# ---- 2-D tensor (a matrix) ----
matrix = torch.tensor([[1, 2], [3, 4]])   # a list of lists becomes a 2-D tensor (rows x columns)
print(matrix)
print(matrix.ndim)                # 2
print(matrix.shape)                # torch.Size([2, 2])  -- 2 rows, 2 columns

# ---- 3-D tensor (e.g. a tiny "image": 2 channels x 2 height x 2 width) ----
tensor_3d = torch.tensor([[[1, 2], [3, 4]],
                           [[5, 6], [7, 8]]])
print(tensor_3d.shape)             # torch.Size([2, 2, 2])  -- 2 channels, each a 2x2 grid


### Every common way to *create* a tensor

You will use these constructors constantly, so it's worth seeing all of them together. The pattern to notice: **most of them take a `shape` (also called `size`) as their argument**, e.g. `(3, 4)` means "3 rows, 4 columns".

In [ ]:
# torch.zeros(shape) -> a tensor full of 0s. Used to initialize things like a running total, or a mask.
zeros = torch.zeros(2, 3)          # 2 rows, 3 columns, every value 0.0
print("zeros:\n", zeros)

# torch.ones(shape) -> a tensor full of 1s. Used for masks, or as a starting point for weights.
ones = torch.ones(2, 3)
print("ones:\n", ones)

# torch.full(shape, value) -> a tensor filled with any constant you choose
full = torch.full((2, 3), 7)       # every value is 7
print("full:\n", full)

# torch.rand(shape) -> random numbers drawn UNIFORMLY from [0, 1). Good for demos / simple random init.
rand = torch.rand(2, 3)
print("rand (uniform 0-1):\n", rand)

# torch.randn(shape) -> random numbers drawn from a STANDARD NORMAL distribution (mean 0, std 1).
# This is what's actually used to randomly initialize real neural network weights.
randn = torch.randn(2, 3)
print("randn (normal, mean 0 std 1):\n", randn)

# torch.arange(start, end, step) -> like Python's range(), but returns a tensor
arange = torch.arange(0, 10, 2)    # 0, 2, 4, 6, 8  (end=10 is NOT included, just like range())
print("arange:", arange)

# torch.linspace(start, end, steps) -> `steps` numbers evenly spaced between start and end (end IS included)
linspace = torch.linspace(0, 1, 5) # 0.0, 0.25, 0.5, 0.75, 1.0 -- exactly 5 numbers
print("linspace:", linspace)

# torch.eye(n) -> an n x n identity matrix (1s on the diagonal, 0s elsewhere). Used in linear algebra.
identity = torch.eye(3)
print("identity:\n", identity)

# torch.empty(shape) -> allocates memory WITHOUT initializing values (contains "garbage" leftover memory).
# Faster than zeros/ones because it skips the initialization step -- only use it when you're about
# to immediately overwrite every value yourself, e.g. inside a custom training loop buffer.
empty = torch.empty(2, 3)
print("empty (uninitialized, values are meaningless):\n", empty)

# The "_like" family: create a new tensor with the SAME SHAPE as an existing one. Extremely common
# in real code, e.g. "give me a zero tensor shaped exactly like my model's output".
existing = torch.rand(2, 3)
print("zeros_like:\n", torch.zeros_like(existing))   # same shape as `existing`, filled with 0
print("ones_like:\n", torch.ones_like(existing))     # same shape as `existing`, filled with 1
print("rand_like:\n", torch.rand_like(existing))     # same shape as `existing`, new random values


**Real-world example:** when you build a neural network layer, PyTorch uses `torch.randn`-style random initialization internally to give every weight a small random starting value (never all-zero — if every weight started identical, every neuron would learn the exact same thing, which defeats the purpose of having many neurons — this is called the *symmetry problem*, and it's why "random" initialization matters, not just "any" initialization).

### Tensor attributes: `.shape`, `.dtype`, `.device`, `.ndim`, `.numel()`

Every tensor carries a small amount of metadata about itself. **You will check these four attributes constantly while debugging** — the single most common PyTorch bug is a *shape mismatch* between two tensors you tried to combine, and the second most common is a *device mismatch* (one tensor on CPU, one on GPU).

In [ ]:
x = torch.rand(3, 4)         # a random 3x4 tensor to inspect

print("shape :", x.shape)     # torch.Size([3, 4])  -- size along each dimension. x.size() does the same thing.
print("ndim  :", x.ndim)      # 2  -- how many dimensions (== len(x.shape))
print("dtype :", x.dtype)     # torch.float32 -- the data type of every element (see next cell)
print("device:", x.device)    # cpu -- WHERE the tensor's memory physically lives (cpu / cuda:0 / mps)
print("numel :", x.numel())   # 12 -- total NUMber of ELements = 3 * 4 = 12 (product of the shape)
print("requires_grad:", x.requires_grad)  # False by default -- whether PyTorch should track this tensor
                                           # for calculus (autograd). Explained fully in the Autograd section.


### Data types (`dtype`) — and why they matter

A tensor's `dtype` decides two things: **how much memory each number uses**, and **whether it can hold decimals or only whole numbers**. Picking the right dtype is not just a technicality — mixing incompatible dtypes is one of the most common PyTorch errors, and using the wrong dtype (e.g. float64 everywhere) can silently double your memory usage and slow down GPU training for no benefit.

| dtype | Meaning | Typical use |
|---|---|---|
| `torch.float32` (`torch.float`) | 32-bit decimal number | **The default for almost all neural network weights and data.** Best balance of precision vs. speed/memory. |
| `torch.float16` / `torch.bfloat16` | 16-bit decimal number | Used for *mixed precision training* to save memory and go faster on modern GPUs (see the [Advanced Topics](#advanced) section). |
| `torch.float64` (`torch.double`) | 64-bit decimal number | Rarely needed in deep learning; used when extreme numerical precision matters (e.g. some scientific computing). |
| `torch.int64` (`torch.long`) | 64-bit whole number | **Required** for class labels in classification, and for indexing into tensors. |
| `torch.int32` / `torch.int8` | smaller whole numbers | Memory-saving in specific cases (e.g. quantized models). |
| `torch.bool` | True/False | Masks — "which elements should I keep/ignore?" |

In [ ]:
# Setting a dtype explicitly at creation time
float_tensor = torch.tensor([1, 2, 3], dtype=torch.float32)   # force float32 even though inputs look like ints
int_tensor   = torch.tensor([1, 2, 3], dtype=torch.int64)     # force int64 (a.k.a. "long") -- used for labels

print(float_tensor.dtype)   # torch.float32
print(int_tensor.dtype)     # torch.int64

# Converting (casting) an EXISTING tensor to a different dtype -- this creates a NEW tensor, it does not
# modify the original in place.
x = torch.tensor([1.9, 2.5, 3.1])
print(x.int())          # tensor([1, 2, 3])  -- .int() TRUNCATES decimals, it does not round!
print(x.long())         # same idea, but int64 -- the dtype you need for classification labels
print(x.to(torch.float64))  # .to(dtype) is the general-purpose way to cast to ANY dtype

# WHY this matters: this line would crash with a dtype error in many operations, because you generally
# can't mix an integer tensor and a float tensor in certain operations without an explicit cast:
a = torch.tensor([1, 2, 3])          # dtype: int64
b = torch.tensor([1.0, 2.0, 3.0])    # dtype: float32
c = a.float() + b                    # FIX: cast `a` to float32 first, then add -- now dtypes match
print(c)


### 🎯 Interview Recap — Tensors & Attributes

**If you remember nothing else:** a tensor is an N-dimensional grid of one shared data type; the default dtype is `float32`; `.shape`, `.dtype`, and `.device` are the three attributes you check first when something crashes.

**Q: What is a tensor, in one sentence?**
A: A grid of numbers of any number of dimensions, all sharing one data type, that can live on a CPU or GPU and optionally track its own operation history for gradients.

**Q: How is a PyTorch tensor different from a NumPy array?**
A: A tensor can run on a GPU and can automatically track operations for gradient computation (autograd); a NumPy array can do neither.

**Q: What's the default dtype when you write `torch.tensor([1.0, 2.0])`?**
A: `torch.float32`.

**Q: Why would an operation between two tensors fail with a dtype error?**
A: PyTorch won't silently mix incompatible dtypes (e.g. an int64 tensor and a float32 tensor) in every operation — you need to explicitly cast one with `.to(dtype)`, `.float()`, `.long()`, etc.

**Q: What does `.item()` do, and when do you use it?**
A: It pulls a plain Python number out of a single-element tensor — the standard way to grab a loss value for logging/printing without dragging along the whole tensor/graph.

**Q: What's the difference between `torch.zeros(3,4)` and `torch.zeros_like(x)`?**
A: `zeros(3,4)` takes an explicit shape; `zeros_like(x)` copies the shape (and dtype/device) of an existing tensor `x` automatically.

<a id="tensors-ops"></a>
## 2. Tensor Operations — Indexing, Reshaping, Math, Broadcasting

This section covers everything you do *to* a tensor after you've created it. These operations are the actual "verbs" of PyTorch — almost every line inside a real model's `forward()` method is one of the operations below.

### Indexing and slicing
PyTorch indexing works almost exactly like NumPy / Python list indexing, just extended to N dimensions. **Core intuition:** each comma-separated position picks a dimension; `:` means "take everything along this dimension".

In [ ]:
x = torch.arange(1, 13).reshape(3, 4)   # numbers 1..12 arranged as 3 rows, 4 columns
print(x)
# tensor([[ 1,  2,  3,  4],
#         [ 5,  6,  7,  8],
#         [ 9, 10, 11, 12]])

print(x[0])          # row 0 entirely            -> tensor([1, 2, 3, 4])
print(x[0, 2])        # row 0, column 2 (single element) -> tensor(3)
print(x[:, 1])        # ALL rows, column 1 only    -> tensor([2, 6, 10])
print(x[0:2, :])       # rows 0 and 1 (row 2 excluded), all columns
print(x[:, 1:3])       # all rows, columns 1 and 2 only
print(x[-1])          # last row (negative indexing works just like Python lists)

# Boolean ("mask") indexing -- extremely common for filtering data
mask = x > 6           # a same-shaped tensor of True/False
print(mask)
print(x[mask])         # returns a flat 1-D tensor of only the elements where mask is True: [7,8,9,10,11,12]

# torch.where(condition, a, b): element-wise "if condition then a else b" -- vectorized if/else
print(torch.where(x > 6, x, torch.zeros_like(x)))  # keep values > 6, replace everything else with 0


### Reshaping — `view`, `reshape`, `squeeze`, `unsqueeze`, `permute`, `transpose`

**Why reshaping exists:** the *data* inside a tensor rarely needs to change, but the *shape you're allowed to see it as* very often does. A neural network layer might require input shaped `(batch, features)`, but your data loader hands you `(batch, height, width)` — you reshape to reconcile the two without touching any actual numbers.

In [ ]:
x = torch.arange(12)              # a flat 1-D tensor: [0, 1, ..., 11]
print(x.shape)                     # torch.Size([12])

# .reshape(new_shape): returns the SAME 12 numbers arranged into a new shape.
# One dimension can be -1, meaning "figure this size out automatically".
y = x.reshape(3, 4)                 # 3 rows x 4 columns = 12 elements, matches
print(y)
z = x.reshape(2, -1)                 # 2 rows, PyTorch computes columns = 12/2 = 6 automatically
print(z.shape)                      # torch.Size([2, 6])

# .view() does the same job as .reshape(), with one difference: .view() REQUIRES the tensor's
# underlying memory to already be arranged in a compatible ("contiguous") way, and will error out
# if not. .reshape() is the safer default -- it uses .view() when possible and silently makes a
# copy when it isn't. Use .reshape() unless you specifically need to guarantee no copy is made.
v = x.view(3, 4)
print(v.shape)

# .unsqueeze(dim): INSERTS a new dimension of size 1 at position `dim`.
# Extremely common use: your model expects a "batch" dimension, but you only have ONE example.
single_image = torch.rand(28, 28)          # one grayscale 28x28 image, shape (28, 28)
batched = single_image.unsqueeze(0)         # add a batch dimension at position 0
print(batched.shape)                        # torch.Size([1, 28, 28])  -- "a batch containing 1 image"

# .squeeze(dim): REMOVES dimensions of size 1. The inverse of unsqueeze.
back_to_single = batched.squeeze(0)
print(back_to_single.shape)                 # torch.Size([28, 28])

# .permute(*dims): REORDERS the dimensions (does not change the data, just how it's indexed).
# Classic use case: images are usually loaded as (Height, Width, Channels) but PyTorch's CNN
# layers expect (Channels, Height, Width).
img_hwc = torch.rand(28, 28, 3)             # (H, W, C) -- e.g. how you'd load a raw image file
img_chw = img_hwc.permute(2, 0, 1)          # move dim 2 (channels) to the front -> (C, H, W)
print(img_chw.shape)                        # torch.Size([3, 28, 28])

# .transpose(dim0, dim1): swaps exactly two dimensions -- a special case of permute, common for matrices.
m = torch.rand(2, 3)
print(m.transpose(0, 1).shape)              # torch.Size([3, 2]) -- rows and columns swapped


### Arithmetic operations & matrix multiplication

**Core distinction to memorize:** `*` is **element-wise** multiplication (each cell multiplied with the matching cell), while `@` / `torch.matmul` is **matrix multiplication** (the "real" linear-algebra kind, where rows are dotted with columns). Confusing these two is one of the most common bugs when building a neural network layer by hand.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# Element-wise arithmetic -- same shape in, same shape out, operates position-by-position
print(a + b)     # tensor([5., 7., 9.])   -- equivalent to torch.add(a, b)
print(a - b)     # tensor([-3., -3., -3.])
print(a * b)     # tensor([4., 10., 18.]) -- ELEMENT-WISE multiply, NOT matrix multiplication
print(a / b)     # tensor([0.25, 0.4, 0.5])
print(a ** 2)    # tensor([1., 4., 9.])   -- element-wise power

# Matrix multiplication: for 2-D tensors, this is the standard linear-algebra "row dot column" product.
# Rule: (n, k) @ (k, m) -> (n, m). The INNER dimensions (k) must match.
A = torch.rand(2, 3)   # 2 rows, 3 columns
B = torch.rand(3, 4)   # 3 rows, 4 columns
C = A @ B               # (2,3) @ (3,4) -> (2,4).  `@` is shorthand for torch.matmul(A, B)
print(C.shape)          # torch.Size([2, 4])

# This is EXACTLY the operation happening inside every nn.Linear layer:
# output = input @ weight.T + bias
# so understanding matmul shapes is the key to understanding why layers connect the way they do.


### Broadcasting — how PyTorch handles operations on *different-shaped* tensors

**Core intuition:** broadcasting is PyTorch's rule for "stretching" a smaller tensor so it lines up with a bigger one, *without actually copying data in memory* (it's done virtually, so it's free). It exists because writing `for` loops to manually repeat values would be slow and ugly.

**The rule (compare shapes from the right / trailing dimension):**
1. If two dimensions are equal, they're compatible.
2. If one of them is 1, it gets "stretched" to match the other.
3. If a tensor has fewer dimensions, it's padded with size-1 dimensions on the left until the number of dimensions matches.
4. If neither of the above is true for some dimension, broadcasting fails with a shape error.

In [ ]:
# Example 1: adding a scalar to a matrix -- the scalar broadcasts to every element
m = torch.ones(3, 3)
print(m + 10)             # 10 is treated as if it were a 3x3 matrix of 10s

# Example 2: adding a row-vector to a matrix -- a very common real pattern (e.g. adding a bias to every row)
matrix = torch.ones(4, 3)        # shape (4, 3): 4 samples, 3 features each
row_bias = torch.tensor([1.0, 2.0, 3.0])   # shape (3,) -- one bias value per feature
result = matrix + row_bias        # (4,3) + (3,) -> the (3,) row is "copied" across all 4 rows
print(result.shape)               # torch.Size([4, 3])
print(result)

# Example 3: shapes that CANNOT broadcast -- this raises a RuntimeError, shown here as a comment:
# bad = torch.ones(4, 3) + torch.ones(4, 2)   # 3 != 2 and neither is 1 -> incompatible, error!

# Real-world example: this exact mechanism is how a bias term gets added to EVERY sample in a batch
# inside a linear layer, without writing a manual loop over the batch dimension.


### Aggregation — `sum`, `mean`, `max`, `min`, `argmax`, `argmin`

Aggregation operations **collapse** a dimension down to a single summary number. The `dim` argument tells PyTorch *which* dimension to collapse — this trips up beginners constantly, so the mental model below is worth memorizing.

In [ ]:
x = torch.tensor([[1., 2., 3.],
                   [4., 5., 6.]])       # shape (2, 3) -- think "2 samples, 3 features"

print(x.sum())            # 21.0  -- no dim given: collapses EVERYTHING into one number
print(x.sum(dim=0))        # tensor([5., 7., 9.]) -- collapses dim 0 (rows): "sum down each column"
print(x.sum(dim=1))        # tensor([6., 15.])    -- collapses dim 1 (columns): "sum across each row"
# MEMORY TRICK: dim=X means "the size of dimension X disappears from the output shape".
# x.shape is (2,3). sum(dim=0) removes the "2" -> output shape (3,). sum(dim=1) removes the "3" -> shape (2,).

print(x.mean())            # 3.5   -- average of all elements
print(x.mean(dim=1))        # tensor([2., 5.])  -- average of each row (used to compute per-sample loss, etc.)

print(x.max())              # 6.0   -- the single largest value
print(x.max(dim=1))          # returns BOTH the max values AND their indices along dim 1:
                             # torch.return_types.max(values=tensor([3., 6.]), indices=tensor([2, 2]))

print(x.argmax(dim=1))       # tensor([2, 2]) -- INDEX of the max value in each row.
# This is THE operation used to turn a classifier's raw output scores into a predicted class label:
# "which class got the highest score for this sample?"


### In-place operations (the trailing underscore `_`)

Any method ending in an underscore modifies the tensor **in place** (overwrites its own memory) instead of returning a new tensor. They save memory, but should be used carefully.

In [ ]:
x = torch.tensor([1., 2., 3.])
x.add_(10)          # in-place add: x itself becomes [11., 12., 13.] -- nothing is returned/assigned
print(x)

y = torch.tensor([1., 2., 3.])
y = y.add(10)        # NON-in-place: creates a NEW tensor, original y's old memory is untouched
print(y)

# CAUTION: avoid in-place ops on any tensor that is part of an autograd computation graph
# (i.e. requires_grad=True and used in a forward pass) -- overwriting values in place can destroy
# information autograd needs to compute gradients correctly, and PyTorch will often raise an error
# to protect you. Prefer the non-in-place version unless you have a specific, deliberate memory reason.


### The NumPy bridge

Because so much of the Python data ecosystem (pandas, scikit-learn, matplotlib, OpenCV...) speaks NumPy, PyTorch makes converting between the two essentially free.

In [ ]:
import numpy as np

# NumPy array -> PyTorch tensor
np_array = np.array([1.0, 2.0, 3.0])
tensor_from_np = torch.from_numpy(np_array)     # shares the SAME underlying memory as np_array!
print(tensor_from_np)

# PyTorch tensor -> NumPy array
tensor = torch.tensor([4.0, 5.0, 6.0])
np_from_tensor = tensor.numpy()                  # also shares memory (only works for CPU tensors)
print(np_from_tensor)

# IMPORTANT GOTCHA: torch.from_numpy() and .numpy() SHARE memory by default -- changing one changes
# the other! This is a deliberate speed optimization (no copying), but it surprises beginners:
np_array[0] = 999.0
print(tensor_from_np)     # tensor_from_np ALSO changed, because they share the same memory block!

# If you want an independent copy instead, use torch.tensor(np_array) or tensor.clone() / np_array.copy()
safe_copy = torch.tensor(np_array)   # this one does NOT share memory


### 🎯 Interview Recap — Tensor Operations

**If you remember nothing else:** `*` is element-wise, `@`/`matmul` is real matrix multiplication; `dim=X` in a reduction means "dimension X disappears from the output"; broadcasting stretches size-1 dimensions for free; NumPy↔tensor conversions share memory by default.

**Q: What's the difference between `*` and `@` on two 2-D tensors?**
A: `*` multiplies matching positions element-by-element (shapes must already match or broadcast); `@` performs proper matrix multiplication, where the inner dimensions must match and rows are dotted with columns.

**Q: What's the difference between `.view()` and `.reshape()`?**
A: Both return a tensor with a new shape over the same data when possible; `.view()` requires the underlying memory to already be contiguous and errors out otherwise, while `.reshape()` falls back to making a copy automatically — so `.reshape()` is the safer default.

**Q: Explain broadcasting in one sentence.**
A: PyTorch automatically "stretches" dimensions of size 1 (or pads missing leading dimensions) so two differently-shaped tensors can be combined without manually copying data.

**Q: On a `(2, 3)` tensor, what's the output shape of `.sum(dim=0)` vs `.sum(dim=1)`?**
A: `dim=0` collapses the rows, giving shape `(3,)`; `dim=1` collapses the columns, giving shape `(2,)` — the named dimension is the one that disappears.

**Q: Why can `torch.from_numpy()` cause a subtle bug?**
A: It shares the same underlying memory as the original NumPy array by default, so modifying one silently modifies the other — use `torch.tensor(np_array)` if you want an independent copy.

**Q: Why would you call `.unsqueeze(0)` on a single input before feeding it to a model?**
A: Models expect a batch dimension; `unsqueeze(0)` inserts a size-1 dimension at the front so one sample looks like "a batch of 1".

<a id="gpu"></a>
## 3. Using the GPU

### Why GPUs matter (the core intuition)
A **CPU** is like a small team of extremely skilled generalists — a few powerful cores that can do complex tasks one after another very quickly. A **GPU** is like a stadium of thousands of workers who are individually simpler, but can all do the *same simple task* (e.g. "multiply these two numbers") **at the exact same time**.

Neural network math — matrix multiplication above all — is made up of millions of small, independent multiply-and-add operations. That is precisely the kind of work a GPU is built for: instead of doing those millions of multiplications one after another (CPU), a GPU does thousands of them simultaneously. This is why training a network that might take days on a CPU can take hours or minutes on a GPU.

### Why this needs explicit handling in code
Unlike some frameworks, PyTorch does not automatically decide where to run your computation. **Every tensor lives on exactly one device** (CPU, or a specific GPU), and an operation between two tensors on *different* devices will raise an error. You are always in control of, and responsible for, where your data and your model live.

In [ ]:
import torch

# Check what hardware is available
print("CUDA (NVIDIA GPU) available:", torch.cuda.is_available())
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())  # Mac M-series GPUs

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))          # e.g. "NVIDIA A100"
    print("Number of GPUs:", torch.cuda.device_count())


### The device-agnostic code pattern (industry standard)

Real code should run correctly whether or not a GPU is present, without you having to edit anything. The universal pattern used in essentially every professional PyTorch codebase is: **detect the best available device once, store it in a variable, then pass that variable everywhere.**

In [ ]:
# THE standard pattern -- copy this into every PyTorch project you write.
device = (
    "cuda" if torch.cuda.is_available()          # prefer an NVIDIA GPU if one exists
    else "mps" if torch.backends.mps.is_available()  # otherwise, prefer an Apple Silicon GPU
    else "cpu"                                    # otherwise, fall back to the CPU
)
print(f"Using device: {device}")

# Moving a TENSOR to a device: .to(device) returns a NEW tensor living on that device
x = torch.rand(3, 3)
x = x.to(device)          # from now on, x's data physically lives on `device`
print(x.device)

# Moving a MODEL to a device: .to(device) moves ALL of the model's internal weights, in place
# (unlike tensors, calling .to() on a model modifies it in place AND returns it, so both of these work)
model = torch.nn.Linear(3, 1)   # a tiny model, explained fully in the nn.Module section
model = model.to(device)         # every weight and bias inside `model` now lives on `device`

# THE GOLDEN RULE: a model and the data you feed it must be on the SAME device, or PyTorch will
# raise: "RuntimeError: Expected all tensors to be on the same device". This is one of the single
# most common PyTorch errors beginners hit -- always double check .device when debugging a crash.
sample_input = torch.rand(1, 3).to(device)   # move the input to match the model's device
output = model(sample_input)                  # now this works, both are on `device`
print(output.device)


**Real-world example:** in a training script running on a cloud GPU machine, this exact three-line `device` pattern is usually one of the first things written — and copy-pasting your notebook code onto a GPU server should "just work" without editing a single other line, precisely because the rest of the code always refers to `device` instead of hardcoding `"cpu"` or `"cuda"`.

**When to move data to the GPU:** as late as possible, and in small pieces (e.g. one batch at a time inside the training loop) — not your entire dataset all at once, which usually won't even fit in GPU memory. This pattern is covered fully in the [Training Loop](#training-loop) section.

### 🎯 Interview Recap — GPU / Device Management

**If you remember nothing else:** a GPU wins by running thousands of simple operations in parallel, which matches matrix multiplication; a tensor and the model it feeds must be on the *same* device; always write device-agnostic code.

**Q: Why are GPUs faster than CPUs for deep learning?**
A: A GPU has thousands of simpler cores that can all run the same simple operation (like multiply-add) at once, matching how neural network math — mostly matrix multiplication — is structured; a CPU has far fewer, more powerful cores optimized for complex sequential work.

**Q: What happens if your model is on the GPU but your input tensor is on the CPU?**
A: PyTorch raises a `RuntimeError` telling you tensors are expected to be on the same device — you'd fix it by calling `.to(device)` on the input.

**Q: What's the standard device-agnostic pattern used in real PyTorch code?**
A: Detect the best available device once — `"cuda" if torch.cuda.is_available() else "cpu"` — store it in a `device` variable, and call `.to(device)` on both the model and every batch of data, instead of hardcoding `"cpu"` or `"cuda"` anywhere.

**Q: Does calling `.to(device)` on a tensor change it in place?**
A: No — for a tensor it returns a new tensor on that device; for an `nn.Module`, `.to(device)` moves the model's parameters in place (and also returns the model for convenience).

<a id="autograd"></a>
## 4. Autograd — Automatic Differentiation

This is arguably **the single most important idea in this entire notebook**. Everything from here onward — training loops, CNNs, transformers — is really just "autograd, applied to a bigger function."

### Why autograd exists (the problem it solves)
"Training" a neural network means: given how wrong the model's current prediction is (the **loss**), figure out how to nudge every single weight in the network so the wrongness goes down. Calculus gives us the tool for this: the **derivative** (or, with many variables, the **gradient**) tells you the direction of steepest increase of a function. If you know the gradient of the loss with respect to a weight, you know exactly which direction to move that weight to *decrease* the loss.

A real network can have millions or billions of weights. Manually working out (and then correctly coding) the calculus for millions of interacting weights by hand is not humanly feasible. **Autograd is PyTorch's engine for computing every one of those derivatives automatically, exactly, and fast.**

### Core intuition: the computational graph
Every time you do math on a tensor that has `requires_grad=True`, PyTorch secretly builds a **graph** behind the scenes: a record of "this new tensor was made by applying operation X to these input tensor(s)". This happens invisibly as your code runs (this is what "define-by-run" means).

When you finally call `.backward()` on some output (almost always a single number — the loss), PyTorch walks that graph **backward**, from the output to every input, applying the **chain rule** from calculus at every step, and accumulates the result into each input's `.grad` attribute.

**Real-world analogy:** imagine a factory assembly line where a product passes through several machines (Machine A → Machine B → Machine C → final product). If the final product turns out to be 5% defective, and you want to know "how much did Machine B's calibration setting contribute to that defect rate?", you would trace *backward* through the line, asking at each machine "how sensitive is the next machine's output to this machine's output?", and multiply those sensitivities together. That chain of multiplied sensitivities *is* the chain rule, and that backward trace *is* what `.backward()` does.

### The chain rule, in the exact form autograd uses it

If $y = f(x)$ and $z = g(y)$, i.e. $z = g(f(x))$, then:
$$\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}$$

In words: *the sensitivity of the final output to the very first input equals the product of every "local" sensitivity along the chain of operations connecting them.* Autograd computes exactly this product, automatically, for a chain (graph) that can be millions of operations long.

In [ ]:
import torch

# ---- Step 1: create a tensor and tell PyTorch to track it for gradients ----
x = torch.tensor(3.0, requires_grad=True)   # requires_grad=True: "remember every operation done to me"

# ---- Step 2: do some math (this builds the computational graph automatically) ----
y = x ** 2          # y = x^2.  By calculus: dy/dx = 2x
print(y)             # tensor(9., grad_fn=<PowBackward0>)
                      # notice `grad_fn` -- this is PyTorch's record of "y was created by a Power operation".
                      # This is the graph! Every non-leaf tensor has a grad_fn pointing to how it was made.

# ---- Step 3: call .backward() to trigger the chain-rule walk backward through the graph ----
y.backward()          # since y is a single number (scalar), no arguments are needed

# ---- Step 4: read the computed gradient off of x.grad ----
print(x.grad)          # tensor(6.)  ==  2 * x  ==  2 * 3.0  -- EXACTLY matches the calculus by hand!


### A slightly bigger, multi-step example (this is what "backpropagation" really is)

Let's chain a few operations together, exactly like a tiny neural network layer would, and verify the chain rule by hand.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x * 3        # y = 3x            -> dy/dx = 3
z = y ** 2        # z = y^2 = (3x)^2 = 9x^2  -> dz/dx = 18x

z.backward()       # walk backward: dz/dx = dz/dy * dy/dx = (2y) * (3) = (2 * 6) * 3 = 36
print(x.grad)       # tensor(36.)  --  matches 18 * x = 18 * 2 = 36 exactly.

# What actually happened internally, step by step:
#   1. PyTorch recorded: z came from "square" applied to y   -> local derivative rule: dz/dy = 2y
#   2. PyTorch recorded: y came from "multiply by 3" applied to x -> local derivative rule: dy/dx = 3
#   3. .backward() starts at z with an implicit gradient of 1 (dz/dz = 1)
#   4. multiplies backward through each recorded step: 1 * dz/dy * dy/dx = 1 * (2*6) * 3 = 36
# This "multiply local derivatives together as you walk backward" IS backpropagation.


### Leaf tensors vs. the rest of the graph

A **leaf tensor** is one *you* created directly (like `x` above) — it sits at the start of the graph. Only leaf tensors (with `requires_grad=True`) accumulate a usable `.grad` by default; intermediate tensors like `y` and `z` above are NOT leaves, and their `.grad` is discarded after `.backward()` to save memory, unless you explicitly ask PyTorch to keep it with `.retain_grad()`. In practice, the leaf tensors in a real network are its **weights and biases** — exactly the things you want gradients for, since those are what the optimizer will update.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3

print("x.is_leaf:", x.is_leaf)   # True  -- created directly by you
print("y.is_leaf:", y.is_leaf)   # False -- created by an operation (multiplication)


### Gradient accumulation, and why you must call `optimizer.zero_grad()`

**Critical, easy-to-miss fact: gradients ACCUMULATE (add up) by default across multiple `.backward()` calls — they do not automatically reset to zero.** This is intentional (it's occasionally useful for advanced techniques), but it means that if you forget to clear `.grad` before each new training step, your gradients will silently be the WRONG, inflated sum of every previous step's gradient — a very common and very confusing bug for beginners.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("after 1st backward:", x.grad)   # tensor(4.)  == 2*x == 2*2

y2 = x ** 2
y2.backward()                            # backward() called AGAIN without clearing the old gradient first
print("after 2nd backward:", x.grad)   # tensor(8.)  -- NOT 4 again! It ADDED to the previous 4 -> 4+4=8

# THE FIX: reset the gradient to zero before computing a new one. In a real training loop, this is
# handled by calling optimizer.zero_grad() once per iteration, right before loss.backward().
x.grad.zero_()          # manually zero it here to demonstrate the fix
y3 = x ** 2
y3.backward()
print("after zeroing then 3rd backward:", x.grad)   # tensor(4.) again -- correct, fresh gradient


### `torch.no_grad()` and `.detach()` — turning autograd tracking OFF

Building the computational graph costs extra memory and compute. Once you're done *training* and just want to *use* a model to make predictions (**inference**), or whenever you need to do a calculation that should NOT be part of the graph (e.g. an evaluation metric, or manually updating a weight), you should switch tracking off.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# torch.no_grad(): a context manager -- EVERYTHING computed inside this block is NOT tracked.
# Standard usage: wrap your entire evaluation/inference loop in this.
with torch.no_grad():
    y = x ** 2
    print(y.requires_grad)     # False -- no graph was built, saving memory and time

# .detach(): pulls a SPECIFIC tensor OUT of the graph, returning a new tensor that shares the same
# data but has requires_grad=False and no grad_fn. Useful when you want to keep using a value
# (e.g. for logging/plotting) without dragging its entire computation history along with it.
y = x ** 2
y_value_only = y.detach()
print(y_value_only.requires_grad)   # False
print(y.requires_grad)              # True -- the ORIGINAL y is untouched; detach() returns a new tensor

# Common real-world pattern: converting a loss value to a plain number for logging/printing,
# WITHOUT accidentally keeping its graph alive (which would waste memory over a long training run):
loss = (x ** 2)
loss_for_logging = loss.item()   # .item() implicitly detaches AND converts to a plain Python float
print(loss_for_logging, type(loss_for_logging))


### Why this matters for real training (the intuition to carry forward)

When you later write:
```python
loss.backward()
optimizer.step()
```
here is exactly what's happening, tying every piece above together:
1. Your model's `forward()` pass builds a computational graph from the input all the way to a single scalar `loss`, because every weight involved has `requires_grad=True` (PyTorch sets this automatically for the parameters of every `nn.Module` layer — covered in the [next section](#nn-module)).
2. `loss.backward()` walks that graph backward, filling in `.grad` for every weight using the chain rule.
3. `optimizer.step()` reads each weight's `.grad` and nudges the weight a small amount in the direction that reduces the loss (the actual math of *how* it nudges is covered in [Optimizers](#optimizers)).
4. `optimizer.zero_grad()` clears every `.grad` back to zero, so the next iteration starts fresh instead of accumulating stale gradients.

Every neural network you will ever train in PyTorch — from a 2-layer toy network to a transformer with billions of parameters — is running exactly this same four-step loop underneath.

### 🎯 Interview Recap — Autograd

**⭐ This is one of the single most frequently asked PyTorch interview topics — know it cold.**

**If you remember nothing else:** `requires_grad=True` tells PyTorch to track a tensor; `.backward()` walks the recorded graph backward applying the chain rule; gradients ACCUMULATE, so you must `zero_grad()`; `no_grad()`/`.detach()` turn tracking off.

**Q: What is autograd, in one sentence?**
A: PyTorch's engine for automatically computing exact derivatives of any function built from tensor operations, by recording every operation as a graph while the code runs and applying the chain rule backward through it.

**Q: What does `.backward()` actually do?**
A: It walks the computational graph backward from the output tensor to every leaf tensor with `requires_grad=True`, applying the chain rule at each recorded operation, and accumulates the result into each leaf's `.grad`.

**Q: Why do you have to call `optimizer.zero_grad()` every training iteration?**
A: Because gradients accumulate (add up) across multiple `.backward()` calls instead of resetting automatically; skipping this means each step uses the wrong, inflated sum of every previous step's gradient.

**Q: What's the difference between `torch.no_grad()` and `.detach()`?**
A: `no_grad()` is a context manager that stops the graph from being built for everything inside the block; `.detach()` pulls one specific tensor out of an already-built graph, returning a new tensor with no history, while the original tensor's graph is untouched.

**Q: What is a "leaf" tensor?**
A: A tensor created directly by you (not produced by an operation) — model weights and biases are leaf tensors, and only leaves accumulate a usable `.grad` by default.

**Q: Why does `.backward()` normally require the output to be a single number (scalar)?**
A: Backward computes vector-Jacobian products starting from an initial gradient; for a scalar, that initial gradient is implicitly `1`. For a non-scalar output, there's no single well-defined "starting point," so you must explicitly pass a gradient tensor matching its shape.

**Q: What is the computational graph, conceptually?**
A: A record, built automatically as your code runs, of which operations produced which tensors — it's what lets `.backward()` know exactly how to trace back from the loss to every parameter that contributed to it.

<a id="workflow"></a>
## 5. The PyTorch Workflow — Linear Regression, Three Ways

### The general workflow (every PyTorch project ever, at a high level)

```
1. Prepare data        -> turn raw data into tensors (features X, targets y)
2. Build a model        -> a function with learnable parameters (weights) that maps X -> prediction
3. Choose a loss + optimizer -> loss measures "how wrong"; optimizer decides "how to fix it"
4. Training loop         -> repeat: forward pass -> compute loss -> backward pass -> optimizer step
5. Evaluate              -> check performance on data the model did NOT train on
6. Save / deploy / iterate
```

The rest of this notebook is essentially a deep dive into each of these six steps. To make the whole workflow concrete right now, we'll solve the simplest possible learning problem — **linear regression** (fit a line `y = w*x + b` to some noisy data) — **three times**, each time removing one layer of "doing it by hand". This progression is the fastest way to build a rock-solid intuition for *why* every later abstraction (`nn.Module`, loss functions, optimizers) exists.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)   # makes randomness reproducible -- same "random" numbers every run of this notebook

# ---- Synthetic data: the TRUE relationship is y = 2x + 1, plus a bit of random noise ----
# In the real world you never know the true relationship -- that's exactly what training discovers.
X = torch.linspace(0, 10, 100).unsqueeze(1)          # 100 evenly spaced x-values, shape (100, 1)
true_w, true_b = 2.0, 1.0
y = true_w * X + true_b + torch.randn(X.shape) * 1.0  # y = 2x + 1 + random noise

plt.figure(figsize=(6, 4))
plt.scatter(X.numpy(), y.numpy(), s=10, alpha=0.6, label="noisy data")
plt.plot(X.numpy(), (true_w * X + true_b).numpy(), color="red", label="true relationship (unknown to the model)")
plt.legend(); plt.title("Our toy dataset"); plt.xlabel("x"); plt.ylabel("y"); plt.show()


### Attempt 1 — Fully manual (no autograd at all)

Here we compute the **Mean Squared Error** loss and its gradients **by hand**, using the calculus formulas we derive ourselves, then update the weights ourselves. This is what deep learning looked like before automatic differentiation existed — and why it doesn't scale to real networks.

**The math:** for prediction $\hat y = wx + b$ and loss $L = \frac{1}{n}\sum (\hat y_i - y_i)^2$, calculus gives us:
$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum (\hat y_i - y_i)\, x_i \qquad\qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum (\hat y_i - y_i)$$

In [ ]:
# Start with random-ish initial guesses for the parameters (NOT using requires_grad here on purpose)
w = torch.tensor(0.0)
b = torch.tensor(0.0)
lr = 0.01           # "learning rate" -- how big a step to take each update. Explored fully in Optimizers.

for epoch in range(200):                    # one "epoch" = one full pass over updating using all the data
    y_pred = w * X + b                       # forward pass: our current best-guess prediction
    loss = ((y_pred - y) ** 2).mean()         # Mean Squared Error: average squared prediction error

    # ---- gradients, computed BY HAND using the calculus formulas derived above ----
    grad_w = (2 * (y_pred - y) * X).mean()
    grad_b = (2 * (y_pred - y)).mean()

    # ---- manual gradient descent update: move w and b a small step AGAINST the gradient ----
    # (against the gradient, because the gradient points toward INCREASING loss, and we want to DECREASE it)
    w -= lr * grad_w
    b -= lr * grad_b

    if epoch % 40 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | w={w.item():.3f} b={b.item():.3f}")

print(f"\nFinal learned: w={w.item():.3f}, b={b.item():.3f}  (true values were w=2.0, b=1.0)")


**Why this doesn't scale:** we had to derive $\partial L/\partial w$ and $\partial L/\partial b$ ourselves. A real network might have millions of parameters chained through dozens of layers — deriving and correctly coding that many gradient formulas by hand is not realistic. This is exactly the problem autograd solves.

### Attempt 2 — Same problem, using autograd

Identical setup, but now we let `requires_grad=True` and `.backward()` compute the gradients for us — no hand-derived calculus required. Notice the training loop's *shape* barely changes; only *how the gradient is obtained* changes.

In [ ]:
w = torch.tensor(0.0, requires_grad=True)   # track gradients for w
b = torch.tensor(0.0, requires_grad=True)   # track gradients for b
lr = 0.01

for epoch in range(200):
    y_pred = w * X + b
    loss = ((y_pred - y) ** 2).mean()

    loss.backward()          # autograd computes grad_w and grad_b for us, stored in w.grad / b.grad

    with torch.no_grad():     # the UPDATE step itself must NOT be tracked by autograd (see Autograd section)
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_()         # reset gradients to 0 -- remember, they ACCUMULATE otherwise!
        b.grad.zero_()

    if epoch % 40 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | w={w.item():.3f} b={b.item():.3f}")

print(f"\nFinal learned: w={w.item():.3f}, b={b.item():.3f}  (true values were w=2.0, b=1.0)")


### Attempt 3 — The idiomatic, real-world PyTorch way

Now we replace: our hand-written `w*x + b` with `nn.Linear` (a proper **layer**), our hand-written squared-error with `nn.MSELoss` (a proper **loss function**), and our manual `w -= lr * w.grad` update with `torch.optim.SGD` (a proper **optimizer**). This is exactly the code structure you'll use for every model in the rest of this notebook — a tiny linear regression and a giant transformer are trained with the *same four-line loop skeleton*.

In [ ]:
import torch.nn as nn

model = nn.Linear(in_features=1, out_features=1)   # ONE learnable layer: y = w*x + b, w and b created for us
loss_fn = nn.MSELoss()                               # the same Mean Squared Error, provided by PyTorch
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # SGD = the same update rule as Attempt 2,
                                                             # model.parameters() automatically finds w AND b

for epoch in range(200):
    y_pred = model(X)               # forward pass -- calling the model like a function runs its forward()
    loss = loss_fn(y_pred, y)        # compute how wrong the predictions are

    optimizer.zero_grad()             # step 1: clear old gradients (same idea as w.grad.zero_() above)
    loss.backward()                    # step 2: autograd computes fresh gradients for EVERY parameter
    optimizer.step()                   # step 3: update EVERY parameter using those gradients

    if epoch % 40 == 0:
        w_learned, b_learned = model.weight.item(), model.bias.item()
        print(f"epoch {epoch:3d} | loss {loss.item():.4f} | w={w_learned:.3f} b={b_learned:.3f}")

w_learned, b_learned = model.weight.item(), model.bias.item()
print(f"\nFinal learned: w={w_learned:.3f}, b={b_learned:.3f}  (true values were w=2.0, b=1.0)")


### What we just proved to ourselves
All three attempts converge to essentially the same answer (`w≈2.0`, `b≈1.0`) because **they are mathematically the exact same algorithm** — gradient descent on mean squared error. The only thing that changed across the three attempts is *how much PyTorch does for us*:

| | Gradients | Model | Update rule |
|---|---|---|---|
| Attempt 1 | hand-derived | hand-written `w*x+b` | hand-written |
| Attempt 2 | **autograd** | hand-written `w*x+b` | hand-written |
| Attempt 3 | **autograd** | **`nn.Module`** | **`optimizer`** |

This is the mental model to carry through the rest of the notebook: **`nn.Module`, loss functions, and optimizers are not new concepts — they are convenient, tested, reusable packaging around the exact loop you just wrote by hand.** The next sections open up each of those three boxes in detail.

### 🎯 Interview Recap — The PyTorch Workflow

**If you remember nothing else:** every training iteration is `zero_grad() → forward → loss → backward() → step()`, and `nn.Module`/loss functions/optimizers are just tested, reusable packaging around math you could write by hand.

**Q: What are the four core steps inside every PyTorch training iteration?**
A: Clear old gradients (`optimizer.zero_grad()`), run a forward pass to get predictions, compute the loss, call `loss.backward()` to get fresh gradients, then `optimizer.step()` to update the weights using those gradients.

**Q: Why must `optimizer.step()` come after `loss.backward()`?**
A: `.step()` reads the `.grad` values that `.backward()` just filled in; calling it first would update parameters using stale or empty gradients.

**Q: Could you train a linear regression model in PyTorch without `nn.Linear`, `nn.MSELoss`, or `torch.optim`?**
A: Yes — you can hand-write the prediction formula (`w*x + b`), the squared-error loss, and manually update the weights using their `.grad` after `.backward()`; the built-in classes do the exact same math, just packaged and tested for you.

<a id="nn-module"></a>
# Part B — Neural Network Building Blocks
## 6. `nn.Module` — How Every PyTorch Model Is Built

### Why `nn.Module` exists
In [Section 5](#workflow) we saw `nn.Linear` create `w` and `b` for us and `model.parameters()` magically find them for the optimizer. `nn.Module` is the base class that makes this possible: it's PyTorch's standard way of packaging **learnable numbers (parameters)** together with **the computation that uses them (`forward`)** into one reusable, inspectable, saveable object.

### Core intuition
Think of `nn.Module` as a **Lego brick that remembers its own internal numbers.** Every layer you use (`nn.Linear`, `nn.Conv2d`, `nn.LSTM`...) is a Lego brick built this way, and — crucially — **you build your own custom models by snapping these bricks together inside a bigger brick that is *also* an `nn.Module`.** This is why the same class is used whether you're building a 1-layer toy model or a 100-layer network: modules nest inside modules, indefinitely.

### `nn.Parameter` — how PyTorch recognizes "this tensor is learnable"

A regular tensor with `requires_grad=True` gets tracked by autograd, but `nn.Module` needs a way to know *which* of its internal tensors are learnable weights (so it can hand them to an optimizer, save them, move them to a GPU, etc.). That's what `nn.Parameter` is: **a tensor subclass that automatically registers itself with the surrounding `nn.Module`** the moment you assign it as an attribute.

In [ ]:
import torch
import torch.nn as nn

class TinyManualLayer(nn.Module):
    # A from-scratch re-implementation of nn.Linear(1, 1), to see what's happening under the hood.
    def __init__(self):
        super().__init__()                                      # ALWAYS call this first -- sets up
                                                                    # the internal bookkeeping nn.Module needs
        self.w = nn.Parameter(torch.randn(1))                     # a learnable weight, randomly initialized
        self.b = nn.Parameter(torch.zeros(1))                     # a learnable bias, initialized to 0

    def forward(self, x):                                          # defines what happens on a forward pass
        return x * self.w + self.b

layer = TinyManualLayer()
print(list(layer.parameters()))         # both w and b show up automatically -- because nn.Parameter
                                          # registers itself, we never had to manually maintain a list

x = torch.tensor([[2.0]])
print(layer(x))    # calling layer(x) -- NOT layer.forward(x) directly, explained next -- runs forward()


> ⚠️ **Always call `layer(x)`, never `layer.forward(x)` directly.** `nn.Module` overrides Python's `__call__` method to run some important extra bookkeeping (like switching training-mode behavior for layers such as Dropout, and hooks used for debugging) *around* your `forward()` method. Calling `.forward()` directly skips all of that and can cause subtle bugs.

### Two ways to build a model: `nn.Sequential` vs. subclassing `nn.Module`

**`nn.Sequential`** is the fast, no-fuss way to stack layers when data simply flows straight through them, one after another, with no branching or custom logic. **Subclassing `nn.Module`** (writing your own `__init__` + `forward`) is the flexible, general way — required the moment your model needs branches, multiple inputs, loops, skip-connections, or any custom logic.

In [ ]:
# ---- Option 1: nn.Sequential -- great for simple "straight-line" architectures ----
simple_model = nn.Sequential(
    nn.Linear(10, 32),   # layer 1: 10 input features -> 32 outputs
    nn.ReLU(),            # activation function (explained below)
    nn.Linear(32, 16),    # layer 2: 32 -> 16
    nn.ReLU(),
    nn.Linear(16, 1),     # layer 3: 16 -> 1 final output
)
print(simple_model)
x = torch.rand(4, 10)      # a batch of 4 samples, 10 features each
print(simple_model(x).shape)   # torch.Size([4, 1])

# ---- Option 2: subclassing nn.Module -- the general, flexible way ----
class MyModel(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()                            # required first line, always
        self.layer1 = nn.Linear(in_features, hidden)   # define every layer you'll use as an attribute
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden, out_features)

    def forward(self, x):                                # define HOW those layers connect on a forward pass
        x = self.layer1(x)     # pass input through layer 1
        x = self.relu(x)        # apply the activation function
        x = self.layer2(x)      # pass through layer 2 to get the final output
        return x

model = MyModel(in_features=10, hidden=32, out_features=1)
print(model)
print(model(x).shape)          # same shape as before: torch.Size([4, 1])

# The subclass form lets you do things nn.Sequential physically cannot express, e.g. reusing a layer
# twice, adding a residual/skip connection (x = self.layer(x) + x), or handling multiple inputs:
class ResidualModel(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.layer = nn.Linear(features, features)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.layer(x)) + x    # <-- a "skip connection": impossible to express in Sequential


### What is an activation function, and why do we need it? (`ReLU`, `Sigmoid`, `Tanh`, `GELU`)

**The problem:** if you stack `Linear` layers directly on top of each other with nothing in between, the *entire stack* mathematically collapses into a single equivalent `Linear` layer — no matter how many layers you use, because a linear function of a linear function is still just linear. Stacking layers would then be pointless.

**The fix:** insert a **non-linear** function between layers. This breaks the "collapses into one line" problem and is what lets a deep network represent complex, curved, real-world patterns instead of only straight lines/planes.

| Activation | Formula | Typical use |
|---|---|---|
| `ReLU` | $\max(0, x)$ | The default choice for hidden layers in most networks — simple, fast, works well. |
| `Sigmoid` | $\frac{1}{1+e^{-x}}$ | Squashes to (0,1) — used on the OUTPUT layer for binary classification (as a probability). |
| `Tanh` | $\frac{e^x - e^{-x}}{e^x+e^{-x}}$ | Squashes to (-1,1) — historically common in RNNs. |
| `GELU` / `SiLU` | smooth ReLU-like curves | The default in most modern transformers (smoother than ReLU, tends to train slightly better). |
| `Softmax` | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | Turns a vector of raw scores into probabilities that sum to 1 — used on the OUTPUT layer for multi-class classification. |

In [ ]:
x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])

print("ReLU   :", nn.functional.relu(x))     # negatives become 0, positives pass through unchanged
print("Sigmoid:", torch.sigmoid(x))            # every value squashed into (0, 1)
print("Tanh   :", torch.tanh(x))               # every value squashed into (-1, 1)
print("Softmax:", torch.softmax(x, dim=0))     # all 5 outputs are positive AND sum to exactly 1.0
print("sums to:", torch.softmax(x, dim=0).sum().item())


### A quick-reference table of common `nn` layers

You'll meet every one of these in detail in later sections (CNN, RNN, Transformer) — this table is here so you have one place to look them up.

| Layer | What it does | Typical use |
|---|---|---|
| `nn.Linear(in, out)` | fully-connected layer: `y = xW^T + b` | tabular data, the final layers of almost any network |
| `nn.Conv2d(in_ch, out_ch, kernel)` | slides learnable filters over an image | image data — see [CNN section](#cnn) |
| `nn.MaxPool2d(kernel)` | downsamples by taking the max in each window | shrinking image feature maps — see [CNN section](#cnn) |
| `nn.BatchNorm2d` / `nn.LayerNorm` | normalizes activations for stable training | almost every modern deep network — see [Regularization](#regularization) |
| `nn.Dropout(p)` | randomly zeroes some activations during training | preventing overfitting — see [Regularization](#regularization) |
| `nn.RNN` / `nn.LSTM` / `nn.GRU` | processes sequences, step by step | text, time series — see [RNN section](#rnn) |
| `nn.MultiheadAttention` / `nn.TransformerEncoder` | lets every position attend to every other position | modern NLP/vision — see [Transformer section](#transformers) |
| `nn.Embedding(vocab_size, dim)` | looks up a learnable vector for each integer ID | turning word/token IDs into vectors |
| `nn.Flatten()` | reshapes everything after the batch dim into 1 dimension | bridging a Conv2d output into a Linear layer |

### Weight initialization — why it matters and how to control it

By default, PyTorch's layers (`nn.Linear`, `nn.Conv2d`, ...) initialize their own weights using a sensible random scheme automatically — you don't have to think about this for most projects. But it's useful to understand *why* it's random and small, and how to override it for special cases (e.g. reproducing a research paper's exact setup).

**Why not zero?** If every weight started at exactly 0 (or any identical value), every neuron in a layer would compute the exact same output and receive the exact same gradient during backprop — they would learn to become identical copies of each other forever, wasting most of the network's capacity. This is called the **symmetry problem**, and it's why initialization must be *random*.

**Why not too big or too small?** Very large initial weights tend to make activations explode (or saturate an activation like `sigmoid`, causing tiny gradients). Very small ones can make gradients vanish as they multiply through many layers. Modern schemes like **Kaiming/He initialization** (good default for ReLU networks) and **Xavier/Glorot initialization** (good default for `tanh`/`sigmoid` networks) scale the randomness based on the layer's size specifically to keep the *variance* of activations roughly stable from layer to layer.

In [ ]:
layer = nn.Linear(10, 10)
print("default init sample:", layer.weight[0, :3])   # PyTorch already applied a sensible default for you

# Manually applying He/Kaiming initialization (common for ReLU-based networks)
nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")   # trailing _ -> in-place, as covered in Part 2
nn.init.zeros_(layer.bias)                                    # biases are commonly initialized to 0
print("after manual He init:", layer.weight[0, :3])

# Applying an initialization scheme to EVERY layer in a bigger model at once, using .apply()
def init_weights(module):
    if isinstance(module, nn.Linear):                 # only touch Linear layers
        nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
        nn.init.zeros_(module.bias)

model = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 1))
model.apply(init_weights)     # .apply() recursively runs a function on EVERY sub-module


### Inspecting a model: parameters, parameter counts, and `state_dict`

Two things you'll do constantly: counting how big a model is, and peeking at its structure. (Full saving/loading is covered in its own [dedicated section](#save-load) — this is just how to *look* at a model.)

In [ ]:
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))

# Count total learnable numbers in the model -- a standard "how big is this model?" metric
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")   # (10*32 + 32) + (32*1 + 1) = 352 + 33 = 385

# Count only the TRAINABLE parameters (relevant once we cover freezing layers in Transfer Learning)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

# named_parameters() gives you the name AND the tensor -- useful for debugging which layer is which
for name, param in model.named_parameters():
    print(f"{name:20s} shape={tuple(param.shape)}")

# state_dict(): an ordered dictionary of {parameter_name: tensor} -- this is THE object that gets
# saved to disk when you save a model. More in the Saving & Loading section.
print(list(model.state_dict().keys()))


### 🎯 Interview Recap — `nn.Module`

**If you remember nothing else:** `nn.Module` packages learnable `Parameter`s with a `forward()` method; always call the model, never `.forward()` directly; use `Sequential` for straight-line stacks and subclassing for anything with branches/custom logic; random init breaks the "symmetry problem."

**Q: Why must you call `super().__init__()` first inside a custom `nn.Module`'s `__init__`?**
A: It sets up the internal bookkeeping (the parameter and sub-module registries) that `nn.Module` relies on; skipping it breaks automatic parameter tracking.

**Q: What's the difference between calling `model(x)` and `model.forward(x)`?**
A: `model(x)` goes through `nn.Module.__call__`, which runs important extra logic (like train/eval-mode behavior and hooks) around your `forward()`; calling `.forward()` directly skips that and can cause subtle bugs — always call the model itself.

**Q: When would you subclass `nn.Module` instead of using `nn.Sequential`?**
A: Whenever the architecture isn't a simple straight line — branches, skip/residual connections, multiple inputs, loops, or any custom logic `Sequential` can't express.

**Q: Why do neural networks need non-linear activation functions between layers?**
A: Stacking only `Linear` layers mathematically collapses into one equivalent `Linear` layer no matter how many you stack — a non-linearity is what lets a deep network represent curved, complex functions instead of only straight lines/planes.

**Q: Why is initializing every weight to zero a problem?**
A: Every neuron in a layer would compute the same output and get the same gradient, so they'd stay identical copies of each other forever — random initialization breaks this "symmetry problem."

**Q: What does `model.parameters()` return, and what is it used for?**
A: An iterator over every `nn.Parameter` registered anywhere inside the model, including inside nested sub-modules — it's what you pass to an optimizer so it knows what to update.

<a id="loss-functions"></a>
## 7. Loss Functions

### Why loss functions exist
A loss function converts "how wrong was the model's prediction?" into a **single number** that gets smaller as the model gets better. This single number is exactly what `.backward()` computes gradients *of* — it is the very top of the computational graph discussed in [Autograd](#autograd). Choosing the wrong loss function for your problem is one of the most common ways a model fails to train well, so understanding *why* each one exists matters as much as knowing its name.

### Regression losses (predicting a continuous number)

**Mean Squared Error (`nn.MSELoss`)**
$$L = \frac{1}{n}\sum_{i=1}^n (\hat y_i - y_i)^2$$
**Intuition:** squaring the error means big mistakes are punished *disproportionately* more than small ones (an error of 10 contributes 100, an error of 2 contributes only 4). **When to use it:** the default choice for regression. **Watch out for:** it's sensitive to outliers — one wildly wrong prediction can dominate the whole loss, because of that squaring.

**Mean Absolute Error (`nn.L1Loss`)**
$$L = \frac{1}{n}\sum_{i=1}^n |\hat y_i - y_i|$$
**Intuition:** every unit of error is punished equally, regardless of size. **When to use it:** your data has outliers you don't want to dominate training (e.g. predicting house prices where a few mansions could otherwise skew everything).

**Huber Loss / Smooth L1 (`nn.HuberLoss` / `nn.SmoothL1Loss`)**
Behaves like MSE for small errors (smooth, easy to optimize) and like MAE for large errors (robust to outliers) — literally the best of both, with a threshold `delta` controlling where it switches between the two. **When to use it:** a strong general-purpose default when you suspect some outliers but still want smooth gradients near zero error.

In [ ]:
import torch
import torch.nn as nn

predictions = torch.tensor([3.0, 5.0, 2.5, 8.0])
targets     = torch.tensor([2.5, 5.0, 2.0, 20.0])   # notice the last target is a big outlier (20 vs pred 8)

mse = nn.MSELoss()
mae = nn.L1Loss()
huber = nn.HuberLoss()

print("MSE  :", mse(predictions, targets).item())    # dominated by the outlier -> large value
print("MAE  :", mae(predictions, targets).item())    # outlier still counted, but not squared -> smaller
print("Huber:", huber(predictions, targets).item())  # a compromise between the two


### Classification losses (predicting a category)

**⚠️ The single most important gotcha in this whole section:** PyTorch's classification losses expect **raw, unbounded scores straight from your last `Linear` layer ("logits")** — **not** probabilities you've already run through `Sigmoid`/`Softmax` yourself. The loss functions apply that squashing internally, in a numerically safer, combined way. Applying `Sigmoid`/`Softmax` yourself *and then* using these loss functions is a very common beginner bug that quietly hurts training.

**Binary Cross-Entropy (`nn.BCEWithLogitsLoss`)** — for problems with exactly 2 classes, or several *independent* yes/no labels per sample (multi-label).
$$L = -\big[y \log(\sigma(x)) + (1-y)\log(1-\sigma(x))\big]$$
where $x$ is the raw model output (logit) and $\sigma$ is the sigmoid function. **Intuition:** the loss is small when the predicted probability is close to the true 0/1 label, and grows sharply (toward infinity) the more confidently wrong the model is — this strongly punishes confident mistakes. **Use `nn.BCEWithLogitsLoss`** (combines sigmoid + BCE in one numerically stable step) — **not** the older `nn.BCELoss` + manual `sigmoid`, unless you have a specific reason to.

**Cross-Entropy Loss (`nn.CrossEntropyLoss`)** — for problems with more than 2 mutually-exclusive classes (exactly one correct answer per sample).
$$L = -\log\left(\frac{e^{x_{y}}}{\sum_j e^{x_j}}\right)$$
This is `softmax` followed by taking the negative log of the probability assigned to the *correct* class. **Intuition:** if the model puts high probability on the correct class, $-\log(\text{high probability})$ is close to 0 (low loss); if it puts low probability on the correct class, $-\log(\text{low probability})$ shoots up toward infinity (high loss). **Important detail:** `nn.CrossEntropyLoss` expects the targets as plain **integer class indices** (e.g. `2` meaning "class 2"), **not** one-hot vectors — this trips up a lot of beginners coming from other frameworks.

In [ ]:
# ---- Binary classification example ----
logits_binary = torch.tensor([2.0, -1.5, 0.3])     # raw model outputs, one per sample (NOT yet sigmoid'd)
labels_binary = torch.tensor([1.0, 0.0, 1.0])       # true labels: 1 = positive class, 0 = negative class

bce = nn.BCEWithLogitsLoss()
print("BCEWithLogitsLoss:", bce(logits_binary, labels_binary).item())
# internally this does: sigmoid(logits_binary) to get probabilities, THEN compares to labels_binary

# ---- Multi-class classification example (e.g. classifying an image into 1 of 5 categories) ----
logits_multiclass = torch.tensor([[2.0, 0.5, 0.1, -1.0, 0.3],    # sample 0's raw score for each of 5 classes
                                   [0.1, 3.0, 0.2,  0.0, 0.1]])   # sample 1's raw score for each of 5 classes
true_class_indices = torch.tensor([0, 1])            # sample 0's true class is 0, sample 1's true class is 1
                                                        # NOTE: these are plain integers, NOT one-hot vectors!

ce = nn.CrossEntropyLoss()
print("CrossEntropyLoss:", ce(logits_multiclass, true_class_indices).item())
# Low loss here is expected: sample 0 already scores highest on class 0, sample 1 highest on class 1

# WHY you should NOT apply softmax yourself first:
wrong_way = ce(torch.softmax(logits_multiclass, dim=1), true_class_indices)  # double-softmaxes internally!
print("Loss if you WRONGLY pre-apply softmax yourself:", wrong_way.item(), "<- notice this is a different, WRONG number")


### Quick decision guide

| Your problem | Use this loss |
|---|---|
| Predicting a continuous number (price, temperature...) | `nn.MSELoss` (default) or `nn.HuberLoss` (outlier-robust) |
| Exactly 2 classes (spam / not spam) | `nn.BCEWithLogitsLoss` |
| Several independent yes/no tags per sample (multi-label) | `nn.BCEWithLogitsLoss` |
| More than 2 mutually-exclusive classes | `nn.CrossEntropyLoss` |
| Model's last layer | Leave it as a plain `nn.Linear` with **no activation** — let the loss function apply sigmoid/softmax internally |

### 🎯 Interview Recap — Loss Functions

**If you remember nothing else:** regression → MSE/MAE/Huber; binary or multi-label → `BCEWithLogitsLoss`; multi-class → `CrossEntropyLoss`; always feed **raw logits**, never pre-softmaxed probabilities; `CrossEntropyLoss` wants integer class indices, not one-hot vectors.

**Q: Why shouldn't you apply `Sigmoid` yourself before `nn.BCEWithLogitsLoss`?**
A: `BCEWithLogitsLoss` applies sigmoid internally in a single, numerically stable step; applying it yourself first effectively double-applies sigmoid and produces the wrong loss and gradients.

**Q: What label format does `nn.CrossEntropyLoss` expect?**
A: Plain integer class indices (e.g. `2` meaning "class 2"), not one-hot encoded vectors.

**Q: When would you pick MAE over MSE for a regression task?**
A: When your data has outliers you don't want to dominate training — MSE squares errors so large mistakes are punished disproportionately, while MAE punishes every unit of error equally.

**Q: What loss would you use for a multi-label problem, where a sample can belong to several independent classes at once?**
A: `nn.BCEWithLogitsLoss`, applied per label — `CrossEntropyLoss` assumes exactly one correct, mutually-exclusive class per sample, which doesn't fit multi-label problems.

<a id="optimizers"></a>
## 8. Optimizers & Learning-Rate Schedulers

### Core intuition: the ball rolling downhill
Picture the loss function as a hilly landscape, and your model's current parameters as a ball's position on that landscape. Height = loss. Training means: **repeatedly nudge the ball a little bit downhill** until it settles near the bottom (a low-loss region). The gradient (from autograd) tells us the *uphill* direction at the ball's current position, so we always step in the *opposite* direction. The **optimizer** is the specific strategy for deciding exactly how big and in what refined direction each step should be — plain gradient descent is the simplest possible strategy, and everything below is a smarter refinement of it.

### SGD (Stochastic Gradient Descent) — the foundation
$$\theta_{t+1} = \theta_t - \text{lr} \cdot \nabla L(\theta_t)$$
The parameter $\theta$ moves a small step (`lr`, the **learning rate**) in the direction opposite the gradient $\nabla L$. "Stochastic" refers to computing this on a random mini-batch of data rather than your whole dataset at once (see [DataLoader](#datasets-dataloaders)) — this makes each step cheap and adds helpful noise that can bounce the ball out of small, bad dips in the landscape.

**The learning rate is arguably the single most important hyperparameter in all of deep learning:**
- **Too small:** training crawls forward and can take forever, or gets permanently stuck in a mediocre dip.
- **Too large:** the ball overshoots the bottom of the valley and can bounce around chaotically, or even diverge (loss shoots up toward infinity instead of down).

### SGD with Momentum
Plain SGD can zig-zag badly in narrow valleys (steep in one direction, shallow in another). **Momentum** adds a "velocity" term — like a heavy ball that keeps rolling in the direction it's been consistently moving, which smooths out the zig-zag and speeds up progress along consistent directions.
$$v_{t+1} = \mu\, v_t + \nabla L(\theta_t) \qquad \theta_{t+1} = \theta_t - \text{lr}\cdot v_{t+1}$$
where $\mu$ (typically 0.9) controls how much of the previous velocity carries over.

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 1)

# Plain SGD
optimizer_sgd = torch.optim.SGD(model.parameters(), lr=0.01)

# SGD with momentum -- almost always better than plain SGD, at essentially no extra cost
optimizer_sgd_momentum = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)


### Adaptive optimizers — RMSProp and Adam

SGD uses the *same* learning rate for every parameter. But some parameters might need big updates and others tiny ones. **Adaptive optimizers give every individual parameter its own effective learning rate**, based on that parameter's recent gradient history.

**RMSProp** divides each parameter's step by a running average of that parameter's *recent squared gradients* — parameters with consistently large gradients get their steps shrunk (to avoid overshooting), and parameters with consistently small gradients get their steps enlarged (so they still learn at a reasonable pace).
$$v_t = \beta v_{t-1} + (1-\beta) g_t^2 \qquad\qquad \theta_{t+1} = \theta_t - \frac{\text{lr}}{\sqrt{v_t}+\epsilon}\, g_t$$

**Adam (Adaptive Moment Estimation)** — **the default choice for the vast majority of modern deep learning** — combines momentum (a running average of the gradient itself, called the 1st moment) *with* RMSProp-style adaptive scaling (a running average of the squared gradient, the 2nd moment), plus a bias-correction step because both running averages start at zero and would otherwise be biased low early in training.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \qquad\qquad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\hat m_t = \frac{m_t}{1-\beta_1^t} \qquad\qquad \hat v_t = \frac{v_t}{1-\beta_2^t} \qquad\qquad \theta_{t+1} = \theta_t - \text{lr}\cdot\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}$$

**Real-world example / intuition:** think of $m_t$ (momentum) as *"which direction has this parameter consistently wanted to move lately?"* and $v_t$ as *"how large and volatile have this parameter's updates been lately?"*. Dividing by $\sqrt{\hat v_t}$ means a parameter that's been changing wildly gets a smaller, more cautious step, while a parameter that's barely been moving gets a relatively bigger, more confident step — this is exactly why Adam tends to "just work" with far less learning-rate tuning than plain SGD.

**AdamW** is the version used almost everywhere today — it fixes a subtle bug in how the original Adam combined `weight_decay` (explained next) with the adaptive scaling, and is now the standard default for training transformers.

In [ ]:
optimizer_rmsprop = torch.optim.RMSprop(model.parameters(), lr=0.001)

optimizer_adam = torch.optim.Adam(model.parameters(), lr=0.001)   # lr=0.001 (1e-3) is a very common Adam default

# AdamW: the modern go-to default for training most networks, including virtually all transformers
optimizer_adamw = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)


### `weight_decay` (L2 regularization) — a quick preview
`weight_decay` adds a tiny extra pull on every weight toward zero at each step, discouraging the model from relying on any single weight being extremely large. This is one specific technique for **regularization** (fighting overfitting), covered in full in its [own section](#regularization) — it's mentioned here because it's set directly on the optimizer, not as a separate layer.

### Choosing an optimizer — quick guide
| Situation | Good default |
|---|---|
| You're not sure / general purpose | **Adam** or **AdamW**, `lr=1e-3` as a starting point |
| Training a Transformer / LLM | **AdamW**, usually with a warmup + decay schedule (below) |
| Classic computer vision (ResNets etc.) | **SGD with momentum** often generalizes slightly better, with a well-tuned LR schedule |
| Very simple / textbook problems | Plain **SGD** — easiest to reason about by hand |

### Learning-rate schedulers

**Why schedule the learning rate instead of keeping it fixed?** Early in training, big steps help you make fast progress across the landscape. Late in training, once you're near a good minimum, big steps cause you to bounce around it instead of settling in — so shrinking the learning rate over time typically improves final performance. A scheduler automates this shrinking according to some rule.

In [ ]:
model = nn.Linear(10, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# StepLR: multiply the LR by `gamma` every `step_size` epochs -- simple, staircase-shaped decay
step_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# CosineAnnealingLR: smoothly decays the LR following a cosine curve down to (near) zero -- very
# popular for training modern vision and language models because the smooth decay tends to help
# the model settle into a good minimum near the end of training.
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# ReduceLROnPlateau: watches a metric (usually validation loss) and shrinks the LR only when that
# metric STOPS improving for `patience` epochs in a row -- reacts to the actual training dynamics
# instead of following a fixed schedule.
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

# Usage pattern inside a training loop (full training loops are covered in the next section):
for epoch in range(3):                       # tiny demo loop
    # ... forward pass, loss.backward(), optimizer.step(), optimizer.zero_grad() would go here ...
    step_scheduler.step()                      # most schedulers: call .step() once per epoch, AFTER optimizer.step()
    print(f"epoch {epoch}: lr = {optimizer.param_groups[0]['lr']:.5f}")

# ReduceLROnPlateau is the one exception: it needs the metric passed in, e.g.:
# plateau_scheduler.step(validation_loss)


### 🎯 Interview Recap — Optimizers & Schedulers

**If you remember nothing else:** SGD steps against the gradient; momentum smooths zig-zag; Adam = momentum + per-parameter adaptive scaling + bias correction; the learning rate is the single most important hyperparameter; schedulers shrink LR over time.

**Q: What problem does momentum solve?**
A: It smooths out zig-zagging in narrow valleys by adding a "velocity" term that keeps moving in a direction the gradient has consistently pointed toward, which speeds up convergence.

**Q: Why is Adam usually the default choice over plain SGD?**
A: It gives every parameter its own adaptive effective learning rate based on that parameter's recent gradient history, so it typically needs far less manual learning-rate tuning to train well.

**Q: What happens if the learning rate is set too high?**
A: Updates overshoot the minimum and training can bounce around chaotically, or even diverge (loss shoots toward infinity/NaN).

**Q: What's the practical difference between Adam and AdamW?**
A: AdamW decouples `weight_decay` from Adam's adaptive gradient scaling, fixing a subtle interaction bug in the original Adam — it's now the standard default, especially for training transformers.

**Q: Why use a learning-rate scheduler instead of a constant learning rate?**
A: Large steps help make fast progress early in training, but the same large steps cause bouncing around a good minimum later on — shrinking the LR over time typically improves the final result.

<a id="datasets-dataloaders"></a>
## 9. `Dataset` & `DataLoader`

### Why these exist
Real datasets are often too large to fit in memory (or GPU memory) all at once, and training works better when data is fed in small random **batches** rather than either one sample at a time (slow, noisy) or the whole dataset at once (memory-prohibitive, and mathematically tends to generalize worse). PyTorch splits this responsibility into two clean, reusable pieces:

- **`Dataset`** answers one question: *"given an index `i`, give me sample `i`."* It doesn't know anything about batching, shuffling, or parallel loading.
- **`DataLoader`** wraps a `Dataset` and handles *everything about serving it efficiently*: grouping samples into batches, shuffling their order every epoch, and optionally loading the next batch on background worker processes while your GPU is busy with the current one (so the GPU is never left waiting on the CPU to prepare data).

**Real-world example:** ImageNet has over a million images — you'd never load all of them into RAM as tensors at once. A `Dataset` just knows how to fetch and preprocess image #482,003 when asked; the `DataLoader` decides "assemble a random batch of 64 such images" and keeps doing this, batch after batch, feeding your training loop.

### Writing a custom `Dataset`

Every custom `Dataset` needs exactly three methods:
- `__init__`: load or reference your raw data (e.g. a list of file paths, or an in-memory array)
- `__len__`: return the total number of samples — lets `DataLoader` know how many batches make up one epoch
- `__getitem__(i)`: return the single sample (and its label) at index `i`, as tensors

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class RegressionDataset(Dataset):
    # A minimal custom Dataset wrapping simple in-memory arrays.

    def __init__(self, n_samples=200):
        # In a real project this __init__ might read a CSV, list image file paths, etc.
        # Here we just generate synthetic data: y = 3x1 - 2x2 + noise
        torch.manual_seed(0)
        self.X = torch.randn(n_samples, 2)                                 # 2 input features per sample
        self.y = 3 * self.X[:, 0] - 2 * self.X[:, 1] + torch.randn(n_samples) * 0.1  # target

    def __len__(self):
        return len(self.X)                 # DataLoader calls this to know how many samples exist

    def __getitem__(self, idx):
        # DataLoader calls this once per index it needs -- return ONE sample (features, label)
        return self.X[idx], self.y[idx]

dataset = RegressionDataset(n_samples=200)
print("Dataset length:", len(dataset))          # calls __len__
sample_x, sample_y = dataset[5]                   # calls __getitem__(5) behind the scenes
print("Sample 5:", sample_x, sample_y)


### `DataLoader` — batching, shuffling, and parallel loading

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=32,     # how many samples per batch -- a key hyperparameter, discussed more below
    shuffle=True,       # reshuffle the order EVERY epoch -- crucial for training (see explanation below)
    num_workers=0,       # how many background processes pre-fetch batches (0 = load in the main process;
                          # >0 speeds up training when loading/preprocessing is slow, e.g. reading images from disk)
    drop_last=False,      # if the dataset size isn't divisible by batch_size, keep (True) or drop (False)
                           # the final, smaller leftover batch
)

# Iterating over a DataLoader gives you one batch at a time, ready to feed straight into a model
for batch_idx, (X_batch, y_batch) in enumerate(loader):
    print(f"batch {batch_idx}: X shape {X_batch.shape}, y shape {y_batch.shape}")
    if batch_idx == 2:      # just show the first few batches for this demo
        break

# 200 samples / batch_size 32 = 6 full batches + 1 partial batch of 8 (since drop_last=False)
print("\nTotal batches per epoch:", len(loader))


**Why `shuffle=True` for training matters:** if batches were always served in the same fixed order, the model could partly learn to exploit that *ordering* rather than the underlying pattern, and batches drawn from a narrow slice of the data (e.g. all one class in a row) create noisy, biased gradient estimates. Reshuffling every epoch fixes both problems. **For validation/test loaders, shuffling is unnecessary** (it doesn't affect correctness, just wastes a tiny bit of computation) and is conventionally left `False` so results are easy to compare and inspect in a stable order.

**Why `batch_size` matters:** it's a genuine trade-off, not just a speed knob:
- **Larger batches:** smoother, more accurate gradient estimates, better GPU utilization (more parallel work per step) — but each step gives "less frequent feedback", uses more memory, and can sometimes generalize slightly worse.
- **Smaller batches:** noisier gradients (which can actually help escape bad minima), less memory, but slower wall-clock training due to less GPU parallelism per step, and noisier loss curves.
- **Common practice:** pick the largest batch size that comfortably fits in GPU memory, often a power of 2 (32, 64, 128, 256...) for hardware efficiency, then tune from there.

### `transforms` — preprocessing that happens per-sample, on the fly

For image data especially, you rarely want to store already-preprocessed data — instead you apply preprocessing (resizing, converting to a tensor, normalizing, and often randomized data augmentation) **inside `__getitem__`**, so it happens fresh each time a sample is loaded. `torchvision.transforms` provides ready-made building blocks for this, chained together with `Compose`.

In [ ]:
# NOTE: this cell shows the STANDARD pattern used with real image datasets (needs torchvision installed).
# import torchvision.transforms as T
#
# transform = T.Compose([                    # Compose chains multiple transforms into one pipeline
#     T.Resize((224, 224)),                    # resize every image to the same fixed size
#     T.ToTensor(),                              # convert a PIL image (H,W,C), pixels 0-255 -> a
#                                                  # float tensor (C,H,W), pixels scaled to 0.0-1.0
#     T.Normalize(mean=[0.485, 0.456, 0.406],      # subtract the ImageNet mean...
#                 std=[0.229, 0.224, 0.225]),       # ...and divide by the ImageNet std, per channel.
#                                                     # This centers pixel values around 0, which
#                                                     # makes optimization noticeably easier/faster.
# ])
#
# A Dataset's __getitem__ would then do:
#   image = self.transform(raw_pil_image)

# You can plug a transform straight into a custom Dataset's __getitem__ just as easily:
class ImageLikeDataset(Dataset):
    def __init__(self, data, labels, transform=None):
        self.data, self.labels, self.transform = data, labels, transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        if self.transform:                    # apply preprocessing HERE, freshly, every time this is called
            sample = self.transform(sample)
        return sample, self.labels[idx]


### Splitting into train / validation / test sets

- **Train set:** what the model actually learns from (weights are updated using this data).
- **Validation set:** used *during* development to check progress, compare architectures, and tune hyperparameters — the model never directly learns from it, but you as the human make decisions based on it, so it's not a fully unbiased final judge.
- **Test set:** touched only ONCE, at the very end, to report a final, honest, unbiased performance number. If you tune anything based on test-set results, it silently stops being a fair test set.

In [ ]:
from torch.utils.data import random_split

full_dataset = RegressionDataset(n_samples=200)

train_size = int(0.7 * len(full_dataset))     # 70% for training
val_size   = int(0.15 * len(full_dataset))     # 15% for validation
test_size  = len(full_dataset) - train_size - val_size   # remaining 15% for testing

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)   # fixes the random split so it's reproducible
)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)   # no need to shuffle eval data
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)


### 🎯 Interview Recap — `Dataset` & `DataLoader`

**If you remember nothing else:** `Dataset` answers "give me sample `i`"; `DataLoader` handles batching/shuffling/parallel loading; shuffle training data, not eval data; batch size trades gradient smoothness and GPU use against memory and update frequency.

**Q: What three methods must a custom `Dataset` implement?**
A: `__init__`, `__len__`, and `__getitem__`.

**Q: Why shuffle the training set every epoch but not the validation/test set?**
A: Shuffling training data prevents the model from exploiting a fixed ordering and reduces bias in each batch's gradient estimate; shuffling eval data doesn't change correctness, so it's left off for stable, easy-to-compare results.

**Q: What's the actual division of responsibility between `Dataset` and `DataLoader`?**
A: `Dataset` defines how to fetch and preprocess ONE sample given an index; `DataLoader` wraps a `Dataset` to serve it efficiently in shuffled batches, optionally using background worker processes so the GPU isn't left waiting.

**Q: What's the trade-off of increasing batch size?**
A: Smoother, more accurate gradients and better GPU utilization, at the cost of more memory use, less frequent weight updates per epoch, and sometimes slightly worse generalization.

**Q: Why must the test set stay untouched until the very end of a project?**
A: Any tuning decision made based on a dataset's results introduces bias toward it — the moment you tune based on the test set, it stops being an unbiased measure of real-world performance.

<a id="training-loop"></a>
## 10. The Training Loop & Evaluation Loop

We've now met every ingredient — tensors, autograd, `nn.Module`, loss functions, optimizers, `DataLoader` — so this section assembles them into the loop you'll write (with only minor variations) in every PyTorch project you ever build.

### Anatomy of one training step, line by line

```python
model.train()                        # (1) switch the model into "training mode"
for X_batch, y_batch in train_loader:  # (2) get one batch at a time from the DataLoader
    X_batch = X_batch.to(device)         # (3) move data to the same device as the model
    y_batch = y_batch.to(device)
    optimizer.zero_grad()                 # (4) clear old gradients (remember: they ACCUMULATE otherwise!)
    y_pred = model(X_batch)                # (5) forward pass: get the model's current predictions
    loss = loss_fn(y_pred, y_batch)         # (6) compute how wrong those predictions are
    loss.backward()                          # (7) backward pass: autograd fills in .grad for every weight
    optimizer.step()                          # (8) update every weight using its gradient
```
That's genuinely the entire algorithm. Everything else in deep learning — CNNs, transformers, GANs — is decoration around this eight-line skeleton: a different `model`, sometimes a different `loss_fn`, but the *loop itself* barely changes.

### `model.train()` vs. `model.eval()` — why this line matters

A small number of layers — most importantly **`Dropout`** and **`BatchNorm`** (both covered fully in [Regularization](#regularization)) — are supposed to behave *differently* during training versus during evaluation/inference:
- **Dropout** randomly disables neurons *only while training* (to fight overfitting); at evaluation time you want the model's full, deterministic capacity switched back on.
- **BatchNorm** normalizes using the *current batch's* statistics while training, but uses a stable *running average* of statistics collected during training when evaluating — because at inference time you might not even have a "batch" (e.g. predicting on a single sample).

`model.train()` and `model.eval()` don't run any computation themselves — they just flip an internal `self.training` flag (recursively, on every layer inside the model), which those specific layers check internally. **Forgetting to call `model.eval()` before evaluating is a classic, quietly-wrong-results bug** — your evaluation numbers will be noisier and slightly worse than they should be, because Dropout will still be randomly zeroing neurons.

### The evaluation loop

Almost identical to training, with three deliberate differences: `model.eval()` instead of `model.train()`, no `optimizer.zero_grad()` / `.backward()` / `.step()` (we are not learning here, only measuring), and the whole thing wrapped in `torch.no_grad()` since we don't need gradients at all — this also makes evaluation noticeably faster and uses less memory.

```python
model.eval()                           # (1) switch to evaluation mode
total_loss, correct, total = 0, 0, 0
with torch.no_grad():                   # (2) no need to track gradients -- we're not training
    for X_batch, y_batch in val_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)            # (3) forward pass only -- no backward, no optimizer step
        loss = loss_fn(y_pred, y_batch)
        total_loss += loss.item() * len(X_batch)
        # ... accumulate accuracy or other metrics here ...
```

### Overfitting vs. underfitting — the most important diagnostic skill in deep learning

- **Underfitting:** both training loss AND validation loss stay high — the model is too simple, or hasn't trained long enough, to capture the pattern at all.
- **A good fit:** training loss and validation loss both decrease and stay close together.
- **Overfitting:** training loss keeps decreasing, but validation loss stops improving or starts *increasing* — the model has started memorizing quirks specific to the training data (including its noise) rather than learning the general pattern, so it performs worse on data it hasn't seen.

This is precisely *why* we keep a separate validation set: **training loss alone can be dangerously misleading** — a model can have a near-zero training loss while being nearly useless on new data. Watching the *gap* between train and validation loss, epoch by epoch, is the main tool for catching this early (and is exactly what [early stopping](#regularization) automates).

---
## 🏗️ Full Worked Example: An ANN (Artificial Neural Network) Classifier, Start to Finish

We'll now build, train, and evaluate a real (if small) neural network on the classic **"two moons"** dataset — two interleaving crescent shapes that are **not** linearly separable, meaning a single straight line can never fully separate the two classes. This is the perfect toy example for a first ANN because it forces the network to actually use its hidden layers and non-linear activations (a plain `Linear` layer alone could never solve this — exactly the point made in the [`nn.Module` section](#nn-module) about why activations matter).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_moons
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- 1. Data ----
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=42)   # 1000 2-D points, 2 classes
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)               # shape (1000, 1) to match model output

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=y_np, cmap="coolwarm", s=10)
plt.title("Two-moons dataset (not linearly separable)")
plt.show()

# Wrap the raw tensors in a TensorDataset -- a ready-made Dataset for when your data is ALREADY
# fully-loaded tensors in memory (no custom __getitem__ logic needed, unlike Section 9's example)
full_dataset = TensorDataset(X, y)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_ds, test_ds = torch.utils.data.random_split(full_dataset, [train_size, test_size],
                                                     generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)


In [ ]:
# ---- 2. Model: a small ANN (a.k.a. Multi-Layer Perceptron / MLP) ----
class ANNClassifier(nn.Module):
    def __init__(self, in_features=2, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),  # input layer -> hidden layer
            nn.ReLU(),                        # non-linearity -- without this, stacking Linears is pointless
            nn.Linear(hidden, hidden),         # hidden -> hidden (a second hidden layer, makes it "deep")
            nn.ReLU(),
            nn.Linear(hidden, 1),               # hidden -> a SINGLE output logit (raw score, not yet a probability)
        )

    def forward(self, x):
        return self.net(x)     # returns raw logits -- BCEWithLogitsLoss will apply sigmoid internally

model = ANNClassifier(in_features=2, hidden=16).to(device)
print(model)

loss_fn = nn.BCEWithLogitsLoss()                                  # binary classification -> BCE, see Section 7
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)          # Adam, see Section 8


In [ ]:
# ---- 3. The training + evaluation loop, with metrics tracked every epoch ----
def evaluate(model, loader):
    # Runs the model over `loader` in eval mode and returns (average_loss, accuracy).
    model.eval()                                # switch off Dropout/BatchNorm's training behavior
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():                        # no gradients needed for evaluation -- faster, less memory
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            total_loss += loss.item() * len(X_batch)          # un-average the batch loss, sum properly
            preds = (torch.sigmoid(logits) > 0.5).float()       # convert raw logits -> 0/1 predicted class
            correct += (preds == y_batch).sum().item()
            total += len(X_batch)
    return total_loss / total, correct / total

train_losses, test_losses, test_accuracies = [], [], []

for epoch in range(100):
    # ---- training phase ----
    model.train()                                 # switch ON Dropout/BatchNorm's training behavior
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()          # (1) clear old gradients
        logits = model(X_batch)          # (2) forward pass
        loss = loss_fn(logits, y_batch)    # (3) compute loss
        loss.backward()                     # (4) backward pass -- compute gradients
        optimizer.step()                     # (5) update weights

        running_loss += loss.item() * len(X_batch)

    train_loss = running_loss / len(train_ds)
    test_loss, test_acc = evaluate(model, test_loader)     # ---- evaluation phase ----

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    if epoch % 10 == 0 or epoch == 99:
        print(f"epoch {epoch:3d} | train_loss {train_loss:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.3f}")


In [ ]:
# ---- 4. Visualize training curves (this is exactly how you'd spot over/underfitting in practice) ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(train_losses, label="train loss")
axes[0].plot(test_losses, label="test loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title("Loss curves (train and test tracking together = good sign)")

axes[1].plot(test_accuracies, color="green")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("test accuracy")
axes[1].set_title(f"Final test accuracy: {test_accuracies[-1]:.1%}")

plt.tight_layout(); plt.show()


In [ ]:
# ---- 5. Visualize the learned decision boundary -- a great sanity check for 2-D toy problems ----
model.eval()
xx, yy = torch.meshgrid(torch.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 200),
                          torch.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 200), indexing="xy")
grid = torch.stack([xx.ravel(), yy.ravel()], dim=1).to(device)

with torch.no_grad():
    probs = torch.sigmoid(model(grid)).cpu().reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, probs, levels=20, cmap="coolwarm", alpha=0.6)   # shaded predicted-probability regions
plt.scatter(X[:, 0], X[:, 1], c=y_np, cmap="coolwarm", edgecolors="k", s=15)
plt.title("Learned decision boundary")
plt.show()

# Notice the boundary is CURVED, not a straight line -- this is the hidden layers + ReLU non-linearities
# at work, exactly as promised back in the nn.Module section.


### 🎯 Interview Recap — Training & Evaluation Loops

**If you remember nothing else:** the 8-line training loop skeleton is universal; `model.train()`/`model.eval()` flips Dropout/BatchNorm behavior (it computes nothing itself); evaluation is wrapped in `no_grad()`; overfitting = training loss keeps falling while validation loss plateaus or rises.

**Q: Why call `model.eval()` before evaluating, and what does it actually change?**
A: It flips layers like Dropout and BatchNorm out of training behavior — Dropout stops randomly zeroing neurons, and BatchNorm switches from the current batch's statistics to a stable running average — it doesn't perform any computation itself, just a mode flag.

**Q: Why wrap the evaluation loop in `torch.no_grad()`?**
A: We're not training, so we don't need gradients — skipping graph construction saves memory and makes evaluation faster.

**Q: How do you spot overfitting from training curves?**
A: Training loss keeps decreasing while validation loss plateaus or starts increasing — the gap between the two widens over epochs.

**Q: What's the difference between an epoch and an iteration/batch?**
A: An iteration is one forward + backward + weight update using a single mini-batch; an epoch is one complete pass over the entire training dataset, made up of many iterations.

**Q: Why can a model have near-zero training loss but still perform poorly on new data?**
A: It has overfit — memorized specifics (including noise) of the training set rather than learning the general underlying pattern, so it fails to generalize.

<a id="cnn"></a>
# Part C — Architectures
## 11. Convolutional Neural Networks (CNN)

### Why CNNs exist — the problem with feeding images into a plain ANN
Imagine feeding a 224×224 color image into the `nn.Linear`-only ANN from the previous section. You'd first flatten it into a single vector of `224 * 224 * 3 = 150,528` numbers. A first hidden layer of even modest size (say 512 neurons) would then need `150,528 * 512 ≈ 77 million` weights — for ONE layer, on ONE small image size. Beyond the sheer parameter explosion, flattening also **throws away spatial structure**: a `Linear` layer treats pixel (0,0) and pixel (223,223) as no more related than any other pair, even though nearby pixels in a real image are strongly related (they're often part of the same edge, texture, or object), and a `Linear` layer also has no way to recognize "the same pattern, just shifted to a different part of the image" without learning it separately at every position.

**CNNs fix both problems** with one idea: instead of connecting every input pixel to every output neuron, slide a **small, shared filter** across the image, reusing the *same* small set of weights at every position.

### The convolution operation, in intuition, then in math

**Intuition first:** think of a kernel (filter) as a small stencil — say 3×3 — that represents a pattern to look for (e.g. "a vertical edge"). You slide this stencil across every position in the image, and at each position you compute *how strongly the image matches the stencil right there* (via a dot product). The result is a new grid — a **feature map** — that lights up wherever that pattern was found.

**The math:** for a kernel $K$ of size $k \times k$ and an input $I$, the output at position $(i,j)$ is:
$$\text{out}(i,j) = \sum_{m=0}^{k-1}\sum_{n=0}^{k-1} I(i+m,\, j+n)\cdot K(m,n) \;+\; b$$
— literally "overlay the kernel here, multiply every overlapping pair, sum it all up, add a bias." Slide this window across the whole image and you get the full feature map.

**Two properties this buys you, for free:**
- **Parameter sharing:** the *same* 3×3 = 9 weights (plus 1 bias) are reused at every position in the image, instead of learning separate weights per pixel — this is why CNNs have dramatically fewer parameters than an equivalent `Linear` layer.
- **Translation invariance:** because the same filter is applied everywhere, a pattern (say, a cat's ear) is recognized wherever it appears in the image, not just in one specific spot the network happened to memorize.

In [ ]:
import torch
import torch.nn as nn

# A tiny hand-made example: detecting a vertical edge with a manually-chosen (not learned) kernel
image = torch.tensor([[10., 10., 10., 0., 0., 0.],
                       [10., 10., 10., 0., 0., 0.],
                       [10., 10., 10., 0., 0., 0.],
                       [10., 10., 10., 0., 0., 0.]])   # a bright square on the left, dark on the right

vertical_edge_kernel = torch.tensor([[ 1., 0., -1.],    # classic hand-designed "Sobel-like" edge detector
                                      [ 1., 0., -1.],
                                      [ 1., 0., -1.]])

conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False)
with torch.no_grad():
    conv.weight[:] = vertical_edge_kernel     # manually set the kernel instead of random init, for this demo

feature_map = conv(image.unsqueeze(0).unsqueeze(0))   # Conv2d needs shape (batch, channels, H, W)
print(feature_map.squeeze())
# Notice the large values right where the bright/dark boundary is -- the kernel "lit up" exactly
# at the vertical edge, and stayed near zero over flat, uniform regions. In a REAL CNN, the network
# LEARNS kernels like this itself via backprop, instead of us hand-designing them.


### Stride, padding, and the output-size formula

- **Stride:** how many pixels the kernel jumps between positions. Stride 1 = slide one pixel at a time (dense, large output). Stride 2 = skip every other position (output roughly halves in each dimension) — a cheap way to downsample.
- **Padding:** how many extra (usually zero-valued) pixels to add around the border before sliding the kernel. Without padding, the output shrinks every time you apply a convolution (border pixels get "seen" by fewer kernel positions than center pixels), which limits how deep a network can go. `padding = kernel_size // 2` with stride 1 keeps the output the *same* height/width as the input — this is called "same" padding.

**The formula** for one spatial dimension:
$$\text{output size} = \left\lfloor \frac{\text{input size} + 2\times\text{padding} - \text{kernel size}}{\text{stride}} \right\rfloor + 1$$

In [ ]:
def conv_output_size(input_size, kernel_size, stride=1, padding=0):
    return (input_size + 2 * padding - kernel_size) // stride + 1

print("28x28 input, 3x3 kernel, stride 1, no padding ->", conv_output_size(28, 3, 1, 0))     # 26 (shrinks)
print("28x28 input, 3x3 kernel, stride 1, padding 1   ->", conv_output_size(28, 3, 1, 1))     # 28 ('same')
print("28x28 input, 3x3 kernel, stride 2, padding 1   ->", conv_output_size(28, 3, 2, 1))     # 14 (halved)

# Verifying with a real nn.Conv2d
conv_same = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
x = torch.rand(1, 1, 28, 28)          # (batch=1, channels=1, height=28, width=28)
print("Conv2d(k=3,s=1,p=1) output shape:", conv_same(x).shape)    # torch.Size([1, 8, 28, 28]) -- H,W unchanged


### Multiple channels and multiple filters

A color image has 3 input **channels** (Red, Green, Blue). A kernel for such an image is actually 3×3×3 (one 3×3 slice per input channel), and its outputs across all 3 channels are summed into a single number per position — so ONE filter still produces ONE output channel, regardless of how many input channels it reads.

To detect *many different patterns* (edges, colors, textures...) at once, a conv layer uses **many filters in parallel** — `out_channels` in `nn.Conv2d` — each producing its own feature map, stacked together as the layer's output channels.

In [ ]:
conv_multi = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
# in_channels=3   -> expects an RGB image
# out_channels=16 -> learns 16 DIFFERENT filters, each looking for a different pattern,
#                    producing 16 separate feature maps stacked together
x = torch.rand(1, 3, 32, 32)     # a batch of 1 RGB 32x32 image
out = conv_multi(x)
print(out.shape)                  # torch.Size([1, 16, 32, 32]) -- 16 output channels now

# Parameter count check: each of the 16 filters has 3 (in_channels) * 3 * 3 (kernel) weights + 1 bias
expected_params = 16 * (3 * 3 * 3) + 16
actual_params = sum(p.numel() for p in conv_multi.parameters())
print("expected params:", expected_params, "| actual:", actual_params)


### Pooling — downsampling feature maps

Pooling shrinks a feature map's height/width (without any learnable parameters), which reduces computation for later layers and adds a small amount of translation invariance (a pattern shifted by 1-2 pixels still produces roughly the same pooled output).

- **Max pooling** (`nn.MaxPool2d`): takes the *maximum* value in each window — "did this pattern appear strongly ANYWHERE in this small region?" The overwhelmingly common choice in modern CNNs.
- **Average pooling** (`nn.AvgPool2d`): takes the *average* value in each window — smoother, sometimes used at the very end of a network (`AdaptiveAvgPool2d`) to collapse each feature map down to a single number regardless of its input size.

In [ ]:
x = torch.tensor([[[[1., 2., 5., 6.],
                     [3., 4., 7., 8.],
                     [9., 10., 13., 14.],
                     [11., 12., 15., 16.]]]])   # shape (1, 1, 4, 4)

maxpool = nn.MaxPool2d(kernel_size=2, stride=2)   # non-overlapping 2x2 windows
print(maxpool(x))
# Each 2x2 corner block's max: top-left block [1,2,3,4]->4, top-right [5,6,7,8]->8,
# bottom-left [9,10,11,12]->12, bottom-right [13,14,15,16]->16
# Output shape halves in both height and width: (1,1,2,2)


### Putting it together: the classic CNN architecture pattern

A typical CNN repeats blocks of **[Conv → Activation → (optional) Pooling]** to progressively build up more abstract feature maps (early layers detect edges/textures, deeper layers detect shapes/parts, the deepest layers detect whole objects), then **flattens** the final feature maps into a vector and feeds them through one or more `Linear` layers to produce the final prediction.

In [ ]:
class SimpleCNN(nn.Module):
    # A small CNN for 28x28 grayscale images (e.g. FashionMNIST-style), classifying into 10 classes.

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 1 input channel (grayscale) -> 16 feature maps, then shrink spatially
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # (1,28,28) -> (16,28,28), padding keeps size
            nn.BatchNorm2d(16),                             # normalizes activations, see Regularization section
            nn.ReLU(),
            nn.MaxPool2d(2),                                  # (16,28,28) -> (16,14,14)

            # Block 2: go deeper (more channels = more learned patterns) while shrinking further
            nn.Conv2d(16, 32, kernel_size=3, padding=1),      # (16,14,14) -> (32,14,14)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                                    # (32,14,14) -> (32,7,7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                       # (32,7,7) -> a single vector of 32*7*7 = 1568 numbers
            nn.Linear(32 * 7 * 7, 128),           # standard fully-connected layers from here, like an ANN
            nn.ReLU(),
            nn.Dropout(0.3),                        # regularization, see dedicated section
            nn.Linear(128, num_classes),             # final layer: one raw score (logit) per class
        )

    def forward(self, x):
        x = self.features(x)       # extract visual features
        x = self.classifier(x)      # classify based on those features
        return x

model = SimpleCNN(num_classes=10)
dummy_batch = torch.rand(4, 1, 28, 28)      # a fake batch of 4 grayscale 28x28 images
print("Output shape:", model(dummy_batch).shape)   # torch.Size([4, 10]) -- 10 raw class scores per image
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


### Training on a real image dataset

This uses the exact same [training loop from Section 10](#training-loop) — a CNN is trained no differently from the ANN example, only the model architecture and the data are different. The code below shows the standard, real-world pattern; it needs internet access to download the dataset the first time, so we also show a synthetic version you can run right now, offline, to see the shapes flow through correctly.

In [ ]:
# ---- The STANDARD real-world pattern (needs internet on first run to download the dataset) ----
# import torchvision
# import torchvision.transforms as T
#
# transform = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])   # scale pixels to roughly [-1, 1]
# train_data = torchvision.datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
# test_data  = torchvision.datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)
# train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
# test_loader  = DataLoader(test_data, batch_size=64, shuffle=False)
#
# model = SimpleCNN(num_classes=10).to(device)
# loss_fn = nn.CrossEntropyLoss()                              # 10 mutually-exclusive classes -> CrossEntropy
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# # ... then reuse the EXACT training loop from Section 10 ...

# ---- OFFLINE, runnable version using synthetic "image-shaped" random data, to verify the plumbing ----
from torch.utils.data import TensorDataset

fake_images = torch.randn(200, 1, 28, 28)              # 200 fake grayscale 28x28 "images"
fake_labels = torch.randint(0, 10, (200,))               # 200 random labels from 10 classes

fake_loader = DataLoader(TensorDataset(fake_images, fake_labels), batch_size=32, shuffle=True)

model = SimpleCNN(num_classes=10)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
X_batch, y_batch = next(iter(fake_loader))     # grab just ONE batch to demonstrate a single training step
optimizer.zero_grad()
logits = model(X_batch)
loss = loss_fn(logits, y_batch)
loss.backward()
optimizer.step()
print("Ran one full training step successfully. Loss:", loss.item())


**Real-world example:** this exact `Conv → BatchNorm → ReLU → Pool` block pattern, repeated and scaled up, is the backbone of famous architectures like **ResNet**, **VGG**, and **EfficientNet** — the main differences in more advanced CNNs are usually *how many* blocks, *how wide* each block is, and clever connectivity tricks like **residual/skip connections** (letting a block's input skip ahead and be added to its output, which makes very deep networks much easier to train by giving gradients a direct path backward).

### 🎯 Interview Recap — CNNs

**If you remember nothing else:** convolution = a shared, sliding filter → parameter sharing + translation invariance; stride/padding control output size (`(in+2p-k)/s + 1`); pooling downsamples with no learnable parameters; the classic pattern is `Conv → BatchNorm → ReLU → Pool`, repeated, then `Flatten → Linear`.

**Q: Why not just flatten an image and feed it into `Linear` layers?**
A: It throws away spatial structure — nearby pixels' relationships — and needs a huge number of parameters (tens of millions for even one layer on a modest image), because every pixel would need its own weight instead of a small filter being reused across positions.

**Q: What is "parameter sharing" in a CNN, and why does it matter?**
A: The same small kernel's weights are reused at every position across the image, instead of learning separate weights per location — this makes CNNs far more parameter-efficient than a fully-connected layer and gives them translation invariance (a pattern is recognized wherever it appears).

**Q: What does the `padding` argument in `Conv2d` control, and what is "same" padding?**
A: How many extra (usually zero) pixels are added around the input's border before sliding the kernel; using `padding = kernel_size // 2` with stride 1 keeps the output the same height/width as the input.

**Q: What's the difference between `in_channels` and `out_channels` in `Conv2d`?**
A: `in_channels` is how many channels the input has (e.g. 3 for RGB); `out_channels` is how many different learned filters run in parallel, each producing its own feature map — every filter reads across all input channels regardless of `out_channels`.

**Q: Why use max pooling?**
A: It downsamples a feature map — reducing computation for later layers — by keeping only the strongest activation in each small region, which also adds a bit of robustness to small shifts in the input.

**Q: Given a 28×28 input, a 3×3 kernel, stride 1, and padding 1, what's the output size?**
A: 28×28 — with `padding = kernel_size // 2 = 1` and stride 1, the output stays the same size as the input ("same" padding).

<a id="rnn"></a>
## 12. Recurrent Neural Networks — RNN, LSTM, GRU

### Why RNNs exist
CNNs and plain ANNs expect a **fixed-size** input and have no built-in notion of "order" or "what came before." But a lot of real data is **sequential**, where order is essential and length varies: a sentence, a stock price history, sensor readings over time, audio. Feeding a whole sequence into a `Linear` layer would require a fixed length and would ignore that position 5 in the sequence is meaningfully *after* position 4. RNNs are built specifically to process a sequence **one step at a time while carrying forward a memory of everything seen so far.**

### Core intuition: a memory that updates as it reads
Picture reading a sentence word by word while keeping a running mental summary of "what the sentence has been about so far." Each new word updates that summary rather than replacing it — you don't forget the beginning of the sentence the moment you read the next word. That running summary is the RNN's **hidden state** $h_t$, and the same update rule is applied at every time step, using the *same* shared weights (just like convolution's shared kernel — this is the sequential analog of parameter sharing).

### The math
At every time step $t$, given the current input $x_t$ and the previous hidden state $h_{t-1}$:
$$h_t = \tanh(W_{xh}\, x_t + W_{hh}\, h_{t-1} + b)$$
The *same* weight matrices $W_{xh}$ (input → hidden) and $W_{hh}$ (hidden → hidden) are reused at every single time step — this is what makes an RNN able to handle sequences of *any* length with a fixed number of parameters. Depending on the task, you either use every $h_t$ (e.g. tagging every word) or just the final $h_T$ (e.g. classifying a whole sentence).

In [ ]:
import torch
import torch.nn as nn

# nn.RNN expects input shaped (batch, seq_len, input_size) when batch_first=True
rnn = nn.RNN(input_size=1, hidden_size=8, batch_first=True)

x = torch.rand(4, 10, 1)      # batch of 4 sequences, 10 time steps each, 1 feature per time step

output, h_n = rnn(x)
print("output shape:", output.shape)   # (4, 10, 8) -- the hidden state h_t at EVERY time step
print("h_n shape   :", h_n.shape)      # (1, 4, 8)  -- just the FINAL hidden state (1 = num_layers)
# output[:, -1, :] and h_n[0] are actually the same values -- the last time step's hidden state,
# available both ways for convenience depending on what your downstream code needs.


### The vanishing/exploding gradient problem

Training an RNN uses **Backpropagation Through Time (BPTT):** conceptually, "unroll" the RNN into one very deep feed-forward network — one layer per time step, all sharing the same weights — then apply ordinary backpropagation. The problem: computing the gradient for an early time step requires **multiplying together the same weight matrix (roughly) once per time step it flows back through**. If that repeated factor is even slightly less than 1, the gradient shrinks *exponentially* the further back it flows (**vanishing** — early time steps effectively stop learning anything, so the network can't learn long-range dependencies). If it's slightly more than 1, the gradient grows *exponentially* instead (**exploding** — training becomes unstable, loss can spike to `NaN`). This is the single biggest limitation of plain (`nn.RNN`, sometimes called "vanilla RNN") RNNs, and it's exactly the problem LSTM and GRU were invented to fix.

### LSTM (Long Short-Term Memory) — fixing vanishing gradients with gates

**Core intuition:** alongside the hidden state, an LSTM keeps a second running value called the **cell state** $c_t$, which acts like a conveyor belt running straight through time, updated mostly by simple *addition* rather than repeated multiplication — addition doesn't shrink gradients the way repeated matrix multiplication does, so information (and gradients, flowing backward) can travel much further along this "highway" without vanishing. Three learned **gates** (each a small neural network outputting values between 0 and 1, acting like a dimmer switch — "0 = block completely" to "1 = let everything through") control what happens to that conveyor belt at each step:

- **Forget gate** $f_t$: "how much of the OLD cell state should I keep vs. throw away?"
- **Input gate** $i_t$: "how much of the NEW candidate information should I add in?"
- **Output gate** $o_t$: "how much of the cell state should I actually expose as this step's hidden output?"

$$f_t = \sigma(W_f[h_{t-1}, x_t] + b_f) \qquad i_t = \sigma(W_i[h_{t-1}, x_t] + b_i) \qquad o_t = \sigma(W_o[h_{t-1}, x_t] + b_o)$$
$$\tilde c_t = \tanh(W_c[h_{t-1}, x_t] + b_c) \qquad\qquad c_t = f_t \odot c_{t-1} + i_t \odot \tilde c_t \qquad\qquad h_t = o_t \odot \tanh(c_t)$$

($\odot$ means element-wise multiplication; $\sigma$ is sigmoid, squashing gate values to (0,1).) **Real-world example:** reading a long document, an LSTM's forget gate lets it drop irrelevant filler information (like "the weather that day") while its cell state keeps carrying forward a key fact from paragraph 1 (like a character's name) all the way to paragraph 20, because addition — not repeated multiplication — is what's preserving it.

In [ ]:
lstm = nn.LSTM(input_size=1, hidden_size=8, batch_first=True)

x = torch.rand(4, 10, 1)
output, (h_n, c_n) = lstm(x)     # LSTM returns TWO states: hidden state h_n AND cell state c_n
print("output shape:", output.shape)   # (4, 10, 8) -- hidden state at every time step
print("h_n shape   :", h_n.shape)       # (1, 4, 8)  -- final hidden state
print("c_n shape   :", c_n.shape)       # (1, 4, 8)  -- final cell state (the "conveyor belt" value)


### GRU (Gated Recurrent Unit) — a simpler, faster alternative

GRU merges the forget and input gates into a single **update gate**, and drops the separate cell state entirely (folding its role into the hidden state), using a **reset gate** to decide how much past information to ignore when computing a new candidate hidden state. Fewer gates means fewer parameters and slightly faster training, and in practice GRU often performs comparably to LSTM. **Rule of thumb:** LSTM and GRU are both reasonable defaults over a plain RNN; try GRU first for speed, switch to LSTM if you need a bit more capacity or are matching a paper/architecture that specifies it.

In [ ]:
gru = nn.GRU(input_size=1, hidden_size=8, batch_first=True)
output, h_n = gru(x)          # GRU returns only ONE state (no separate cell state, unlike LSTM)
print("output shape:", output.shape)   # (4, 10, 8)
print("h_n shape   :", h_n.shape)      # (1, 4, 8)


---
## 🏗️ Full Worked Example: Forecasting a Sine Wave with an LSTM

A classic, fully offline, runnable sequence task: given the last 20 points of a sine wave, predict the next point. This requires the model to learn the *pattern* of oscillation, not just memorize fixed values — a clean way to see an RNN's "memory" in action.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)

# ---- 1. Generate the raw sequence: a noisy sine wave ----
t = np.linspace(0, 100, 1000)
series = np.sin(t) + np.random.normal(0, 0.05, size=len(t))   # a sine wave with a little noise

# ---- 2. Turn the raw sequence into (input_window -> next_value) training pairs via a sliding window ----
window_size = 20
X_seq, y_seq = [], []
for i in range(len(series) - window_size):
    X_seq.append(series[i : i + window_size])   # the 20 points the model gets to see
    y_seq.append(series[i + window_size])        # the very next point it must predict

X_seq = torch.tensor(np.array(X_seq), dtype=torch.float32).unsqueeze(-1)   # (num_samples, 20, 1)
y_seq = torch.tensor(np.array(y_seq), dtype=torch.float32).unsqueeze(-1)   # (num_samples, 1)

# Chronological train/test split -- for TIME SERIES, never shuffle across the train/test boundary!
# (shuffling would let the model "see the future" during training via nearby overlapping windows)
split = int(0.8 * len(X_seq))
X_train, y_train = X_seq[:split], y_seq[:split]
X_test,  y_test  = X_seq[split:], y_seq[split:]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
print("train sequences:", len(X_train), "| test sequences:", len(X_test))


In [ ]:
# ---- 3. Model: LSTM followed by a Linear layer to produce a single predicted number ----
class SineLSTM(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)     # maps the final hidden state -> one predicted value

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)     # we only need the FINAL hidden state to predict "what's next"
        out = self.fc(h_n[-1])          # h_n[-1]: hidden state of the last LSTM layer, shape (batch, hidden)
        return out                       # shape (batch, 1)

model = SineLSTM(hidden_size=32)
loss_fn = nn.MSELoss()                                  # predicting a continuous number -> MSE
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ---- 4. Train (identical training loop shape as every previous section) ----
for epoch in range(30):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(X_batch)
    if epoch % 5 == 0 or epoch == 29:
        print(f"epoch {epoch:2d} | train MSE {running_loss/len(X_train):.5f}")


In [ ]:
# ---- 5. Evaluate + visualize predictions on the held-out (future) test portion ----
model.eval()
with torch.no_grad():
    test_preds = model(X_test).squeeze()

plt.figure(figsize=(9, 4))
plt.plot(y_test.squeeze().numpy(), label="actual", linewidth=2)
plt.plot(test_preds.numpy(), label="predicted", linestyle="--")
plt.title("LSTM sine-wave forecast (test/future portion)")
plt.xlabel("time step"); plt.ylabel("value"); plt.legend(); plt.show()

test_mse = loss_fn(test_preds, y_test.squeeze()).item()
print(f"Test MSE: {test_mse:.5f}")


**Real-world example:** this exact "sliding window → LSTM → predict next value" pattern is the backbone of practical time-series forecasting — stock price prediction, demand forecasting, sensor anomaly detection — and the "predict the next token" framing is also, at heart, the same task language models solve, just with words/sub-words instead of numbers (though modern language models use [Transformers](#transformers), covered next, rather than RNNs, largely *because* of the vanishing-gradient and sequential-processing limitations discussed above).

### 🎯 Interview Recap — RNN, LSTM, GRU

**If you remember nothing else:** the hidden state carries memory forward using the *same* shared weights at every time step; vanishing/exploding gradients come from repeated multiplication through Backpropagation Through Time; LSTM's cell state is a mostly-additive "highway" controlled by gates; GRU is a simpler, faster LSTM alternative.

**Q: Why can't you just use a CNN or `Linear` layer for sequential data like text or time series?**
A: They expect fixed-size input and have no built-in notion of order; an RNN processes one step at a time while carrying forward a memory (hidden state) of everything seen so far, handling variable-length sequences with a fixed number of parameters.

**Q: What causes the vanishing gradient problem in plain RNNs?**
A: Backpropagation Through Time multiplies by roughly the same weight matrix once per time step the gradient flows back through; if that factor is less than 1, the gradient shrinks exponentially over long sequences, so early time steps effectively stop learning.

**Q: How does LSTM fix the vanishing gradient problem?**
A: It maintains a cell state updated mostly by addition — via forget/input gates — rather than repeated multiplication, giving gradients a more direct path backward through time.

**Q: What do the three gates in an LSTM each control?**
A: The forget gate controls how much of the old cell state to keep, the input gate controls how much new candidate information to add, and the output gate controls how much of the cell state becomes this step's hidden output.

**Q: What's the practical difference between LSTM and GRU?**
A: GRU merges the forget and input gates into a single update gate and has no separate cell state, giving it fewer parameters and slightly faster training, often with comparable performance to LSTM in practice.

**Q: In `nn.LSTM`, what's the difference between the returned `output` and `h_n`?**
A: `output` contains the hidden state at every time step; `h_n` is just the final time step's hidden state — for a single-layer, single-direction LSTM, `output[:, -1, :]` and `h_n[0]` are the same values.

<a id="transformers"></a>
## 13. Attention & Transformers

### Why Transformers exist — RNNs' remaining problems
RNNs (previous section) process a sequence **one step at a time**, which creates two real problems: (1) step $t$ can't be computed until step $t-1$ finishes, so training can't be parallelized across the sequence length — painfully slow on modern GPUs, which are built for parallel work; and (2) even LSTM's improved "highway" still has to relay information through every intermediate step, so *very* long-range dependencies (e.g. connecting word 1 and word 500) can still degrade. **Transformers solve both problems with one idea: let every position look directly at every other position, all at once, with no step-by-step relay required.**

### Core intuition: Query, Key, Value — a library-search analogy
Imagine each word in a sentence submits a **Query** ("what am I looking for context-wise?"), and every word (including itself) offers up a **Key** ("here's what I contain, in searchable-summary form") and a **Value** ("here's the actual content I'd contribute if you find me relevant"). Every word compares its Query against every other word's Key to get a relevance score, turns those scores into weights (via softmax, so they sum to 1), and then builds its new representation as a **weighted blend of every word's Value**, weighted most heavily toward the most relevant ones. This is called **self-attention**, and it happens for every word, simultaneously.

**Real-world example:** in *"The animal didn't cross the street because it was too tired,"* the word "it" needs to figure out what it refers to. Self-attention lets "it" directly query every other word; "animal" ends up with by far the highest relevance score (a street can't be "tired"), so "it"'s new representation ends up blended heavily toward "animal"'s Value — effectively resolving the reference, without any step-by-step relay through "didn't," "cross," "the," etc.

### The math: Scaled Dot-Product Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Reading this left to right:
1. $Q$ (queries), $K$ (keys), $V$ (values) are all produced from the input embeddings via separate **learned** linear projections — the model learns what to search for, what to advertise, and what to actually deliver.
2. $QK^\top$ computes a **relevance score** between every query and every key, via dot product (a dot product is naturally large when two vectors point in a similar direction — a natural, cheap "similarity" measure).
3. Dividing by $\sqrt{d_k}$ (the dimensionality of each key vector) **prevents the dot products from growing too large** as $d_k$ increases — without this scaling, large dot products push softmax into a region with extremely tiny gradients almost everywhere (a "saturated" softmax), which would badly hurt learning.
4. `softmax` turns the scores for each query into a clean probability distribution over all positions (summing to 1) — "how much attention should I pay to each position?"
5. Multiplying by $V$ produces the final output: a weighted blend of every position's Value vector, weighted by that attention distribution.

In [ ]:
import torch
import torch.nn as nn
import math

def scaled_dot_product_attention(Q, K, V):
    # Q, K, V shape: (batch, seq_len, d_k)
    d_k = Q.shape[-1]                                          # the dimensionality of each key/query vector
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)           # (batch, seq_len, seq_len): every query vs every key
    weights = torch.softmax(scores, dim=-1)                      # normalize each ROW into a probability distribution
    output = weights @ V                                          # weighted blend of Values -> (batch, seq_len, d_k)
    return output, weights

batch, seq_len, d_k = 1, 4, 8
Q = torch.rand(batch, seq_len, d_k)
K = torch.rand(batch, seq_len, d_k)
V = torch.rand(batch, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print("output shape :", output.shape)    # (1, 4, 8) -- same shape as V, one blended vector per position
print("weights shape:", weights.shape)    # (1, 4, 4) -- for each of the 4 positions, a distribution over 4 positions
print("each row sums to 1:", weights.sum(dim=-1))    # softmax guarantees this


### Multi-Head Attention

**Why one attention pass isn't enough:** a single attention computation can only learn *one* notion of "relevance" at a time. But language (and other sequential data) has many simultaneous kinds of relationships — grammatical (subject↔verb), referential ("it"↔its noun), thematic, positional/local, etc. **Multi-head attention runs several smaller attention operations in parallel** ("heads"), each with its own learned Q/K/V projections, so different heads are free to specialize in different kinds of relationships. Their outputs are concatenated and passed through one more learned projection to combine everything back into a single representation.

In [ ]:
# nn.MultiheadAttention -- PyTorch's built-in, production-ready implementation
embed_dim = 16      # size of each token's vector representation
num_heads = 4        # split embed_dim into 4 heads of size 16/4 = 4 each, running in parallel

mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)

x = torch.rand(2, 5, embed_dim)     # batch of 2 sequences, 5 tokens each, 16-dim embeddings

# In SELF-attention, the same tensor `x` is used as query, key, AND value
attn_output, attn_weights = mha(query=x, key=x, value=x)
print("attn_output shape:", attn_output.shape)   # (2, 5, 16) -- same shape as input, context now blended in
print("attn_weights shape:", attn_weights.shape)  # (2, 5, 5) -- averaged across heads by default


### Positional Encoding — giving the model a sense of order

Attention, as described above, treats the input as an *unordered set* of positions — swapping two tokens' order wouldn't change the attention math at all! But order obviously matters ("dog bites man" ≠ "man bites dog"). The fix: before feeding token embeddings into the transformer, **add** a positional encoding vector to each token — a fixed (or learned) pattern that's unique to each position, so the model can tell positions apart.

**The classic sinusoidal formula** (from the original "Attention Is All You Need" paper) uses sine and cosine waves of different frequencies:
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right) \qquad\qquad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$
**Intuition:** think of it like a clock with many hands moving at different speeds — a fast hand distinguishes nearby positions, a slow hand distinguishes far-apart positions, and the *combination* of all these different-frequency waves gives every position a unique, smoothly-varying "fingerprint." Many modern transformers instead use a simpler **learned** positional embedding (just an `nn.Embedding` indexed by position), which works about as well and is easier to implement — we show both below.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)                                # shape (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)     # even dimensions get sine
        pe[:, 1::2] = torch.cos(position * div_term)      # odd dimensions get cosine
        self.register_buffer("pe", pe)                      # a buffer: saved with the model, but NOT a learnable Parameter

    def forward(self, x):
        # x shape: (batch, seq_len, d_model) -- add the position pattern, truncated to this sequence's length
        return x + self.pe[: x.size(1)]

pos_enc = SinusoidalPositionalEncoding(d_model=16)
token_embeddings = torch.rand(2, 5, 16)
with_position = pos_enc(token_embeddings)
print(with_position.shape)   # (2, 5, 16) -- unchanged shape, positional information now blended in

# The simpler, equally common alternative: a LEARNED positional embedding
learned_pos_embedding = nn.Embedding(num_embeddings=512, embedding_dim=16)   # one learnable vector per position
positions = torch.arange(5)                              # positions 0,1,2,3,4 for a 5-token sequence
position_vectors = learned_pos_embedding(positions)         # (5, 16)


### Assembling a Transformer Encoder block

A full **Transformer Encoder layer** wraps multi-head self-attention and a small feed-forward network, each with a **residual (skip) connection** and **layer normalization** around it: `x = LayerNorm(x + Sublayer(x))`. The residual connection (add the input back to the sublayer's output) gives gradients a direct path backward through very deep stacks of these layers — the same "make gradients flow better" idea behind LSTM's cell state and ResNet's skip connections. PyTorch provides this whole block ready-made.

In [ ]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=16,          # the embedding dimension flowing through the model
    nhead=4,               # number of attention heads
    dim_feedforward=64,      # hidden size of the small feed-forward network inside each block
    batch_first=True,
)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)   # stack 2 of these blocks

x = torch.rand(2, 5, 16)          # (batch=2, seq_len=5, d_model=16)
output = transformer_encoder(x)
print(output.shape)                 # (2, 5, 16) -- same shape, but every position now encodes context
                                      # gathered from every other position, through 2 stacked attention blocks

# For autoregressive / decoder-style generation (predicting token t+1 from tokens 1..t, like GPT-style
# models), a CAUSAL MASK is added to the attention scores so a position cannot attend to future positions:
causal_mask = nn.Transformer.generate_square_subsequent_mask(5)   # 5x5 mask for a 5-token sequence
print(causal_mask)   # -inf above the diagonal (blocked/future), 0 on/below it (allowed/past+present)
# After softmax, any -inf score becomes exactly 0 attention weight -- effectively "invisible" future tokens.


### Real-world example, tying it together
A sentiment-classification transformer encoder reads a whole review at once — self-attention lets the word "not" directly influence the representation of "good" three words later ("not... very good" → negative), no matter how far apart they are, in a single parallel pass. A GPT-style decoder generating text uses the same self-attention math, but with the causal mask above, so each new word can only be influenced by words that came before it — exactly matching how text is generated one token at a time.

### 🎯 Interview Recap — Attention & Transformers

**⭐ Along with autograd, this is one of the highest-frequency topics in modern ML interviews — most companies now ask at least one transformer question.**

**If you remember nothing else:** self-attention lets every position look at every other position directly (no step-by-step relay); `Attention(Q,K,V) = softmax(QK^T/√d_k)V`; multi-head = several attention passes in parallel, specializing in different relationships; positional encoding is needed because attention alone has no sense of order.

**Q: Why did Transformers largely replace RNNs for sequence modeling?**
A: RNNs process one step at a time, which can't be parallelized across the sequence and struggles with very long-range dependencies; self-attention lets every position directly attend to every other position in one parallel pass, which trains much faster on GPUs and captures long-range relationships more directly.

**Q: What are Query, Key, and Value in attention, in plain terms?**
A: Query is "what am I looking for," Key is "what do I contain, in searchable form," and Value is "what content would I actually contribute" — each position compares its Query against every position's Key to get relevance weights, then blends together every position's Value using those weights.

**Q: Why divide by `sqrt(d_k)` in scaled dot-product attention?**
A: Without scaling, dot products grow larger as the key dimension increases, pushing softmax into a region with extremely small gradients almost everywhere ("saturation"); dividing by `sqrt(d_k)` keeps the scores in a range where softmax gradients stay useful.

**Q: Why use multiple attention heads instead of one?**
A: A single attention pass can only learn one notion of "relevance" at a time; multiple heads run in parallel with independent learned projections, letting different heads specialize in different kinds of relationships (e.g. grammatical vs. referential) simultaneously.

**Q: Why is positional encoding necessary in a Transformer?**
A: Self-attention treats the input as an unordered set — swapping two tokens wouldn't change the attention computation — so a positional signal must be added to each token's embedding to give the model any sense of sequence order.

**Q: What is a causal mask, and when is it used?**
A: A mask added to attention scores (setting future positions to `-inf` before softmax) so a position can only attend to itself and earlier positions — used in decoder/autoregressive models (like GPT-style generation) so training matches how text is generated one token at a time.

**Q: What role do the residual connections and LayerNorm play in a Transformer block?**
A: The residual connection (`x + Sublayer(x)`) gives gradients a direct path backward through many stacked layers, the same idea behind ResNet and LSTM's cell state; LayerNorm keeps activations at a stable scale, both of which make deep stacks of these blocks much easier to train.

<a id="save-load"></a>
# Part D — Production & Research Readiness
## 14. Saving & Loading Models

### Why this needs care
Training can take hours, days, or weeks. You need to be able to: (1) save a finished model so it can make predictions later without retraining, and (2) save an *in-progress* training run so it can be resumed exactly where it left off if your machine crashes, your cloud instance gets preempted, or you simply want to stop for the day. Getting this wrong (e.g. losing the optimizer's internal state) can silently make a "resumed" training run behave differently from an uninterrupted one.

### `state_dict` — the right thing to save

Recall from [`nn.Module`](#nn-module) that `model.state_dict()` is an ordered dictionary mapping every parameter's name to its tensor value. **PyTorch's official recommendation is to save the `state_dict`, not the whole model object.** Saving the whole object (`torch.save(model, "model.pth")`) pickles the *exact* Python class definition alongside the weights — if you ever refactor your model's code, rename a file, or load it in a different project, unpickling can break in confusing ways. Saving just the `state_dict` (the numbers) and keeping the model-building *code* separately is far more robust and portable.

In [ ]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(10, 2)

    def forward(self, x):
        return self.layer(x)

model = SimpleModel()

# ---- Saving (recommended way): just the learnable numbers ----
torch.save(model.state_dict(), "simple_model.pth")     # writes a file containing the state_dict

# ---- Loading: recreate the model ARCHITECTURE first (same class, same __init__ args), THEN load weights ----
loaded_model = SimpleModel()                              # step 1: build a fresh model with the same architecture
loaded_model.load_state_dict(torch.load("simple_model.pth"))   # step 2: copy the saved numbers into it
loaded_model.eval()                                         # step 3: switch to eval mode before using it for inference

print("Loaded successfully. Sample output:")
with torch.no_grad():
    print(loaded_model(torch.rand(1, 10)))


> ⚠️ **Always call `model.eval()` after loading a model for inference** (see the [Training Loop section](#training-loop) for why) — otherwise layers like Dropout will still behave randomly, giving inconsistent predictions.

### Checkpointing — saving everything needed to resume training

To resume training exactly where you left off, saving only the model's weights is **not enough** — you also need the **optimizer's internal state** (e.g. Adam's running gradient-moment averages — restarting Adam from scratch mid-training effectively resets its "memory" and can visibly disrupt the training trajectory), the current epoch number, and typically the last known loss (for logging/comparison). The standard pattern is to save a single dictionary bundling everything together.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ---- Saving a full checkpoint mid-training ----
checkpoint = {
    "epoch": 12,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),    # includes Adam's per-parameter running averages, etc.
    "loss": 0.245,
}
torch.save(checkpoint, "checkpoint.pth")

# ---- Loading a checkpoint to RESUME training ----
checkpoint = torch.load("checkpoint.pth")
resumed_model = SimpleModel()
resumed_optimizer = torch.optim.Adam(resumed_model.parameters(), lr=0.001)

resumed_model.load_state_dict(checkpoint["model_state_dict"])
resumed_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"] + 1                     # resume from the NEXT epoch
last_loss = checkpoint["loss"]

resumed_model.train()          # switch back to train mode -- we're about to continue training, not infer
print(f"Resuming from epoch {start_epoch}, last recorded loss was {last_loss}")


### Cross-device loading with `map_location`

If you saved a checkpoint on a GPU machine but are loading it on a CPU-only machine (or a different GPU), you need to tell `torch.load` where to put the tensors, or it will try to load them onto the original device and crash if that device isn't available.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# map_location tells torch.load which device to put every tensor on, regardless of where it was saved from
checkpoint = torch.load("checkpoint.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)     # remember: model and future input data must end up on the same device


### A quick note on deployment formats
Saving a `state_dict` is the standard for continuing work *in PyTorch*. When a model needs to run outside of a full Python/PyTorch environment (e.g. on a mobile device, in a C++ service, or via a highly optimized inference server), it's common to first export it to a portable format like **TorchScript** or **ONNX** — this is covered briefly in the [Advanced Topics](#advanced) section, since it's a deployment concern layered on top of the saving/loading basics above.

### 🎯 Interview Recap — Saving & Loading

**If you remember nothing else:** save `state_dict()`, not the whole model object; resuming training needs the optimizer's state too, not just the weights; always `model.eval()` before inference; use `map_location` when loading across devices.

**Q: Why does PyTorch recommend saving `model.state_dict()` instead of the whole model object?**
A: Saving the whole object pickles the exact class definition alongside the weights, which can break if you later refactor the code, rename files, or load it in a different environment; saving just the `state_dict` (the numbers) and keeping the model-building code separate is far more portable and robust.

**Q: If you only save and reload the model weights, what breaks when resuming training?**
A: The optimizer's internal state is lost — e.g. Adam's running gradient-moment averages reset to zero — so training effectively restarts that optimizer's "memory," which can visibly disrupt the training trajectory compared to an uninterrupted run.

**Q: What are the two steps needed to load a `state_dict`?**
A: First recreate the model architecture (instantiate the same class with the same constructor arguments), then call `model.load_state_dict(torch.load(path))` to copy the saved numbers into it.

**Q: Why is `map_location` needed when loading a checkpoint?**
A: A checkpoint saved on one device (e.g. a GPU) will try to load its tensors back onto that same device by default; `map_location` explicitly redirects tensors to the device you actually want (e.g. CPU), preventing a crash when the original device isn't available.

<a id="optuna"></a>
## 15. Hyperparameters & Tuning with Optuna

### Parameters vs. Hyperparameters — a distinction worth being precise about
- **Parameters** are the numbers the model **learns** automatically via `.backward()` + `optimizer.step()` — weights and biases.
- **Hyperparameters** are the choices **you** make *before* training starts, which control *how* learning happens — they are never updated by gradient descent itself.

### The hyperparameters that show up throughout this notebook
| Hyperparameter | Where it appears | Typical range to search |
|---|---|---|
| Learning rate | [Optimizers](#optimizers) | `1e-5` to `1e-1` (usually searched on a log scale) |
| Batch size | [DataLoader](#datasets-dataloaders) | 16, 32, 64, 128, 256... |
| Number of hidden layers / units | [`nn.Module`](#nn-module) | problem-dependent |
| Dropout rate | [Regularization](#regularization) | 0.0 to 0.5 |
| Weight decay | [Optimizers](#optimizers) | `0` to `1e-2` (log scale) |
| Optimizer choice | [Optimizers](#optimizers) | SGD, Adam, AdamW... |
| Kernel size, number of filters | [CNN](#cnn) | 3, 5, 7 / 16, 32, 64... |
| Number of epochs | [Training loop](#training-loop) | often paired with early stopping instead of a fixed number |

### Why manual or grid/random search isn't good enough
- **Manual tuning** doesn't scale and is heavily biased by the researcher's intuition and patience.
- **Grid search** (try every combination) grows **exponentially** with the number of hyperparameters — 5 hyperparameters with 5 values each is $5^5 = 3{,}125$ full training runs.
- **Random search** is a solid, cheap baseline (usually better than grid search for the same budget), but it doesn't use information from *earlier* trials to guide *later* ones — it keeps sampling blindly.

### Why Optuna
**Optuna** is a hyperparameter-optimization framework that searches *intelligently*: it uses information from every completed trial to decide where to sample next (by default via the **TPE — Tree-structured Parzen Estimator — sampler**, a Bayesian-style approach), and it can **prune** (stop early) trials that are clearly performing badly partway through training, saving enormous amounts of compute versus letting every trial run to completion.

**Core Optuna vocabulary:**
- **Study:** one complete optimization session (a collection of trials, all trying to optimize the same objective).
- **Trial:** one single run with one specific combination of hyperparameter values.
- **Objective function:** a function you write that takes a `trial` object, uses it to *suggest* hyperparameter values, trains/evaluates a model with them, and **returns the metric to optimize** (e.g. validation loss, or accuracy).

In [ ]:
# pip install optuna --quiet   # (install once if not already available)
import optuna

def objective(trial):
    # trial.suggest_*(...) is how you define the SEARCH SPACE for each hyperparameter, per trial.
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)     # log=True: sample on a LOG scale, appropriate
                                                                 # whenever the "right" value could be off by
                                                                 # orders of magnitude (very common for LR)
    hidden_size = trial.suggest_int("hidden_size", 8, 128, step=8)   # integer values, in steps of 8
    dropout = trial.suggest_float("dropout", 0.0, 0.5)                # plain (linear-scale) float range
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])   # pick from a list

    # (a real objective would build a model + train it here using these exact sampled values, then
    #  return a validation metric -- shown in the full worked example below)
    return lr + hidden_size * 0.0001    # placeholder "fake" metric just to demonstrate the plumbing

study = optuna.create_study(direction="minimize")   # "minimize" for loss, "maximize" for accuracy
study.optimize(objective, n_trials=5)                 # runs the objective function 5 times, each a new trial

print("Best hyperparameters found:", study.best_params)
print("Best value:", study.best_value)


### Pruning — stopping bad trials early

For a real training objective, you can report intermediate results (e.g. validation loss after each epoch) back to Optuna via `trial.report()`, and ask `trial.should_prune()` whether this trial looks unpromising compared to others so far — if so, you raise `optuna.TrialPruned()` to stop it immediately rather than wasting the remaining epochs.

---
## 🏗️ Full Worked Example: Tuning the ANN Classifier from Section 10 with Optuna

We reuse the exact two-moons ANN classifier from [Section 10](#training-loop) and let Optuna search over its learning rate, hidden layer size, dropout rate, and choice of optimizer — with pruning enabled, so clearly-bad trials are abandoned early.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.datasets import make_moons

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=42)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [train_size, len(dataset) - train_size],
                                  generator=torch.Generator().manual_seed(0))


class TunableANN(nn.Module):
    # An ANN whose SHAPE itself depends on hyperparameters (hidden_size, dropout) chosen per trial.
    def __init__(self, hidden_size, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_size), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x):
        return self.net(x)


def objective(trial):
    # ---- 1. Sample this trial's hyperparameters ----
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    hidden_size = trial.suggest_int("hidden_size", 8, 64, step=8)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    # ---- 2. Build the model + optimizer using those sampled hyperparameters ----
    model = TunableANN(hidden_size, dropout).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer_cls = getattr(torch.optim, optimizer_name)     # looks up torch.optim.Adam or torch.optim.SGD by name
    optimizer = optimizer_cls(model.parameters(), lr=lr)

    # ---- 3. Train for a small, fixed number of epochs, reporting progress to Optuna for pruning ----
    for epoch in range(15):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += loss_fn(model(X_batch), y_batch).item() * len(X_batch)
        val_loss /= len(val_ds)

        trial.report(val_loss, epoch)               # tell Optuna how this trial is doing so far
        if trial.should_prune():                       # ask Optuna: "does this look unpromising?"
            raise optuna.TrialPruned()                    # if so, stop this trial immediately -- saves compute

    return val_loss     # the FINAL validation loss is what Optuna tries to minimize


# MedianPruner: prune a trial if its performance is worse than the median of other trials at the same epoch
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=15)     # in real projects this is often 50-200+ trials

print("Best hyperparameters:", study.best_params)
print("Best validation loss:", study.best_value)
print(f"Trials completed: {len(study.trials)}, pruned: {sum(t.state.name=='PRUNED' for t in study.trials)}")


**What just happened:** instead of us guessing a learning rate or hidden size, Optuna intelligently explored the search space — favoring regions that looked promising based on earlier trials — and abandoned trials early once it was clear they wouldn't catch up to the best ones seen so far. `study.best_params` now hands you back the exact hyperparameter combination to use for your final, full training run.

**A note on visualizing results:** Optuna also ships plotting helpers (`optuna.visualization.plot_optimization_history(study)`, `plot_param_importances(study)`, etc.) that render interactive charts showing which hyperparameters mattered most — extremely useful for understanding *why* certain values won, not just *which* values won.

### 🎯 Interview Recap — Hyperparameters & Optuna

**If you remember nothing else:** hyperparameters are set before training, parameters are learned during it; grid search is exponential and random search is blind; Optuna uses past trials (TPE sampler) to guide future sampling and can prune bad trials early to save compute.

**Q: What's the difference between a parameter and a hyperparameter?**
A: A parameter (weights, biases) is learned automatically via backpropagation and the optimizer; a hyperparameter (learning rate, batch size, number of layers...) is chosen by you before training and controls how learning happens — gradient descent never updates it directly.

**Q: Why does grid search become impractical quickly?**
A: The number of combinations grows exponentially with the number of hyperparameters — e.g. 5 hyperparameters with 5 candidate values each is 5^5 = 3,125 full training runs.

**Q: How does Optuna's default sampling strategy differ from random search?**
A: Optuna's default TPE (Tree-structured Parzen Estimator) sampler is Bayesian-style — it uses the results of previous trials to decide where to sample next, focusing on promising regions, whereas random search keeps sampling blindly without learning from past trials.

**Q: What is pruning in the context of hyperparameter tuning, and why does it matter?**
A: Stopping an unpromising trial early — partway through training — instead of always running it to completion, based on how it compares to other trials at the same point; it can save enormous amounts of compute across a large search.

**Q: What three things does an Optuna objective function typically do?**
A: Use the `trial` object to sample hyperparameter values (`trial.suggest_*`), build and train/evaluate a model using those values, and return the metric Optuna should optimize (e.g. validation loss).

**Q: What's the current recommended way to sample a log-scale hyperparameter like learning rate in Optuna?**
A: `trial.suggest_float("lr", low, high, log=True)` — the older `suggest_loguniform` is deprecated in favor of this unified `suggest_float` API.

<a id="transfer-learning"></a>
## 16. Transfer Learning

### Why transfer learning exists
Training a large CNN or Transformer from scratch (random initialization) to a strong accuracy typically needs **millions** of labeled examples and huge amounts of compute — completely out of reach for most individual projects and companies. But here's the key insight: a network trained on a large, general dataset (like ImageNet's 1.2 million images across 1,000 categories) learns early layers that detect **generic, broadly-reusable** patterns — edges, textures, colors, simple shapes — that are just as useful for classifying X-rays or satellite photos as they are for classifying the original dataset's cats and dogs. **Transfer learning reuses those already-learned features** instead of re-learning them from zero, which means you often need only a few hundred or thousand examples of *your* task to get strong results.

**Real-world example:** a hospital wants to classify a rare type of medical scan, but only has 300 labeled images — nowhere near enough to train a CNN from scratch. By starting from an ImageNet-pretrained ResNet (which already "knows" how to detect edges, blobs, and textures) and only adapting its final layers to the new task, the hospital can get a strong, useful classifier from those 300 images.

### Two strategies: Feature Extraction vs. Fine-Tuning

- **Feature Extraction:** **freeze** the entire pretrained backbone (its weights never update — `requires_grad=False`), and only train a small new "head" (usually just the final classification layer) on top of the frozen features. Fast, needs very little data, and is a great starting point.
- **Fine-Tuning:** start from the pretrained weights, but allow some (often the later layers, closer to the task-specific output) or all layers to keep updating with a **small** learning rate. Needs somewhat more data than pure feature extraction, but can noticeably outperform it, since the features themselves get to adapt slightly to your specific task.

**Rule of thumb for choosing:** the less data you have and the more similar your task is to the original pretraining data, the more you should freeze (favor feature extraction). The more data you have and the more different your task is, the more you should unfreeze and fine-tune.

In [ ]:
# The current, recommended torchvision API uses explicit `weights=` enums (the older `pretrained=True`
# is deprecated). This cell shows the standard real-world pattern -- it needs internet on first run to
# download the pretrained weights, so treat this as reference code to copy into your own environment.

# import torchvision
# from torchvision.models import resnet18, ResNet18_Weights
#
# weights = ResNet18_Weights.DEFAULT                  # DEFAULT = the best available pretrained weights
# model = resnet18(weights=weights)                     # downloads + loads ImageNet-pretrained weights
#
# preprocess = weights.transforms()                       # each weights object KNOWS the exact preprocessing
                                                              # (resize/crop/normalize) it was trained with --
                                                              # always use this instead of guessing your own!


### Freezing layers, and replacing the classification head

A pretrained ImageNet model's final layer outputs scores for 1,000 ImageNet classes — almost never what you want. The standard pattern is: freeze everything, then **replace** the final layer with a fresh, *trainable*, randomly-initialized layer sized for *your* number of classes (a freshly-created layer always has `requires_grad=True` by default, so it will train normally even while everything else is frozen).

In [ ]:
import torch
import torch.nn as nn

# We'll simulate "a pretrained ResNet-like model" with a small stand-in CNN, so this cell runs offline
# and shows the exact freezing/replacing pattern you'd apply to a real torchvision model.
class PretrainedStandIn(nn.Module):
    # Pretend this arrived already trained on a huge dataset -- e.g. imagine this is torchvision's resnet18()
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(32, 1000)     # pretend this was trained for 1000 ImageNet-style classes

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

model = PretrainedStandIn()

# ---- STEP 1: freeze every existing parameter (feature extraction mode) ----
for param in model.parameters():
    param.requires_grad = False               # "don't compute gradients for this -- don't update it"

# ---- STEP 2: replace the final layer with a NEW layer sized for YOUR task ----
num_my_classes = 5                              # e.g. our real task only has 5 classes, not 1000
model.fc = nn.Linear(in_features=32, out_features=num_my_classes)
# `model.fc` is now a BRAND NEW nn.Linear -- freshly created, so its requires_grad defaults back to True,
# even though we just froze everything else above. This is the standard "swap the head" trick.

# ---- Sanity check: confirm only the new head is trainable ----
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable parameters: {trainable:,}  (should just be the new head: 32*5 + 5 = {32*5+5})")
print(f"Frozen parameters: {frozen:,}")

# ---- Only pass TRAINABLE parameters to the optimizer (frozen ones don't need an optimizer at all) ----
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)


### Fine-tuning with differential (discriminative) learning rates

A common, more advanced pattern: instead of a binary frozen/unfrozen split, **unfreeze the whole network but give the pretrained backbone a much smaller learning rate than the new head.** This lets the backbone adapt gently (protecting the valuable features it already learned) while the brand-new head — which starts randomly initialized and has a lot more to learn — trains at a normal, faster pace. `torch.optim` supports this directly via **parameter groups**.

In [ ]:
for param in model.parameters():         # unfreeze everything for full fine-tuning
    param.requires_grad = True

optimizer = torch.optim.Adam([
    {"params": model.backbone.parameters(), "lr": 1e-5},   # pretrained backbone: small, gentle updates
    {"params": model.fc.parameters(), "lr": 1e-3},           # new head: normal, faster learning rate
])
# Both groups still update together every optimizer.step() -- they simply move by different-sized steps.


### Full pattern, tied together (training loop is unchanged)

Once frozen/unfrozen and the head is replaced, training uses **the exact same loop from [Section 10](#training-loop)** — `model.train()`, forward pass, loss, `backward()`, `optimizer.step()`. This is the whole point of building on the standard loop early: nothing about transfer learning requires a different training procedure, only a different *starting point* for the weights and a different choice of *which* weights are allowed to move.

### 🎯 Interview Recap — Transfer Learning

**If you remember nothing else:** early layers learn generic, broadly-reusable features (edges, textures); feature extraction freezes the backbone and trains only a new head; fine-tuning allows some/all layers to keep updating, usually with a small learning rate; always replace the final layer to match your number of classes.

**Q: Why does transfer learning work at all — why are ImageNet features useful for a totally different task?**
A: Early layers of a CNN learn generic, low-level visual patterns (edges, textures, simple shapes) that are broadly useful across almost any visual task, not specific to the original dataset's classes — only the later, more task-specific layers need significant adaptation.

**Q: What's the difference between feature extraction and fine-tuning?**
A: Feature extraction freezes the entire pretrained backbone and trains only a new head on top; fine-tuning allows some or all of the backbone's weights to keep updating too, usually with a small learning rate, letting the features adapt slightly to the new task.

**Q: When would you choose feature extraction over fine-tuning?**
A: When you have limited data and/or your task is visually similar to the original pretraining data — freezing more protects the already-learned features from being disrupted by too few examples.

**Q: Why does a newly-created replacement layer (like a new `fc` head) train even after you've frozen the rest of the model?**
A: Freezing sets `requires_grad=False` only on the parameters that existed at the time; a freshly created `nn.Linear` layer has `requires_grad=True` by default, so assigning it afterward gives you a trainable head sitting on top of a frozen backbone.

**Q: What is a differential (discriminative) learning rate, and why use one?**
A: Using a smaller learning rate for the pretrained backbone and a larger one for the new head (via optimizer parameter groups), so the valuable pretrained features are adjusted gently while the randomly-initialized head — which has much more to learn — trains at a normal pace.

<a id="regularization"></a>
## 17. Regularization & Training Best Practices

Recall from [Section 10](#training-loop): **overfitting** is when a model memorizes training-set specifics (including noise) instead of learning generalizable patterns, visible as validation loss plateauing or rising while training loss keeps falling. **Regularization** is the umbrella term for techniques that fight this — mostly by deliberately making training slightly *harder* or *noisier* in ways that push the model toward simpler, more generalizable solutions.

### Dropout — training with random amnesia

**Intuition:** during each training step, **randomly and temporarily delete** a fraction of neurons (set their output to exactly 0). This forces the network to avoid relying too heavily on any single neuron or tight co-dependent group of neurons (since any of them might vanish at any moment), pushing it toward redundant, more robust representations — similar in spirit to a sports team that practices with random players sitting out, forcing everyone to become independently capable instead of relying on one star player.

**The math:** with dropout probability $p$, each neuron's output is zeroed with probability $p$ during training; the remaining active neurons are scaled up by $\frac{1}{1-p}$ so the *expected* total signal stays roughly the same. At evaluation time (`model.eval()`), dropout does nothing — every neuron is active, using the network's full learned capacity (this is exactly why [`model.train()` vs `model.eval()`](#training-loop) matters so much).

In [ ]:
import torch
import torch.nn as nn

dropout = nn.Dropout(p=0.5)     # 50% of activations get zeroed, per call, during training
x = torch.ones(10)

dropout.train()                  # explicitly set training mode for this demo
print("train mode  :", dropout(x))   # roughly half the values are 0, survivors scaled up (here, to 2.0)

dropout.eval()                    # switch to eval mode
print("eval mode   :", dropout(x))   # unchanged -- ALL values pass through, dropout is a no-op when evaluating


### Batch Normalization & Layer Normalization

**The problem:** as data flows through a deep network, the scale and distribution of activations at each layer can shift and vary a lot, layer to layer and batch to batch — this makes optimization harder (the loss landscape becomes less smooth and well-behaved) and typically forces you to use a smaller, more conservative learning rate. **Normalization layers fix this by re-centering and re-scaling activations** to have (roughly) zero mean and unit variance at each layer, then applying a small learned rescale/shift so the network can still represent whatever scale it actually needs.

**The math**, for a set of values $x$:
$$\hat x = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \qquad\qquad y = \gamma \hat x + \beta$$
where $\mu,\sigma^2$ are the mean/variance to normalize by, $\epsilon$ is a tiny constant preventing division by zero, and $\gamma,\beta$ are **learned** parameters letting the network undo the normalization if that turns out to be optimal for a particular layer.

- **`nn.BatchNorm2d`** (used heavily in CNNs) computes $\mu,\sigma^2$ **across the batch dimension** (and spatial dimensions) — this is exactly why it behaves differently in train mode (uses the current batch's statistics) vs. eval mode (uses a running average collected during training, since you might not even have a "batch" at inference time), as first introduced in [Training Loop](#training-loop).
- **`nn.LayerNorm`** (used heavily in Transformers) computes $\mu,\sigma^2$ **across the feature dimension, per individual sample** instead — this makes it independent of batch size and well-suited to variable-length sequences, which is why Transformers use LayerNorm rather than BatchNorm.

**Practical, well-established benefits:** networks with normalization layers train faster, tolerate higher learning rates, and are less sensitive to weight initialization — even though the *exact* theoretical reason why (the original "internal covariate shift" explanation has been debated by later research, which points more toward a smoother loss landscape) is still an active research topic. The practical recipe works extremely well regardless of exactly which explanation is most correct.

In [ ]:
batchnorm = nn.BatchNorm2d(num_features=16)   # normalizes each of 16 channels, across batch+spatial dims
x = torch.randn(8, 16, 32, 32) * 5 + 10        # a batch with a deliberately large mean/spread, to see the effect

batchnorm.train()
out = batchnorm(x)
print("input  mean/std:", x.mean().item(), x.std().item())     # far from 0/1
print("output mean/std:", out.mean().item(), out.std().item())  # much closer to 0/1 after normalization

layernorm = nn.LayerNorm(normalized_shape=16)     # common in Transformers: normalize across the feature dim
x_seq = torch.randn(2, 5, 16) * 3               # (batch, seq_len, features)
out_seq = layernorm(x_seq)
print("LayerNorm output mean (per token, ~0):", out_seq[0, 0].mean().item())


### Weight decay (L2 regularization) — recap and the math

Introduced briefly in [Optimizers](#optimizers): weight decay adds a penalty proportional to the *squared size* of every weight directly into the loss being minimized:
$$L_{\text{total}} = L_{\text{original}} + \lambda \sum_i w_i^2$$
**Intuition:** this constantly pulls every weight gently toward zero, so the optimizer only "spends" a large weight value where the data really justifies it. This discourages the network from relying on any single extreme weight, which tends to produce simpler, more generalizable solutions. In PyTorch this is set directly on the optimizer (`weight_decay=...`), not as a separate layer.

### Data Augmentation — manufacturing more training data

**Intuition:** if you don't have enough labeled data, generate plausible *variations* of the data you already have — for images: random crops, horizontal flips, slight rotations, color jitter, and similar transformations that shouldn't change the correct label but do change the exact pixel values the network sees. This exposes the model to more diversity without needing any new labels, directly fighting overfitting. As covered in the [`Dataset`/`transforms`](#datasets-dataloaders) section, these are applied fresh, randomly, inside `__getitem__` — so the model essentially never sees the *exact* same image twice across epochs.

In [ ]:
# The standard pattern for TRAINING data (randomized augmentation) vs. EVAL data (fixed, no randomness):
# import torchvision.transforms as T
#
# train_transform = T.Compose([
#     T.RandomResizedCrop(224),        # randomly crop + resize -- teaches scale/position invariance
#     T.RandomHorizontalFlip(),          # randomly mirror left-right (skip this for tasks where L/R matters!)
#     T.ColorJitter(brightness=0.2, contrast=0.2),   # random small lighting/color changes
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])
#
# eval_transform = T.Compose([               # NO randomness for validation/test -- results must be reproducible
#     T.Resize(256), T.CenterCrop(224),
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])


### Early Stopping — knowing when to stop

**Intuition:** track validation loss every epoch; if it hasn't improved for a set number of epochs in a row (the **patience**), stop training and keep the weights from the *best* epoch seen, not the final one. This directly targets the overfitting pattern described at the top of this section — training well past the point validation loss starts rising only makes overfitting worse, so early stopping catches the model near its best generalization point.

In [ ]:
class EarlyStopper:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience              # how many epochs to tolerate no improvement
        self.min_delta = min_delta              # minimum change to count as an "improvement"
        self.best_loss = float("inf")
        self.counter = 0

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:   # meaningful improvement -> reset patience
            self.best_loss = val_loss
            self.counter = 0
            return False       # False = don't stop
        else:                                              # no improvement this epoch
            self.counter += 1
            return self.counter >= self.patience             # True once patience is exhausted -> stop

# Usage inside a training loop:
early_stopper = EarlyStopper(patience=5)
fake_val_losses = [0.9, 0.7, 0.5, 0.48, 0.47, 0.475, 0.474, 0.476, 0.473, 0.472, 0.474]
for epoch, val_loss in enumerate(fake_val_losses):
    if early_stopper.step(val_loss):
        print(f"Stopping early at epoch {epoch} -- no improvement for {early_stopper.patience} epochs")
        break
    print(f"epoch {epoch}: val_loss={val_loss} (best so far: {early_stopper.best_loss})")


### Gradient Clipping — preventing exploding gradients

Introduced conceptually in [RNNs](#rnn): sometimes a gradient becomes extremely (even infinitely) large, causing a single catastrophic weight update that can wreck days of training progress in one step. **Gradient clipping caps the total gradient norm** at a maximum value right before `optimizer.step()`, capping the size of the update without changing its *direction*.

In [ ]:
model = nn.Linear(10, 1)
loss = (model(torch.rand(4, 10)) ** 2).sum()
loss.backward()

# Clip so the OVERALL gradient norm (across all parameters together) never exceeds max_norm
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
# Call this AFTER loss.backward() and BEFORE optimizer.step() -- it modifies .grad in place.
# Extremely common in training RNNs/LSTMs and Transformers, where occasional gradient spikes are common.


### 🎯 Interview Recap — Regularization

**If you remember nothing else:** Dropout randomly zeroes neurons during training only; BatchNorm/LayerNorm normalize activations for more stable, faster training; weight decay pulls weights toward zero; data augmentation manufactures more training diversity; early stopping halts training at the best validation point; gradient clipping caps update size without changing direction.

**Q: How does Dropout behave differently in `model.train()` vs. `model.eval()`?**
A: During training, it randomly zeroes a fraction of activations (scaling survivors up to compensate) on every forward pass; during evaluation it does nothing — every neuron stays active, using the network's full capacity.

**Q: What's the practical difference between BatchNorm and LayerNorm?**
A: BatchNorm normalizes across the batch (and spatial) dimensions, making it dependent on batch statistics and behave differently between train/eval modes; LayerNorm normalizes across the feature dimension per individual sample, making it independent of batch size — which is why Transformers, which often deal with variable-length sequences, use LayerNorm instead.

**Q: What does weight decay actually do mathematically?**
A: It adds a penalty proportional to the sum of squared weights to the loss being minimized, which constantly pulls every weight gently toward zero unless the data strongly justifies a larger value.

**Q: Why does data augmentation help fight overfitting?**
A: It manufactures plausible variations of existing training examples (crops, flips, color jitter...) without needing new labels, exposing the model to more diversity so it's less likely to memorize exact pixel-level specifics of the original training images.

**Q: How does early stopping decide when to stop, and which weights does it keep?**
A: It tracks validation loss every epoch and stops once it hasn't meaningfully improved for a set number of epochs in a row (the "patience"); it keeps the weights from the best-observed epoch, not necessarily the final one.

**Q: What problem does gradient clipping solve, and where in the training loop does it go?**
A: It prevents a single, extremely large gradient from causing a catastrophic weight update ("exploding gradients"), by capping the overall gradient norm; it's called after `loss.backward()` and before `optimizer.step()`.

<a id="advanced"></a>
## 18. Advanced Topics

A tour of techniques that separate "I can train a model" from "I can train models efficiently, debug them fast, and evaluate them properly" — the kind of practical depth that shows up in senior ML Engineer and MLOps interviews.

### Automatic Mixed Precision (AMP) — training faster with less memory

**Why it exists:** the default `float32` (32-bit) gives great precision but costs more memory and compute than necessary for most of a network's operations. Modern GPUs have specialized hardware (e.g. NVIDIA's "Tensor Cores") that runs 16-bit math dramatically faster than 32-bit math. **Mixed precision** runs *most* operations in 16-bit (`float16` or `bfloat16`) for speed and memory savings, while automatically keeping numerically-sensitive operations (like large sums) in 32-bit where precision actually matters — "mixed," because both precisions are used together, automatically, per-operation.

**The one real danger with `float16`:** its range is much smaller than `float32`, so gradients can sometimes underflow to exactly zero (**vanish**) during backpropagation before they're used to update weights. **`GradScaler`** fixes this by temporarily multiplying ("scaling up") the loss by a large factor before `backward()` — which scales up all the gradients too, lifting them out of the underflow range — then un-scales them back down right before the optimizer actually applies the update.

In [ ]:
import torch
import torch.nn as nn

# This is the CURRENT recommended API (the older torch.cuda.amp.autocast / torch.cuda.amp.GradScaler
# are deprecated in favor of the device-agnostic torch.amp versions shown below).

model = nn.Linear(10, 1).to("cuda" if torch.cuda.is_available() else "cpu")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
device_type = "cuda" if torch.cuda.is_available() else "cpu"

scaler = torch.amp.GradScaler(device_type, enabled=(device_type == "cuda"))   # only meaningfully used on GPU

X = torch.rand(32, 10).to(device_type)
y = torch.rand(32, 1).to(device_type)

optimizer.zero_grad()
with torch.amp.autocast(device_type, dtype=torch.float16, enabled=(device_type == "cuda")):
    # Everything computed INSIDE this block automatically runs in float16 where it's safe to do so,
    # and PyTorch decides per-operation when to keep float32 instead (e.g. for numerically sensitive ops)
    prediction = model(X)
    loss = loss_fn(prediction, y)

scaler.scale(loss).backward()      # scale the loss UP before backward(), to avoid tiny-gradient underflow
scaler.step(optimizer)               # unscales gradients back down internally, then calls optimizer.step()
scaler.update()                       # adjusts the scale factor for next time, based on whether overflow occurred

print("Mixed-precision training step completed. Loss:", loss.item())
# On CPU, autocast/GradScaler are set to `enabled=False` above, so this safely just runs in normal float32.


### `torch.compile` — free(-ish) speed with one line

Introduced in PyTorch 2.0, `torch.compile(model)` analyzes your model's computation graph and **fuses multiple small operations into fewer, more efficient GPU kernels**, often giving a meaningful speedup with **no other code changes**. The first call is slower (it's compiling), and subsequent calls with the same input shapes reuse the compiled version.

In [ ]:
model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 1))

# One extra line -- the model behaves identically, just runs faster on supported hardware:
compiled_model = torch.compile(model)

x = torch.rand(32, 10)
output = compiled_model(x)     # first call triggers compilation (slower); later calls reuse it (faster)
print(output.shape)


### Custom Autograd Functions

Almost everything can be built from existing PyTorch operations, which already support autograd automatically. Occasionally, though, you need a genuinely custom operation — e.g. one with no clean derivative (like rounding), or one where you know a smarter, more efficient way to compute the gradient than what autograd would derive automatically by combining existing ops. `torch.autograd.Function` lets you define your own `forward()` **and** your own `backward()` by hand.

In [ ]:
class CustomReLU(torch.autograd.Function):
    # A hand-implemented version of ReLU, to demonstrate the pattern (ReLU already has a normal, safe
    # gradient built in -- this is purely illustrative of HOW to write a custom Function).

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)         # remember `x` -- we'll need it during the backward pass
        return x.clamp(min=0)              # the actual forward computation: max(0, x)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors               # retrieve what we saved during forward
        grad_input = grad_output.clone()
        grad_input[x < 0] = 0                  # ReLU's derivative is 0 where x < 0, and 1 where x > 0
        return grad_input

x = torch.tensor([-1.0, 2.0, -3.0, 4.0], requires_grad=True)
y = CustomReLU.apply(x)          # custom Functions are invoked via .apply(), not called directly
y.sum().backward()
print(y)          # tensor([0., 2., 0., 4.])
print(x.grad)      # tensor([0., 1., 0., 1.]) -- exactly matches ReLU's expected gradient


### A practical debugging checklist

Real PyTorch bugs tend to fall into a small number of recurring buckets. When something breaks, check in roughly this order:

| Symptom | Likely cause | First thing to check |
|---|---|---|
| `RuntimeError: size mismatch` / shape errors | A layer's input doesn't match what it expects | Print `.shape` at every step of `forward()` |
| `Expected all tensors to be on the same device` | Model and data (or two tensors) on different devices | Print `.device` on both sides of the failing operation |
| Loss is `NaN` | Exploding gradients, learning rate too high, or a `log(0)` / divide-by-zero somewhere | Lower the learning rate; add gradient clipping; check for `0` inputs to `log`/division |
| Loss doesn't decrease at all | Forgot `optimizer.zero_grad()`; forgot `.backward()`; forgot `optimizer.step()`; learning rate way too small | Print gradients (`param.grad`) — are they `None` or all zero? |
| Great training accuracy, poor validation accuracy | Overfitting | Add [regularization](#regularization); get more data; simplify the model |
| Poor training AND validation accuracy | Underfitting, or a genuine bug in the data/labels | Try to deliberately overfit a tiny subset (e.g. 10 samples) first — a model that can't even memorize 10 examples usually has a bug, not a capacity problem |
| Works on CPU, breaks on GPU (or vice versa) | A device-specific op, or forgot to move something | Check every `.to(device)` call is actually present |

### Evaluation metrics beyond accuracy

Accuracy alone can be dangerously misleading, especially with **imbalanced classes** — a model that always predicts "not fraud" on a dataset that's 99% not-fraud gets 99% accuracy while being completely useless. These four metrics, built from the **confusion matrix**, give a fuller picture:

$$\text{Precision} = \frac{TP}{TP+FP} \qquad \text{Recall} = \frac{TP}{TP+FN} \qquad F_1 = 2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}$$

- **Precision:** "Of everything I predicted positive, how much was actually positive?" — matters when false positives are costly (e.g. flagging a legitimate transaction as fraud, annoying a real customer).
- **Recall:** "Of everything that was actually positive, how much did I catch?" — matters when false negatives are costly (e.g. missing an actual cancer diagnosis).
- **F1 score:** the harmonic mean of precision and recall — a single number balancing both, useful when you care about both and especially with imbalanced classes.
- **Confusion matrix:** the full breakdown of predicted-vs-actual class counts, which precision/recall/F1 are computed from — always worth looking at directly, not just the summary numbers.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 0]
y_pred = [1, 0, 0, 1, 0, 1, 1, 0, 1, 0]     # model got most right, but 1 false positive and 1 false negative

print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1 score :", f1_score(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
# Rows = actual class, columns = predicted class:
# [[TN, FP],
#  [FN, TP]]


### Brief pointers: experiment tracking, deployment, and multi-GPU training

Full coverage of these is beyond a single notebook section, but you should know what they are and when you'd reach for them:

- **Experiment tracking** (`TensorBoard`, Weights & Biases, MLflow...): logs metrics, hyperparameters, and sometimes model checkpoints across many training runs, so you can compare experiments visually instead of scrolling through printed logs. `torch.utils.tensorboard.SummaryWriter` is PyTorch's built-in option.
- **Deployment formats** (`TorchScript`, `ONNX`): convert a trained model into a portable, optimized format that can run outside a full Python environment — e.g. `torch.jit.script(model)` for TorchScript, or `torch.onnx.export(model, ...)` for ONNX, which many non-Python inference servers and edge devices can run directly. This builds on the [Saving & Loading](#save-load) section — it's the extra step needed specifically for production inference outside PyTorch/Python.
- **Multi-GPU / distributed training** (`nn.DataParallel`, `torch.nn.parallel.DistributedDataParallel`): splits a batch (or the model itself, for extremely large models) across multiple GPUs to train faster or fit bigger models than a single GPU's memory allows. `DistributedDataParallel` (DDP) is the modern, recommended approach — `DataParallel` is simpler but has known performance limitations and is considered mostly legacy at this point. → the mechanics are covered in full in the dedicated [Distributed Training with DDP](#distributed-ddp) section below.

### 🎯 Interview Recap — Advanced Topics

**If you remember nothing else:** AMP trains in float16 where safe (with `GradScaler` to avoid gradient underflow) for speed/memory savings; `torch.compile` fuses operations into faster kernels with one line; custom `autograd.Function`s let you hand-write forward AND backward; accuracy alone can badly mislead on imbalanced data — use precision/recall/F1/confusion matrix too.

**Q: What problem does `GradScaler` solve in mixed-precision training?**
A: `float16`'s smaller numeric range can cause small gradients to underflow to exactly zero during backpropagation; `GradScaler` temporarily multiplies the loss by a large factor before `.backward()` so gradients stay above that underflow range, then unscales them before the optimizer actually applies the update.

**Q: What does `torch.compile` actually do?**
A: It analyzes a model's computation graph and fuses multiple small operations into fewer, more efficient GPU kernels, typically giving a speedup with no other code changes — the first call is slower due to compilation, and later calls with the same input shapes reuse the compiled version.

**Q: When would you write a custom `torch.autograd.Function` instead of relying on autograd automatically?**
A: When an operation has no clean derivative through standard ops (e.g. involves rounding or a non-differentiable step), or when you know a more efficient way to compute the gradient by hand than what autograd would derive by chaining existing operations.

**Q: Why can accuracy alone be a misleading metric?**
A: On imbalanced datasets, a model can achieve very high accuracy by always predicting the majority class while being completely useless at detecting the minority class — precision, recall, and F1 (built from the confusion matrix) reveal that failure where plain accuracy hides it.

**Q: What's the difference between precision and recall, in plain terms?**
A: Precision asks "of everything I predicted positive, how much was actually positive?" (matters when false positives are costly); recall asks "of everything actually positive, how much did I catch?" (matters when false negatives are costly).

**Q: If a model can't even memorize 10 training examples, what does that suggest?**
A: A bug (in the data pipeline, labels, or model code) rather than a capacity/regularization problem — deliberately trying to overfit a tiny subset is a standard, fast debugging technique to isolate whether an issue is "the model can't learn at all" vs. "the model generalizes poorly.

<a id="pytorch-internals"></a>
# Part E — Closing the Interview Gaps
## 19. PyTorch Internals — Memory Layout, Strides & Contiguity

### Why this matters
Everything so far treated a tensor as a black box that just "has a shape." Rigorous interviews — especially at companies building ML infrastructure rather than just using it — probe one level deeper: **how is a tensor actually laid out in memory, and why does `.view()` sometimes mysteriously fail?** Understanding this is also what separates "I called `.reshape()` because `.view()` gave an error" from actually knowing why, which is exactly the kind of distinction an interviewer is listening for.

### Core intuition
A tensor's numbers are really stored as one single, flat, 1-D block of memory — regardless of whether the tensor "looks like" a 3×4 matrix or a 2×3×4 cube. The `shape` and **`stride`** are just a *recipe* for how to reinterpret that flat block as an N-dimensional grid. **Stride** answers: *"to move one step along this dimension, how many elements do I skip over in the underlying flat memory?"*

**Real-world analogy:** think of a flat memory block like a single long bookshelf, and a tensor's shape/stride like instructions for reading it as a grid — *"read 4 books, that's row 1; skip ahead 4 books, read the next 4, that's row 2."* The books never move; only the *instructions for grouping them* change. Many tensor operations (like `.transpose()`) just rewrite the instructions, without touching a single book.

In [ ]:
import torch

x = torch.arange(12).reshape(3, 4)     # a flat block of 12 numbers, read as 3 rows x 4 columns
print(x)
print("shape :", x.shape)               # torch.Size([3, 4])
print("stride:", x.stride())             # (4, 1)
# Reading the stride: to move ONE step along dim 0 (down a row), skip 4 elements in flat memory.
# To move ONE step along dim 1 (across a column), skip 1 element. This is the DEFAULT "row-major"
# layout PyTorch (and NumPy) use: the LAST dimension is the one that's contiguous (packed tightly).

print("is_contiguous:", x.is_contiguous())    # True -- its strides match a fresh row-major allocation


### Views vs. copies — and why `.transpose()`/`.permute()` break contiguity

Some operations (slicing, `.view()`, `.squeeze()`/`.unsqueeze()`) just hand you a **new set of reading instructions over the exact same flat memory** — a *view*, free and instant, no numbers copied. Others (`.transpose()`, `.permute()`) also return a view — but one whose strides no longer match a straightforward row-major layout for the new shape, which makes the tensor **non-contiguous**.

In [ ]:
y = x.t()          # transpose -- swaps dim 0 and dim 1. Shape becomes (4, 3).
print("shape :", y.shape)
print("stride:", y.stride())             # (1, 4) -- literally the OLD strides, just swapped in order!
print("is_contiguous:", y.is_contiguous())    # False

# Proof that y shares memory with x -- NO data was copied, only the reading instructions changed:
print("same underlying memory:", x.data_ptr() == y.data_ptr())   # True
print("same storage, different strides -- that IS a 'view'")

# Now watch what happens if we try to .view() the transposed tensor:
try:
    y.view(12)      # tries to reinterpret y as one flat row of 12
except RuntimeError as e:
    print("\nRuntimeError:", str(e)[:120], "...")
# WHY this fails: .view() refuses to copy data -- it can ONLY relabel memory that's already laid out
# in a way compatible with the new shape using simple strides. After transpose(), walking through y
# in the new shape's natural order would require jumping around non-sequentially -- .view() won't
# silently do that walk-and-copy for you (it wants you to be explicit that a copy is happening).


In [ ]:
# THE FIX -- two options:

# Option 1: .contiguous() -- explicitly forces a real copy into a fresh, properly laid-out memory block
y_fixed = y.contiguous()
print("after .contiguous(): is_contiguous =", y_fixed.is_contiguous())      # True
print("same memory as y now?", y_fixed.data_ptr() == y.data_ptr())           # False -- a real copy happened
print(y_fixed.view(12))         # now .view() works, because y_fixed IS laid out compatibly

# Option 2: .reshape() -- tries .view() first (free); if that's not possible, copies automatically for you
y_reshaped = y.reshape(12)      # just works either way, no try/except needed
print(y_reshaped)

# TAKEAWAY: .reshape() is almost always the safer default for this exact reason (also mentioned back
# in the Tensor Operations section) -- but knowing WHY .view() can fail, and what .contiguous() is
# actually doing under the hood, is the deeper understanding interviewers are checking for.


### Why this isn't just trivia — the performance angle

Non-contiguous memory access is slower to read, on both CPU (breaks cache-line prefetching) and GPU (breaks memory coalescing, where a warp of threads ideally reads one contiguous chunk together). This is why performance-sensitive code sometimes calls `.contiguous()` explicitly before a hot loop — but calling it *unnecessarily* wastes both time and memory on a copy you didn't need. Knowing when a tensor became non-contiguous (usually: right after a `.transpose()`/`.permute()`/certain advanced-indexing operations) is what lets you make that call deliberately instead of by trial-and-error.

### 🎯 Interview Recap — PyTorch Internals

**If you remember nothing else:** a tensor is a flat memory block plus a shape+stride "recipe" for reading it; `.transpose()`/`.permute()` change the recipe without copying (making the tensor non-contiguous); `.view()` refuses to copy, `.reshape()`/`.contiguous()` will.

**Q: What does a tensor's "stride" actually represent?**
A: For each dimension, how many elements to skip over in the underlying flat memory to move one step along that dimension — it's the recipe PyTorch uses to reinterpret a flat 1-D memory block as an N-dimensional tensor.

**Q: Why does calling `.view()` on the result of `.transpose()` often raise a `RuntimeError`?**
A: `.transpose()` returns a view with strides that no longer correspond to a straightforward row-major layout for the tensor's apparent shape; `.view()` is only allowed to relabel memory, never copy it, so it refuses when the requested shape can't be expressed with simple strides over the existing memory.

**Q: What's the actual difference between `.view()` and `.reshape()`, at the memory level?**
A: `.view()` always returns a new view over the exact same memory and raises an error if that's not possible; `.reshape()` returns a view when possible and silently falls back to making a contiguous copy (equivalent to `.contiguous().view(...)`) when it isn't.

**Q: If two tensors share the same underlying storage (e.g. one is a view of the other), what happens if you modify one?**
A: The other changes too, since they're reading from the same physical memory — the same sharing behavior discussed for the NumPy bridge earlier in the notebook.

**Q: Why might calling `.contiguous()` unnecessarily hurt performance?**
A: It forces a real memory copy; calling it when the tensor is already contiguous (or when the following operation doesn't actually need contiguous memory) wastes both time and memory for no benefit.

<a id="debugging-profiling"></a>
## 20. Debugging & Profiling PyTorch Like a Pro

### Why this matters
Section 18's debugging checklist told you *what* the common symptoms mean. This section covers the actual **tools** professionals reach for to go from "something's wrong" to "here's the exact line/layer causing it" — the difference between guessing and knowing, and a very common practical/systems-thinking interview theme.

### `torch.autograd.detect_anomaly()` — finding exactly where a NaN was born

**The problem it solves:** a `NaN` loss tells you training broke, but not *where*. Since a `NaN` propagates and multiplies through every downstream operation, by the time you see it in the final loss it could have originated dozens of operations earlier. `detect_anomaly()` wraps your forward+backward pass and, the moment any operation's backward computation produces a `NaN`/`Inf`, immediately raises an error **naming that exact operation** — like a stack trace, but for numerical instability.

In [ ]:
import torch
import torch.nn as nn

x = torch.tensor([1.0, -1.0, 2.0], requires_grad=True)

with torch.autograd.detect_anomaly():          # turn on anomaly detection for this block
    y = torch.log(x)          # log(-1) is NaN! (log is only defined for positive numbers)
    loss = y.sum()
    try:
        loss.backward()
    except RuntimeError as e:
        print("Caught it -- detect_anomaly names the EXACT operation that produced the bad value:")
        print(str(e)[:200], "...")
# Without detect_anomaly, you'd just see x.grad end up full of NaN, with no clue WHICH of
# potentially dozens of operations in a real model actually introduced it.
# NOTE: detect_anomaly adds real overhead -- turn it on only while debugging, not during normal training.


### Hooks — inspecting a model's insides without changing its code

A **hook** is a function you attach to any layer that PyTorch automatically calls at a specific moment, letting you observe (or even modify) values flowing through — without editing the model's `forward()` at all. **Forward hooks** fire right after a layer computes its output; **backward hooks** fire during backpropagation, exposing the gradients flowing through that layer.

In [ ]:
# ---- Forward hook example: automatically flagging "dead" ReLU neurons ----
# A ReLU neuron is "dead" if it outputs exactly 0 for almost every input -- meaning its gradient
# is also 0 almost always, so it has effectively stopped learning. Catching this early can save
# hours of wondering why a model has stopped improving.

def dead_relu_check(module, layer_input, layer_output):
    fraction_dead = (layer_output == 0).float().mean().item()
    if fraction_dead > 0.5:
        print(f"  ⚠️ warning: {fraction_dead:.0%} of this ReLU's outputs are exactly 0 -- possible dead neurons")

model = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 1))

handle = model[1].register_forward_hook(dead_relu_check)   # attach the hook to the ReLU layer (index 1)
output = model(torch.randn(8, 10) - 5)                        # a deliberately negative-shifted input,
                                                                  # so many ReLUs likely output 0
handle.remove()          # ALWAYS remove a hook once you're done -- otherwise it keeps firing (and
                             # can leak memory) on every future forward pass through that layer

# ---- Backward hook example: inspecting gradient magnitude flowing through a layer ----
def print_grad_norm(module, grad_input, grad_output):
    if grad_input[0] is not None:
        print(f"  gradient flowing INTO this layer has norm: {grad_input[0].norm().item():.4f}")

layer = model[0]
handle = layer.register_full_backward_hook(print_grad_norm)
loss = model(torch.randn(8, 10)).sum()
loss.backward()          # the hook fires automatically during this backward pass
handle.remove()


### `torchviz` — literally seeing the computational graph

Recall from [Autograd](#autograd) that every operation on a `requires_grad=True` tensor secretly builds a graph. `torchviz` renders that graph as an actual picture — every box is an operation, arrows show data flow — which is invaluable when debugging a custom or unusually-wired architecture where a connection might be missing or duplicated.

In [ ]:
# pip install torchviz --quiet   (also requires the system 'graphviz' package to render images)
# from torchviz import make_dot
#
# model = nn.Sequential(nn.Linear(10, 5), nn.ReLU(), nn.Linear(5, 1))
# x = torch.randn(1, 10)
# y = model(x)
# make_dot(y, params=dict(model.named_parameters())).render("computation_graph", format="png")
# This saves an image showing the full chain of operations autograd recorded to produce y --
# genuinely useful for confirming a custom forward() is wired the way you THINK it's wired.


### The PyTorch Profiler — finding out what's actually slow

Guessing where time is going ("maybe it's the convolution layers?") wastes time and is often wrong. The **Profiler** measures every operation's actual CPU/GPU time and memory usage during a real run, and reports it as a sortable table — turning "why is this slow" from a guess into a measurement.

In [ ]:
from torch.profiler import profile, ProfilerActivity

model = nn.Sequential(nn.Linear(1000, 1000), nn.ReLU(), nn.Linear(1000, 10))
x = torch.randn(64, 1000)

activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if torch.cuda.is_available() else [])

with profile(activities=activities, record_shapes=True) as prof:
    for _ in range(5):                     # profile a few iterations for a representative sample
        output = model(x)
        loss = output.sum()
        loss.backward()

# Print a table of the top 5 most time-consuming operations, sorted by total CPU time
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=5))
# In a real slow-training investigation, this table tells you IMMEDIATELY whether the bottleneck
# is a specific layer's forward pass, its backward pass, or something unexpected like data loading
# -- instead of you manually wrapping suspect lines in torch.cuda.Event timers one at a time.


### 🎯 Interview Recap — Debugging & Profiling

**If you remember nothing else:** `detect_anomaly()` names the exact op that first produced a NaN/Inf; forward hooks inspect layer outputs, backward hooks inspect gradients, both without touching model code; `torchviz` draws the actual computation graph; the Profiler measures where time/memory really goes instead of guessing.

**Q: What does `detect_anomaly()` actually help you find that a plain "loss is NaN" doesn't?**
A: The exact operation, during the backward pass, that first produced a `NaN`/`Inf` value — instead of just knowing the final loss became `NaN` with no idea which of potentially many upstream operations caused it.

**Q: What's the difference between a forward hook and a backward hook?**
A: A forward hook fires right after a layer's `forward()` runs, letting you inspect (or modify) its output; a backward hook fires during backpropagation, letting you inspect the gradients flowing through that layer — neither requires editing the model's own code.

**Q: Why must you call `.remove()` on a hook when you're done with it?**
A: Otherwise it keeps firing on every subsequent forward/backward pass through that layer, wasting compute and potentially leaking memory if it holds references to tensors.

**Q: What does the PyTorch Profiler tell you that manual print-statement timing can't easily?**
A: Exactly which specific operation — and whether it ran on CPU or GPU — is consuming the most time or memory across an entire run, automatically, rather than requiring you to guess which lines to manually time.

**Q: What is `torchviz` used for?**
A: Rendering an actual picture of the computational graph autograd built for a given output — useful for visually confirming a custom or complex architecture is wired the way you intended.

<a id="distributed-ddp"></a>
## 21. Distributed Training with DDP

### Why this matters
A single GPU eventually runs out of either time (large datasets) or memory (large models). Scaling to multiple GPUs — often multiple *machines* — is standard practice for any serious training run, and `DistributedDataParallel` (DDP) is the modern, correct way to do it. This section covers the mechanics referenced briefly back in [Advanced Topics](#advanced).

### Why `DataParallel` (the older approach) is considered legacy
`nn.DataParallel` runs as a **single process**, splitting each batch into chunks and farming them out to multiple GPUs using **threads**. Two real problems: Python's Global Interpreter Lock (GIL) means those threads can't truly run Python code in parallel, creating a bottleneck; and it gathers every GPU's outputs back onto **one** "primary" GPU to compute the loss, which makes that GPU's memory usage and workload disproportionately larger than the others.

### How DDP actually works

**Core intuition:** instead of one process juggling multiple GPUs, DDP launches **one independent process per GPU**, each holding its **own full copy** of the model. Each process forward/backward-passes its own slice of the data independently and in true parallel (no GIL bottleneck, since these are separate processes, not threads). Then, at the moment `loss.backward()` computes gradients, DDP automatically **all-reduces** them — averaging every process's gradients together across all GPUs — *before* `optimizer.step()` runs. Since every process started with identical weights and then applies the exact same averaged gradient, **every replica stays perfectly identical after every single step**, even though each one only ever saw a fraction of the data.

**Real-world analogy:** picture a study group where everyone works through a *different* subset of practice problems on their own, then — before anyone writes down a final answer — the group huddles and averages their individual answers together, so everyone ends up writing the exact same (better, less noisy) final answer. Repeat for the next batch of problems.

In [ ]:
# This is REFERENCE code: DDP genuinely needs multiple processes/GPUs to run, so this can't be
# executed as a single notebook cell. It's launched from the command line like:
#     torchrun --nproc_per_node=4 train_ddp.py      (runs 4 copies of this file, one per GPU)
# Every one of those 4 processes executes this ENTIRE script independently and simultaneously.

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

def main():
    dist.init_process_group(backend="nccl")     # nccl: NVIDIA's fast GPU-to-GPU communication library
    rank = dist.get_rank()                        # THIS process's ID: 0, 1, 2, 3...
    torch.cuda.set_device(rank)

    model = nn.Linear(10, 1).to(rank)
    model = DDP(model, device_ids=[rank])           # <-- the key line: wraps the model for gradient sync

    # DistributedSampler makes sure each process's DataLoader sees a DIFFERENT, non-overlapping
    # slice of the dataset -- without this, every process would redundantly train on everything
    sampler = DistributedSampler(train_dataset)        # train_dataset assumed defined elsewhere
    loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
    # note: shuffling is handled BY the sampler here, so DataLoader's own shuffle=True is not used

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(10):
        sampler.set_epoch(epoch)          # reshuffles differently each epoch -- without this call,
                                             # every epoch would reuse the SAME split/order
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(rank), y_batch.to(rank)
            optimizer.zero_grad()
            loss = nn.functional.mse_loss(model(X_batch), y_batch)
            loss.backward()          # <-- DDP transparently all-reduces (averages) gradients across
                                        #     every process HERE, as part of this very call
            optimizer.step()           # every process applies the SAME averaged update -> stays in sync

if __name__ == "__main__":
    main()


### When DDP isn't enough: a pointer to FSDP

DDP assumes the **entire model** fits in a single GPU's memory (since every process holds a full copy). For very large models — including most modern LLMs — even that isn't possible. **`FullyShardedDataParallel` (FSDP)** takes this further: it shards the model's *parameters, gradients, and optimizer states* across GPUs too, not just the data, so no single GPU ever needs to hold the entire model at once. This is the standard approach for training or fully fine-tuning genuinely large models, and connects directly to the memory-saving techniques in the [next section](#llm-finetuning).

### 🎯 Interview Recap — Distributed Training (DDP)

**If you remember nothing else:** DDP runs one process per GPU, each with a full model replica processing its own data shard; gradients are averaged (all-reduced) across processes during `.backward()`, keeping every replica identical after each step; `DistributedSampler` splits the data, not `DataLoader`'s own shuffle.

**Q: What's the key architectural difference between `DataParallel` and `DistributedDataParallel`?**
A: `DataParallel` is single-process, multi-threaded (bottlenecked by Python's GIL) and gathers everything onto one primary GPU; DDP is multi-process — one process per GPU, each with its own full model replica — which avoids both bottlenecks and is the recommended modern approach.

**Q: At what point in the training step does DDP synchronize gradients across GPUs?**
A: During `loss.backward()` — DDP automatically all-reduces (averages) gradients across every process as part of that call, transparently, before `optimizer.step()` runs.

**Q: Why do all DDP replicas end up with identical weights after every step, even though each only saw part of the data?**
A: Because gradients are averaged across all processes before the optimizer update is applied, so every replica — which started identical — applies the exact same averaged update.

**Q: What does `DistributedSampler` do, and why does `sampler.set_epoch()` matter?**
A: It splits the dataset so each process's `DataLoader` sees a different, non-overlapping shard; `set_epoch()` reseeds the shuffle differently per epoch — without calling it, every epoch would reuse the same split and order.

**Q: When would you need FSDP instead of plain DDP?**
A: When the model itself — not just the dataset — is too large to fit on a single GPU; FSDP shards the model's parameters, gradients, and optimizer states across GPUs, whereas DDP requires every GPU to hold a full copy of the model.

<a id="llm-finetuning"></a>
## 22. Fine-Tuning LLMs — LoRA, QLoRA & Quantization

### Why this matters
This is the PyTorch-under-the-hood layer beneath tools like LangChain/LangGraph and any RAG pipeline that involves adapting an open-weight model: if you're fine-tuning a generator or embedding model for a retrieval system, this is almost certainly *how* it actually gets done in practice — full fine-tuning of a modern model is usually not an option at all.

### The problem: full fine-tuning doesn't fit in memory
Consider fine-tuning a "small-by-LLM-standards" 7-billion-parameter model, fully, in float32:
- **Weights:** 7B params × 4 bytes = **28 GB**
- **Gradients:** one gradient per weight, same size = another **28 GB**
- **Adam optimizer state:** Adam keeps two extra running values per parameter (see [Optimizers](#optimizers)) = **56 GB**
- **Total: ~112 GB** — before even counting activations — for a model that's considered *small* today, and on hardware far beyond a typical single consumer or even single datacenter GPU.

**LoRA (Low-Rank Adaptation)** sidesteps this almost entirely.

### The core idea and the math

Instead of updating the full weight matrix $W$ (shape `out_features × in_features` — potentially millions of numbers), **freeze $W$ completely** and learn a small *correction* $\Delta W$, expressed as the product of two much smaller matrices:
$$\Delta W = BA \qquad\qquad h = Wx + \Delta W x = Wx + BAx$$
where $A$ has shape `(r, in_features)` and $B$ has shape `(out_features, r)`, and $r$ (the **rank**) is small — often 8, 16, or 32 — far smaller than `in_features`/`out_features`. Only $A$ and $B$ are trained; $W$ never moves.

**Why "low rank" works at all (the intuition):** empirically, the *change* a pretrained model needs to adapt well to a new task tends to live in a much lower-dimensional space than the full weight matrix's size would suggest — you don't need full freedom to move every one of millions of numbers independently to capture a meaningful adaptation. $B$ and $A$, multiplied together, can only ever produce updates within that constrained low-rank space — and that constraint turns out to be enough for most fine-tuning tasks, while using a tiny fraction of the parameters.

**Why $B$ is initialized to exactly zero (and $A$ isn't):** at the very start of training, $\Delta W = BA = 0$ (since $B$ is all zeros), which means the model behaves **exactly** like the original pretrained model before any LoRA training has happened — a clean, safe starting point. $A$ is initialized with small random values (so gradients aren't stuck at zero); $B$ starting at zero is what guarantees the *initial* behavior is unchanged.

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    # A from-scratch LoRA-adapted linear layer -- implementing this by hand (rather than only
    # calling a library) is exactly the kind of "prove you understand the mechanism" exercise
    # covered in the Career section's portfolio ideas.

    def __init__(self, in_features, out_features, r=8, alpha=16):
        super().__init__()
        # Pretend this weight arrived already pretrained -- frozen, no gradient computed for it
        self.frozen_weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02, requires_grad=False)
        self.frozen_bias = nn.Parameter(torch.zeros(out_features), requires_grad=False)

        # The LoRA adapters -- these ARE trainable (requires_grad=True is the nn.Parameter default)
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)     # small random init
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))            # ZERO init -- see explanation above
        self.scaling = alpha / r          # a scaling factor controlling how much the adapter contributes

    def forward(self, x):
        base_out = x @ self.frozen_weight.T + self.frozen_bias        # the ORIGINAL frozen computation
        lora_out = (x @ self.lora_A.T) @ self.lora_B.T * self.scaling   # the small learned correction
        return base_out + lora_out

layer = LoRALinear(in_features=512, out_features=512, r=8)

frozen_params = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in layer.parameters() if p.requires_grad)
print(f"Frozen (untouched) parameters:   {frozen_params:,}")
print(f"Trainable (LoRA) parameters:     {trainable_params:,}")
print(f"Trainable fraction: {trainable_params / (frozen_params + trainable_params):.2%}")
# For a 512x512 layer: full weight = 262,144 params. LoRA at r=8: (8*512)+(512*8) = 8,192 params
# -- about 3% of the original, and for a real multi-billion-parameter model this fraction shrinks
# even further, which is exactly why LoRA fine-tuning fits on hardware full fine-tuning never could.

x = torch.randn(4, 512)
output = layer(x)
print("output shape:", output.shape)


### QLoRA — LoRA plus quantization

**QLoRA** adds one more memory-saving idea on top of LoRA: store the **frozen** base model's weights in **4-bit** precision instead of 16/32-bit, while keeping the small trainable LoRA adapters ($A$, $B$) in a higher precision (e.g. `bfloat16`) so training still works well. Since the huge frozen weights dominate a model's memory footprint, shrinking *those specifically* to 4 bits — while the comparatively tiny adapters stay precise — gives the best of both: dramatically less memory, with training quality close to full-precision LoRA.

### Quantization, from first principles

**The core idea:** represent a range of numbers using far fewer bits, by remembering a **scale factor** that maps the small integer range back to the original range of real values. This is a form of lossy compression, trading a small amount of numerical precision for a large reduction in memory (and often faster computation, since smaller integer types move through hardware faster).

In [ ]:
def quantize_to_int8(x):
    # Find the scale that maps this tensor's largest-magnitude value to the edge of int8's range (-127..127)
    scale = x.abs().max() / 127.0
    x_int8 = torch.round(x / scale).to(torch.int8)      # compress: divide down, round to nearest integer
    return x_int8, scale

def dequantize(x_int8, scale):
    return x_int8.float() * scale          # decompress: multiply back up to (approximately) the original range

original = torch.randn(4, 4) * 3        # some example weight values
quantized, scale = quantize_to_int8(original)
recovered = dequantize(quantized, scale)

print("original (float32):\n", original)
print("\nquantized (int8, 1 byte each instead of 4):\n", quantized)
print("\nrecovered (dequantized back to float):\n", recovered)
print(f"\nmax reconstruction error: {(original - recovered).abs().max().item():.4f}")
print(f"memory used: float32 = {original.numel()*4} bytes, int8 = {quantized.numel()*1} bytes (4x smaller)")

# This "find a scale, round to a small integer type, remember the scale to undo it later" pattern
# is the core idea behind essentially every practical quantization scheme -- real implementations
# (like bitsandbytes' NF4 format used in QLoRA) use smarter, non-uniform scales tuned to how
# neural network weights are actually distributed, but the fundamental trade-off is identical.


### The standard, real-world tooling

In practice, you rarely hand-roll LoRA or quantization from scratch (the above was to build real understanding) — the Hugging Face `peft` (Parameter-Efficient Fine-Tuning) and `bitsandbytes` libraries provide production-ready, heavily-optimized implementations.

In [ ]:
# pip install peft bitsandbytes transformers accelerate --quiet
#
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
# from peft import LoraConfig, get_peft_model
#
# # ---- Load the base model in 4-bit (this is the "Q" in QLoRA) ----
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,   # LoRA adapter math still happens in bfloat16
#     bnb_4bit_quant_type="nf4",                 # a quantization scheme tuned for neural network weights
# )
# base_model = AutoModelForCausalLM.from_pretrained("some-7b-model", quantization_config=bnb_config)
#
# # ---- Attach trainable LoRA adapters on top ----
# lora_config = LoraConfig(
#     r=8, lora_alpha=16,                          # matches the math covered above
#     target_modules=["q_proj", "v_proj"],           # WHICH layers get LoRA adapters (commonly, attention projections)
#     lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
# )
# model = get_peft_model(base_model, lora_config)
# model.print_trainable_parameters()
# # typical output: "trainable params: 4,194,304 || all params: 7,000,000,000 || trainable%: 0.06%"
#
# # ---- From here, training uses the EXACT same loop from Section 10 ----
# # optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
# # ... loss.backward(); optimizer.step(); optimizer.zero_grad() ...


**Real-world example, tying it back:** if you're fine-tuning an embedding model to better match your RAG pipeline's specific documents, or lightly specializing an open-weight generator model for an agent's tone/format, LoRA is what makes that realistically doable on a single GPU — the exact same underlying `nn.Module` + optimizer + training loop from earlier in this notebook, just with 99%+ of the parameters frozen.

### 🎯 Interview Recap — LoRA, QLoRA & Quantization

**If you remember nothing else:** full fine-tuning needs memory for weights + gradients + optimizer state (roughly 4x a model's weight size for Adam); LoRA freezes the big weight matrix and trains two tiny ones instead (`ΔW = BA`, with `B` initialized to zero); QLoRA additionally stores the frozen base in 4-bit; quantization is "find a scale, round to fewer bits, remember the scale to undo it."

**Q: Why does full fine-tuning of a large model need roughly 4x the memory of just storing its weights?**
A: Beyond the weights themselves, you need equal-sized storage for their gradients, plus (with Adam) two more equal-sized running-average buffers for the optimizer's momentum and adaptive scaling — roughly weights + gradients + 2×optimizer-state ≈ 4x the base weight memory.

**Q: What does LoRA actually train, and what stays frozen?**
A: The original weight matrix `W` is frozen entirely; LoRA adds a low-rank correction `ΔW = BA` (two small matrices `A` and `B`) that gets added to `W`'s output, and only `A` and `B` are trained.

**Q: Why is `B` initialized to zero instead of small random values like `A`?**
A: So that `ΔW = BA` is exactly zero at the very start of training, meaning the model behaves identically to the original pretrained model before any LoRA training has happened — a safe, unchanged starting point.

**Q: What's the difference between LoRA and QLoRA?**
A: QLoRA is LoRA plus quantization — it stores the large, frozen base model's weights in 4-bit precision to save memory, while still training the small LoRA adapters in a higher precision like bfloat16.

**Q: What is quantization doing, at its core?**
A: Representing a range of numbers with far fewer bits by computing a scale factor that maps the reduced integer range back to (approximately) the original value range — trading a small amount of precision for a large reduction in memory and often faster computation.

**Q: Why does the "low rank" assumption behind LoRA work in practice?**
A: Empirically, the change needed to adapt a pretrained model to a new task tends to live in a much lower-dimensional space than the full weight matrix — so constraining the update to a low-rank form (`BA`) still captures most of the useful adaptation, using a tiny fraction of the parameters.

<a id="career"></a>
## 23. Career Roadmap, Portfolio Ideas & Final Interview Review

### Where PyTorch skills show up across ML roles

| Role | How PyTorch shows up |
|---|---|
| **ML Engineer** | Building, training, and productionizing models; custom training loops; performance (AMP, `torch.compile`); deployment (TorchScript/ONNX) |
| **Data Scientist** | Prototyping models quickly; understanding architectures well enough to choose and adapt them; evaluation metrics |
| **MLOps / AI Infrastructure** | Checkpointing, distributed training, mixed precision, reproducibility, hyperparameter tuning pipelines (Optuna), monitoring training runs |
| **Applied / Research Scientist** | Implementing architectures from papers, custom autograd, understanding the exact math well enough to modify it |

If you've been working with RAG pipelines, LangGraph, and LangChain, the connection to this notebook is direct: embedding models, the transformer backbones behind the LLMs those pipelines call, and any fine-tuning you do to an open-weight model are all built and trained using exactly the PyTorch concepts covered here — [tensors](#tensors-creation), [autograd](#autograd), and especially [attention/transformers](#transformers) are literally what's running under the hood.

### Portfolio project ideas, roughly increasing in difficulty

1. **Tabular classifier ANN** — apply [Section 10](#training-loop)'s pattern to a Kaggle tabular dataset; practice the full data → train → evaluate → metrics pipeline.
2. **Image classifier CNN with transfer learning** — combine [CNN](#cnn) + [Transfer Learning](#transfer-learning) on a small custom image dataset (even one you collect yourself).
3. **Time-series forecaster (LSTM/GRU)** — extend [Section 12](#rnn)'s sine-wave example to a real dataset (energy demand, stock prices, sensor data).
4. **Hyperparameter-tuned pipeline** — take project 1 or 2 and wrap the training in [Optuna](#optuna), then write up which hyperparameters mattered most.
5. **A Transformer from scratch** — implement [Section 13](#transformers)'s attention math yourself (not just calling `nn.MultiheadAttention`) for a small classification or translation task; this is a very common "prove you understand it" project.
6. **End-to-end MLOps project** — training script with checkpointing, mixed precision, a config-driven Optuna sweep, TensorBoard logging, and a TorchScript/ONNX export for serving — demonstrates the *engineering* side, not just the modeling side.

Being able to clearly explain **why** you made each design decision in a project (not just that it works) is usually worth more in an interview than the project's raw performance number.

### Where to go next
- **Official PyTorch tutorials** (pytorch.org/tutorials) — always up to date with the latest APIs.
- **Foundational papers**, now that the intuition is in place: *"Attention Is All You Need"* (Transformers), *"Deep Residual Learning for Image Recognition"* (ResNet/skip connections), *"Batch Normalization..."*, *"Dropout: A Simple Way to Prevent Neural Networks from Overfitting"*.
- **Courses** for structured deeper study: fast.ai (very practical, code-first), Stanford CS231n (CNNs/vision), CS224n (NLP), CS230 (general deep learning).
- **Kaggle competitions** — realistic, messy data and a leaderboard forces genuinely good practices, not just toy-dataset intuition.
- **Read real model source code** — e.g. a small, well-written model implementation on GitHub (Hugging Face's `transformers` library is a great one) — seeing production-quality PyTorch code is one of the fastest ways to level up past "notebook code" habits.

---
## 🐛 Practice: Spot the Bugs

A classic PyTorch interview/take-home format is: *"here's some broken training code, find the bugs."* The snippet below has **four intentional bugs** covering ideas from across this notebook. Try to spot all four before scrolling to the fixed version and explanation.

In [ ]:
# ---- BUGGY VERSION -- can you find all 4 bugs before reading the fixed version below? ----
#
# model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# loss_fn = nn.CrossEntropyLoss()
#
# for epoch in range(50):
#     for X_batch, y_batch in train_loader:
#         logits = model(X_batch)
#         probs = torch.softmax(logits, dim=1)          # BUG 1 is hiding somewhere near here...
#         loss = loss_fn(probs, y_batch)
#         loss.backward()
#         optimizer.step()                                # ...and BUG 2 is missing from just above this line
#
#     for X_batch, y_batch in val_loader:                    # BUG 3: something is missing before this loop
#         val_logits = model(X_batch)
#         val_loss = loss_fn(val_logits, y_batch)
#
# torch.save(model, "model.pth")                              # BUG 4: not a crash, but not best practice


**The fixed version, with each bug explained:**

```python
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(50):
    model.train()                                    # (fix 3, part A) explicitly set training mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                          # (fix 2) MUST clear gradients each iteration, or
                                                          # they silently accumulate across steps (Autograd section)
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)                  # (fix 1) feed RAW LOGITS, not softmax(logits) --
                                                            # CrossEntropyLoss applies softmax internally;
                                                            # pre-applying it double-softmaxes (Loss Functions section)
        loss.backward()
        optimizer.step()

    model.eval()                                       # (fix 3, part B) switch to eval mode before validating,
    with torch.no_grad():                                # and disable gradient tracking -- both from the
        for X_batch, y_batch in val_loader:                # Training Loop section
            val_logits = model(X_batch)
            val_loss = loss_fn(val_logits, y_batch)

torch.save(model.state_dict(), "model.pth")            # (fix 4) save state_dict, not the whole model object,
                                                           # for portability (Saving & Loading section)
```
- **Bug 1:** applying `softmax` before `CrossEntropyLoss` — it already applies softmax internally.
- **Bug 2:** missing `optimizer.zero_grad()` — gradients accumulate silently across every iteration without it.
- **Bug 3:** missing `model.train()`/`model.eval()` switches (and `torch.no_grad()` around validation) — without them Dropout/BatchNorm behave inconsistently and validation wastes memory tracking unneeded gradients.
- **Bug 4:** `torch.save(model, ...)` instead of `torch.save(model.state_dict(), ...)` — works today, but is fragile against future refactors.

---
## ⚡ Master Rapid-Fire Review — One Line Per Topic

Use this table for a final skim right before an interview. Every row links back to its full section if you need a refresher.

| Topic | The one thing to remember |
|---|---|
| [Tensors](#tensors-creation) | An N-dimensional grid of one shared dtype; check `.shape`/`.dtype`/`.device` first when debugging |
| [Tensor ops](#tensors-ops) | `*` = element-wise, `@` = matrix multiply; `dim=X` in a reduction removes dimension X from the output |
| [GPU](#gpu) | Model and data must share a device; always use the `"cuda" if available else "cpu"` pattern |
| [Autograd](#autograd) | `.backward()` walks the graph applying the chain rule; gradients accumulate — always `zero_grad()` |
| [Workflow](#workflow) | Every training step: `zero_grad → forward → loss → backward → step` |
| [`nn.Module`](#nn-module) | Packages learnable `Parameter`s + `forward()`; non-linear activations are what make depth meaningful |
| [Loss functions](#loss-functions) | Feed raw logits; MSE/MAE for regression, BCEWithLogits for binary, CrossEntropy for multi-class |
| [Optimizers](#optimizers) | Adam = momentum + adaptive per-parameter scaling; learning rate is the most important hyperparameter |
| [Dataset/DataLoader](#datasets-dataloaders) | `Dataset` fetches one sample; `DataLoader` batches/shuffles/parallelizes; shuffle train, not eval |
| [Training loop](#training-loop) | `model.train()`/`.eval()` toggle Dropout/BatchNorm; wrap eval in `no_grad()`; watch the train/val loss gap |
| [CNN](#cnn) | Shared sliding filters → parameter sharing + translation invariance; `Conv→BN→ReLU→Pool`, then `Flatten→Linear` |
| [RNN/LSTM/GRU](#rnn) | Hidden state carries memory forward; LSTM's cell state fixes vanishing gradients via mostly-additive updates |
| [Transformers](#transformers) | Self-attention lets every position attend to every other directly; scale by `√d_k`; needs positional encoding |
| [Save/Load](#save-load) | Save `state_dict()`, not the whole model; resuming training needs optimizer state too |
| [Optuna](#optuna) | TPE sampler learns from past trials; pruning stops bad trials early to save compute |
| [Transfer learning](#transfer-learning) | Early layers = generic features; freeze for feature extraction, unfreeze (small LR) to fine-tune |
| [Regularization](#regularization) | Dropout/weight decay/augmentation fight overfitting; early stopping keeps the best epoch, not the last |
| [Advanced](#advanced) | AMP + `GradScaler` for fast/safe float16 training; precision/recall/F1 beat accuracy alone on imbalanced data |
| [PyTorch internals](#pytorch-internals) | Strides + shape are the recipe for reading flat memory; `.view()` needs contiguous memory, `.reshape()`/`.contiguous()` handle it for you |
| [Debugging/Profiling](#debugging-profiling) | `detect_anomaly()` finds *where* a NaN was born; hooks peek inside layers; the Profiler finds *what's actually slow* |
| [Distributed (DDP)](#distributed-ddp) | Each process trains on its own data shard, then gradients are averaged (all-reduced) before every step — everyone ends up identical |
| [LoRA/QLoRA/Quantization](#llm-finetuning) | Freeze the big weight matrix, train two tiny ones instead (`B` starts at zero); quantization shrinks stored precision, not capability |

### 🎁 Bonus round — trick questions interviewers love

**Q: What's the difference between `optimizer.zero_grad()` and `model.zero_grad()`?**
A: They're functionally similar — `model.zero_grad()` zeroes gradients for all of the model's parameters, `optimizer.zero_grad()` zeroes gradients for whatever parameters were given to that optimizer (usually the same set, but not guaranteed if the optimizer only received a subset). Calling it on the optimizer is the conventional choice.

**Q: What's the difference between `.detach()` and the older `.data`?**
A: Both give you a tensor's values without gradient tracking, but `.detach()` is the safe, recommended way — it properly participates in autograd's internal safety checks; `.data` bypasses those checks entirely and can silently produce incorrect gradients if misused, so it's considered legacy/discouraged in modern code.

**Q: Why does `nn.CrossEntropyLoss` exist separately from `nn.NLLLoss`?**
A: `CrossEntropyLoss` combines `LogSoftmax` + `NLLLoss` in one numerically stable step and expects raw logits; `NLLLoss` alone expects you to already have log-probabilities (e.g. from applying `log_softmax` yourself) — `CrossEntropyLoss` is the convenient, standard choice when starting from raw model outputs.

**Q: What happens if you forget `optimizer.step()`?**
A: Gradients still get computed correctly by `.backward()`, but the weights never actually update, since nothing ever reads `.grad` and applies it — loss will stay flat instead of improving.

**Q: Is `nn.Linear` the same as `torch.nn.functional.linear`?**
A: They perform the identical underlying math; `nn.Linear` is a stateful `Module` that owns and manages its own learnable weight/bias (so it shows up automatically in `model.parameters()`), while `torch.nn.functional.linear` is the raw, stateless function — used directly only if you're managing the weights yourself elsewhere.

**Q: What's the difference between `model.parameters()` and `model.state_dict()`?**
A: `.parameters()` returns only the trainable `Parameter` tensors (what an optimizer needs); `.state_dict()` returns an ordered dictionary of every persistent piece of state, including non-trainable buffers like BatchNorm's running mean/variance — which is why saving/loading uses `state_dict()`, not `.parameters()`.

**Q: For `nn.Linear(in_features, out_features)`, what shape is the weight matrix — and why does the order matter?**
A: Shape `(out_features, in_features)` — the reverse of what many people guess — because the actual computation is `x @ weight.T + bias`; getting this backwards is a common source of confusion when implementing a layer by hand.

---
## 🏁 You've reached the end

If you worked through this notebook top to bottom, you've now covered the same ground as most PyTorch-focused technical interviews, from "what is a tensor" all the way to attention math and hyperparameter search. The [Interview Recap](#tensors-creation) boxes throughout and the rapid-fire table above are built to be re-skimmed — bookmark this notebook and revisit it the night before an interview rather than trying to re-read it in full.

**Good luck — you've got this.** 🔥